<a href="https://colab.research.google.com/github/a-memoir/Automated-Medical-Report-NNH/blob/main/14oct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import os
import json
import pandas as pd
import torch

base = Path("/content/drive/MyDrive/NoiSoi_Matching")
v5 = base / "baseline_model_v5"

print("=" * 70)
print("1. RUNTIME")
print("=" * 70)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Fresh notebook objects:")
for name in ["model_v5", "V5Model", "test_loader", "tokenizer",
             "image_processor", "test_dataset"]:
    print(f"  {name}: {'exists' if name in globals() else 'not initialized'}")

print("\n" + "=" * 70)
print("2. CHECKPOINTS")
print("=" * 70)
for name in ["best.pt", "last.pt"]:
    p = v5 / "checkpoints" / name
    print(f"{name}: exists={p.exists()}")
    if p.exists():
        print(f"  Size: {p.stat().st_size / (1024**3):.3f} GiB")

print("\n" + "=" * 70)
print("3. DATA FILES")
print("=" * 70)
paths = {
    "V4 case manifest": base / "baseline_model_v4" / "v4_training_manifest.csv",
    "Image manifest": base / "final_manifest" / "final_dataset_manifest.csv",
    "Structured targets": base / "v3_ontology_v2" / "v3_structured_targets_v2_2.csv",
    "Ontology mapping": base / "baseline_model_v4" / "diagnostics" / "frozen_concept_ontology_mapping.csv",
    "Training summary": v5 / "diagnostics" / "v5_final_training_summary.json",
}
for label, p in paths.items():
    print(f"{label}: exists={p.exists()}", end="")
    if p.exists():
        print(f" | size={p.stat().st_size / (1024**2):.2f} MB")
    else:
        print()

print("\n" + "=" * 70)
print("4. TRAINING SUMMARY")
print("=" * 70)
summary_path = paths["Training summary"]
if summary_path.exists():
    try:
        summary = json.loads(summary_path.read_text())
        for k in ["model", "status", "num_epochs", "global_step",
                  "best_val", "best_checkpoint", "last_checkpoint"]:
            print(f"{k}: {summary.get(k)}")
    except Exception as e:
        print("Could not read summary:", repr(e))

print("\n" + "=" * 70)
print("5. MANIFEST INTEGRITY")
print("=" * 70)
manifest_path = paths["V4 case manifest"]
if manifest_path.exists():
    try:
        df = pd.read_csv(manifest_path, dtype={"case_id": str})
        print("Shape:", df.shape)
        print("Unique case IDs:", df["case_id"].nunique())
        print("Split counts:", df["split"].value_counts(dropna=False).to_dict())
        print("Required vector columns present:", {
            c: c in df.columns for c in [
                "concept_vector_str",
                "negative_vector_str",
                "attribute_vector_str"
            ]
        })
    except Exception as e:
        print("Could not read manifest:", repr(e))

print("\nR21A COMPLETE")


Mounted at /content/drive
1. RUNTIME
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
Fresh notebook objects:
  model_v5: not initialized
  V5Model: not initialized
  test_loader: not initialized
  tokenizer: not initialized
  image_processor: not initialized
  test_dataset: not initialized

2. CHECKPOINTS
best.pt: exists=True
  Size: 4.321 GiB
last.pt: exists=True
  Size: 4.321 GiB

3. DATA FILES
V4 case manifest: exists=True | size=1.83 MB
Image manifest: exists=True | size=51.84 MB
Structured targets: exists=True | size=5.94 MB
Ontology mapping: exists=True | size=0.00 MB
Training summary: exists=True | size=0.00 MB

4. TRAINING SUMMARY
model: V5
status: TRAINING_COMPLETE
num_epochs: 5
global_step: 1920
best_val: 0.424518172663671
best_checkpoint: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/checkpoints/best.pt
last_checkpoint: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/checkpoints/last.pt

5. MANIFEST INTEGRITY
Shape: (7606, 15)
Unique

In [2]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import torch

ckpt_path = Path(
    "/content/drive/MyDrive/NoiSoi_Matching/"
    "baseline_model_v5/checkpoints/best.pt"
)

print("Checkpoint exists:", ckpt_path.exists())
print("Checkpoint size (GiB):", round(ckpt_path.stat().st_size / 1024**3, 3))
print("PyTorch:", torch.__version__)

print("\nLoading checkpoint metadata with mmap=True, weights_only=True...")
try:
    checkpoint = torch.load(
        ckpt_path,
        map_location="cpu",
        weights_only=True,
        mmap=True,
    )

    print("Checkpoint object type:", type(checkpoint).__name__)

    if isinstance(checkpoint, dict):
        print("\nTop-level keys:")
        for key, value in checkpoint.items():
            if isinstance(value, dict):
                print(f"  {key}: dict, {len(value)} entries")
            elif torch.is_tensor(value):
                print(f"  {key}: tensor, shape={tuple(value.shape)}")
            else:
                print(f"  {key}: {type(value).__name__}")

        # Find a likely model state dictionary.
        state_candidates = []
        for key, value in checkpoint.items():
            if isinstance(value, dict) and value:
                tensor_count = sum(torch.is_tensor(v) for v in value.values())
                if tensor_count:
                    state_candidates.append((key, value, tensor_count))

        if all(torch.is_tensor(v) for v in checkpoint.values()):
            state_candidates.insert(0, ("<top-level state_dict>", checkpoint,
                                        len(checkpoint)))

        print("\nPotential state dictionaries:")
        for name, state, tensor_count in state_candidates:
            print(f"\n--- {name}: {tensor_count} direct tensor entries ---")
            shown = 0
            for key, value in state.items():
                if torch.is_tensor(value):
                    print(f"  {key}: {tuple(value.shape)} | {value.dtype}")
                    shown += 1
                    if shown >= 35:
                        print("  ... truncated")
                        break

    else:
        print("Unexpected checkpoint format; no model reconstruction attempted.")

except Exception as e:
    print("CHECKPOINT INSPECTION FAILED")
    print(type(e).__name__ + ":", str(e)[:1500])
    print("No training or checkpoint modification was performed.")

print("\nR21B COMPLETE")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint exists: True
Checkpoint size (GiB): 4.321
PyTorch: 2.11.0+cu130

Loading checkpoint metadata with mmap=True, weights_only=True...
CHECKPOINT INSPECTION FAILED
UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray._reconstruct was not an allowed glo

In [3]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import torch
import numpy as np

ckpt_path = Path(
    "/content/drive/MyDrive/NoiSoi_Matching/"
    "baseline_model_v5/checkpoints/best.pt"
)

# Allowlist only the NumPy reconstruction function named in the error.
try:
    reconstruct = np._core.multiarray._reconstruct
except AttributeError:
    reconstruct = np.core.multiarray._reconstruct

print("Checkpoint exists:", ckpt_path.exists())
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)

try:
    with torch.serialization.safe_globals([reconstruct]):
        checkpoint = torch.load(
            ckpt_path,
            map_location="cpu",
            weights_only=True,
            mmap=True,
        )

    print("\nLOAD PASS")
    print("Checkpoint type:", type(checkpoint).__name__)

    if isinstance(checkpoint, dict):
        print("\nTop-level keys:")
        for key, value in checkpoint.items():
            if isinstance(value, dict):
                print(f"  {key}: dict ({len(value)} entries)")
            elif torch.is_tensor(value):
                print(f"  {key}: tensor {tuple(value.shape)}")
            else:
                print(f"  {key}: {type(value).__name__}")

        print("\nLikely model state dictionaries:")
        for name, value in checkpoint.items():
            if isinstance(value, dict):
                tensor_items = [
                    (k, v) for k, v in value.items()
                    if torch.is_tensor(v)
                ]
                if tensor_items:
                    print(f"\n[{name}] tensor entries: {len(tensor_items)}")
                    for key, tensor in tensor_items[:30]:
                        print(f"  {key}: {tuple(tensor.shape)}")
                    if len(tensor_items) > 30:
                        print("  ... truncated")
    else:
        print("Unexpected checkpoint format; stop for inspection.")

except Exception as e:
    print("\nLOAD BLOCKED")
    print(type(e).__name__ + ":", str(e)[:2500])
    print("No training or checkpoint modification was performed.")

print("\nR21C COMPLETE")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint exists: True
PyTorch: 2.11.0+cu130
NumPy: 2.1.3

LOAD BLOCKED
UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy.ndarray was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy.ndarray])` or the `torch.serialization.safe_glo

In [4]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import torch
import numpy as np

ckpt_path = Path(
    "/content/drive/MyDrive/NoiSoi_Matching/"
    "baseline_model_v5/checkpoints/best.pt"
)

try:
    reconstruct = np._core.multiarray._reconstruct
except AttributeError:
    reconstruct = np.core.multiarray._reconstruct

safe_types = [reconstruct, np.ndarray]

print("Checkpoint exists:", ckpt_path.exists())
print("Using weights_only=True and mmap=True")

try:
    with torch.serialization.safe_globals(safe_types):
        checkpoint = torch.load(
            ckpt_path,
            map_location="cpu",
            weights_only=True,
            mmap=True,
        )

    print("\nLOAD PASS")
    print("Object type:", type(checkpoint).__name__)

    if isinstance(checkpoint, dict):
        print("Top-level keys:")
        for key, value in checkpoint.items():
            if isinstance(value, dict):
                print(f"  {key}: dict ({len(value)} entries)")
            elif torch.is_tensor(value):
                print(f"  {key}: tensor {tuple(value.shape)}")
            else:
                print(f"  {key}: {type(value).__name__}")

        for name, value in checkpoint.items():
            if isinstance(value, dict):
                tensors = [(k, v) for k, v in value.items()
                           if torch.is_tensor(v)]
                if tensors:
                    print(f"\nState candidate: {name}, {len(tensors)} tensors")
                    for key, tensor in tensors[:40]:
                        print(f"  {key}: {tuple(tensor.shape)}")
                    if len(tensors) > 40:
                        print("  ... truncated")
    else:
        print("Unexpected checkpoint format; stop for inspection.")

except Exception as e:
    print("\nLOAD BLOCKED")
    print(type(e).__name__ + ":", str(e)[:2500])
    print("No training or checkpoint modification was performed.")

print("\nR21D COMPLETE")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint exists: True
Using weights_only=True and mmap=True

LOAD BLOCKED
UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy.dtype was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy.dtype])` or the `torch.serialization.safe_glob

In [5]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import torch
import numpy as np

ckpt_path = Path(
    "/content/drive/MyDrive/NoiSoi_Matching/"
    "baseline_model_v5/checkpoints/best.pt"
)

try:
    reconstruct = np._core.multiarray._reconstruct
except AttributeError:
    reconstruct = np.core.multiarray._reconstruct

safe_types = [reconstruct, np.ndarray, np.dtype]

try:
    with torch.serialization.safe_globals(safe_types):
        checkpoint = torch.load(
            ckpt_path,
            map_location="cpu",
            weights_only=True,
            mmap=True,
        )

    print("LOAD PASS")
    print("Checkpoint type:", type(checkpoint).__name__)

    if isinstance(checkpoint, dict):
        print("\nTop-level keys:")
        for key, value in checkpoint.items():
            if isinstance(value, dict):
                print(f"\n{key}: dict ({len(value)} entries)")
                tensors = [
                    (k, v) for k, v in value.items()
                    if torch.is_tensor(v)
                ]
                for k, v in tensors[:25]:
                    print(f"  {k}: {tuple(v.shape)}")
                if len(tensors) > 25:
                    print("  ... truncated")
            elif torch.is_tensor(value):
                print(f"{key}: tensor {tuple(value.shape)}")
            else:
                print(f"{key}: {type(value).__name__}")
    else:
        print("Unexpected checkpoint format; stop for inspection.")

except Exception as e:
    print("\nLOAD BLOCKED")
    print(type(e).__name__ + ":", str(e)[:2500])
    print("Checkpoint was not modified.")

print("\nR21E COMPLETE")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

LOAD BLOCKED
UnpicklingError: Weights only load failed. In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
Please file an issue with the following so that we can make `weights_only=True` compatible with your use case: WeightsUnpickler error: 

Can only build Tensor, Parameter, OrderedDict or types allowlisted via `add_safe_globals`, but got <class 'numpy.dtypes.UInt32DType'>

Check the documentation of torch.load to learn more about types accepted by default with weights_only https://pytorch.org/docs/stable/generated/torch.load.html.
Checkpoint was not modified.

R21E COMPLETE


In [6]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import torch

ckpt_path = Path(
    "/content/drive/MyDrive/NoiSoi_Matching/"
    "baseline_model_v5/checkpoints/best.pt"
)

# Chỉ dùng với checkpoint nội bộ đáng tin cậy do dự án của bạn tạo.
checkpoint = torch.load(
    ckpt_path,
    map_location="cpu",
    weights_only=False,
    mmap=True,
)

print("Checkpoint loaded for inspection.")
print("Type:", type(checkpoint).__name__)

if isinstance(checkpoint, dict):
    print("\nTop-level keys:")
    for key, value in checkpoint.items():
        if isinstance(value, dict):
            print(f"{key}: dict, {len(value)} entries")
            tensor_items = [
                (k, v) for k, v in value.items()
                if torch.is_tensor(v)
            ]
            for k, tensor in tensor_items[:12]:
                print(f"  {k}: {tuple(tensor.shape)}")
        elif torch.is_tensor(value):
            print(f"{key}: tensor {tuple(value.shape)}")
        else:
            print(f"{key}: {type(value).__name__}")
else:
    print("Unexpected checkpoint format.")

print("\nInspection complete.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint loaded for inspection.
Type: dict

Top-level keys:
epoch: int
batch_idx: int
global_step: int
model_state: dict, 403 entries
  vision.embeddings.cls_token: (1, 1, 768)
  vision.embeddings.position_embeddings: (1, 197, 768)
  vision.embeddings.patch_embeddings.projection.weight: (768, 3, 16, 16)
  vision.embeddings.patch_embeddings.projection.bias: (768,)
  vision.layers.0.attention.q_proj.weight: (768, 768)
  vision.layers.0.attention.q_proj.bias: (768,)
  vision.layers.0.attention.k_proj.weight: (768, 768)
  vision.layers.0.attention.k_proj.bias: (768,)
  vision.layers.0.attention.v_proj.weight: (768, 768)
  vision.layers.0.attention.v_proj.bias: (768,)
  vision.layers.0.attention.o_proj.weight: (768, 768)
  vision.layers.0.attention.o_proj.bias: (768,)
optimizer_state: dict, 2 entries
scheduler_state: dict, 7 entries
best_val: float
history: dict

In [7]:

from collections import Counter, defaultdict
import torch

assert "checkpoint" in globals(), "Checkpoint chưa có trong RAM; hãy chạy lại cell đọc checkpoint."
assert isinstance(checkpoint, dict) and "model_state" in checkpoint

state = checkpoint["model_state"]

print("=" * 75)
print("R21F-A — TRAINING STATE")
print("=" * 75)
for key in ["epoch", "batch_idx", "global_step", "best_val"]:
    print(f"{key}: {checkpoint.get(key)}")
print("Total model state entries:", len(state))
print("Tensor entries:", sum(torch.is_tensor(v) for v in state.values()))

print("\n" + "=" * 75)
print("R21F-B — PARAMETER PREFIX COUNTS")
print("=" * 75)

prefix_counts = Counter(
    key.split(".")[0] for key in state
)
for prefix, count in prefix_counts.most_common():
    print(f"{prefix}: {count}")

print("\n" + "=" * 75)
print("R21F-C — ALL NON-mT5 PARAMETER NAMES AND SHAPES")
print("=" * 75)

for key, value in state.items():
    if not key.startswith("mt5."):
        shape = tuple(value.shape) if torch.is_tensor(value) else type(value).__name__
        print(f"{key}: {shape}")

print("\n" + "=" * 75)
print("R21F-D — mT5 PARAMETER SUMMARY")
print("=" * 75)

mt5_keys = [(k, v) for k, v in state.items() if k.startswith("mt5.")]
print("mT5 entries:", len(mt5_keys))

mt5_prefix_counts = Counter(
    ".".join(k.split(".")[:3]) for k, _ in mt5_keys
)
print("mT5 top-level groups:")
for prefix, count in mt5_prefix_counts.most_common(30):
    print(f"  {prefix}: {count}")

print("\nFirst 20 mT5 keys:")
for key, value in mt5_keys[:20]:
    shape = tuple(value.shape) if torch.is_tensor(value) else type(value).__name__
    print(f"  {key}: {shape}")

print("\nLast 20 mT5 keys:")
for key, value in mt5_keys[-20:]:
    shape = tuple(value.shape) if torch.is_tensor(value) else type(value).__name__
    print(f"  {key}: {shape}")

print("\nR21F COMPLETE")


R21F-A — TRAINING STATE
epoch: 5
batch_idx: 0
global_step: 1920
best_val: 0.424518172663671
Total model state entries: 403
Tensor entries: 403

R21F-B — PARAMETER PREFIX COUNTS
vision: 200
mt5: 192
conditioner: 3
projector: 2
concept_head: 2
negative_head: 2
attribute_head: 2

R21F-C — ALL NON-mT5 PARAMETER NAMES AND SHAPES
vision.embeddings.cls_token: (1, 1, 768)
vision.embeddings.position_embeddings: (1, 197, 768)
vision.embeddings.patch_embeddings.projection.weight: (768, 3, 16, 16)
vision.embeddings.patch_embeddings.projection.bias: (768,)
vision.layers.0.attention.q_proj.weight: (768, 768)
vision.layers.0.attention.q_proj.bias: (768,)
vision.layers.0.attention.k_proj.weight: (768, 768)
vision.layers.0.attention.k_proj.bias: (768,)
vision.layers.0.attention.v_proj.weight: (768, 768)
vision.layers.0.attention.v_proj.bias: (768,)
vision.layers.0.attention.o_proj.weight: (768, 768)
vision.layers.0.attention.o_proj.bias: (768,)
vision.layers.0.layernorm_before.weight: (768,)
vision.lay

In [8]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import ast
import json
import torch
from pathlib import Path

# ------------------------------------------------------------
# 1. Locate the original notebook and saved checkpoint
# ------------------------------------------------------------
notebook_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/25sep.ipynb"
)
checkpoint_path = Path(
    "/content/drive/MyDrive/NoiSoi_Matching/"
    "baseline_model_v5/checkpoints/best.pt"
)

assert notebook_path.exists(), f"Original notebook not found: {notebook_path}"
assert checkpoint_path.exists(), f"Checkpoint not found: {checkpoint_path}"

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

cells = notebook.get("cells", [])
sources = [
    "".join(cell.get("source", []))
    for cell in cells
    if cell.get("cell_type") == "code"
]

# Find the latest definitions to avoid the older V5 implementation.
class_nodes = {}
class_cell_indices = {}

for idx, source in enumerate(sources):
    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in tree.body:
        if isinstance(node, ast.ClassDef) and node.name in {
            "SoftStructuredConditioner", "V5Model"
        }:
            class_nodes[node.name] = node
            class_cell_indices[node.name] = idx

required = {"SoftStructuredConditioner", "V5Model"}
missing = required - set(class_nodes)

print("Notebook:", notebook_path.name)
print("Code cells:", len(sources))
print("Found class definitions:", class_cell_indices)

assert not missing, (
    f"Could not find classes: {sorted(missing)}. "
    "Stop rather than reconstructing a guessed architecture."
)

# ------------------------------------------------------------
# 2. Execute only relevant imports from the original notebook
#    and the exact latest class definitions.
# ------------------------------------------------------------
allowed_roots = {
    "torch", "transformers", "typing", "math",
    "numpy", "collections"
}
import_nodes = {}
for source in sources:
    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in tree.body:
        if isinstance(node, ast.Import):
            roots = [alias.name.split(".")[0] for alias in node.names]
        elif isinstance(node, ast.ImportFrom):
            roots = [(node.module or "").split(".")[0]]
        else:
            continue

        if roots and all(root in allowed_roots for root in roots):
            import_nodes[ast.dump(node)] = node

# Import statements first; then execute the two original class definitions.
body = list(import_nodes.values()) + [
    class_nodes["SoftStructuredConditioner"],
    class_nodes["V5Model"],
]
module = ast.Module(body=body, type_ignores=[])
exec(compile(module, str(notebook_path), "exec"), globals())

print("PASS: original V5 classes loaded from notebook source.")

# ------------------------------------------------------------
# 3. Load checkpoint if not already present in memory
#    This is the trusted checkpoint created by this project.
# ------------------------------------------------------------
if "checkpoint" not in globals() or not isinstance(checkpoint, dict):
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
        mmap=True,
    )

assert "model_state" in checkpoint, (
    f"Expected 'model_state'; found keys: {list(checkpoint.keys())}"
)

# ------------------------------------------------------------
# 4. Instantiate exact architecture and restore all weights
# ------------------------------------------------------------
model_v5 = V5Model()
result = model_v5.load_state_dict(
    checkpoint["model_state"],
    strict=True,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_v5 = model_v5.to(device)
model_v5.eval()

print("\n" + "=" * 70)
print("R21G — RESTORATION RESULT")
print("=" * 70)
print("load_state_dict:", result)
print("Checkpoint epoch:", checkpoint.get("epoch"))
print("Global step:", checkpoint.get("global_step"))
print("Best validation loss:", checkpoint.get("best_val"))
print("Device:", device)
print("Model parameters:", sum(p.numel() for p in model_v5.parameters()))
print("Model in eval mode:", not model_v5.training)
print("PASS: V5 restored with strict=True; no training performed.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Notebook: 25sep.ipynb
Code cells: 162
Found class definitions: {'SoftStructuredConditioner': 131, 'V5Model': 131}


ImportError: cannot import name 'MT5Tokenizer' from 'transformers' (/usr/local/lib/python3.13/dist-packages/transformers/__init__.py)

In [9]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import ast
import json
import torch
from pathlib import Path

notebook_path = Path("/content/drive/MyDrive/Colab Notebooks/25sep.ipynb")
checkpoint_path = Path(
    "/content/drive/MyDrive/NoiSoi_Matching/"
    "baseline_model_v5/checkpoints/best.pt"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

cells = notebook.get("cells", [])
code_cells = [
    "".join(c.get("source", []))
    for c in cells if c.get("cell_type") == "code"
]

# Locate the cell containing both original V5 class definitions.
target_source = None
target_index = None

for i, source in enumerate(code_cells):
    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    class_names = {
        node.name for node in tree.body
        if isinstance(node, ast.ClassDef)
    }
    if {"SoftStructuredConditioner", "V5Model"}.issubset(class_names):
        target_source = source
        target_index = i

assert target_source is not None, "Could not find both original V5 classes."

tree = ast.parse(target_source)
imports = []
definitions = []

for node in tree.body:
    if isinstance(node, (ast.Import, ast.ImportFrom)):
        # Preserve imports from the original class cell, but remove
        # the incompatible tokenizer import.
        if isinstance(node, ast.ImportFrom) and node.module == "transformers":
            kept = [
                alias for alias in node.names
                if alias.name != "MT5Tokenizer"
            ]
            if kept:
                imports.append(
                    ast.ImportFrom(
                        module=node.module,
                        names=kept,
                        level=node.level,
                    )
                )
        else:
            imports.append(node)

    elif isinstance(node, ast.ClassDef) and node.name in {
        "SoftStructuredConditioner", "V5Model"
    }:
        definitions.append(node)

assert len(definitions) == 2, (
    f"Expected two class definitions, found {len(definitions)}"
)

module = ast.Module(body=imports + definitions, type_ignores=[])
exec(compile(ast.fix_missing_locations(module), str(notebook_path), "exec"), globals())

print(f"PASS: loaded original classes from notebook code cell {target_index}.")
print("V5Model:", V5Model)
print("SoftStructuredConditioner:", SoftStructuredConditioner)

# Load the trusted checkpoint only if it is not already in memory.
if not isinstance(globals().get("checkpoint"), dict):
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
        mmap=True,
    )

assert "model_state" in checkpoint, (
    f"Unexpected checkpoint keys: {list(checkpoint.keys())}"
)

model_v5 = V5Model()
load_result = model_v5.load_state_dict(
    checkpoint["model_state"],
    strict=True,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_v5 = model_v5.to(device)
model_v5.eval()

print("\n" + "=" * 65)
print("R21G — RESTORATION RESULT")
print("=" * 65)
print("Strict loading:", load_result)
print("Checkpoint epoch:", checkpoint.get("epoch"))
print("Global step:", checkpoint.get("global_step"))
print("Best validation loss:", checkpoint.get("best_val"))
print("Device:", device)
print("Parameter count:", sum(p.numel() for p in model_v5.parameters()))
print("Eval mode:", not model_v5.training)
print("PASS: restored without training.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PASS: loaded original classes from notebook code cell 131.
V5Model: <class '__main__.V5Model'>
SoftStructuredConditioner: <class '__main__.SoftStructuredConditioner'>


config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

...

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.20GB            

model.safetensors: downloading bytes:           |  0.00B            


R21G — RESTORATION RESULT
Strict loading: <All keys matched successfully>
Checkpoint epoch: 5
Global step: 1920
Best validation loss: 0.424518172663671
Device: cuda
Parameter count: 387020219
Eval mode: True
PASS: restored without training.


In [11]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import json
from pathlib import Path

notebook_path = Path("/content/drive/MyDrive/Colab Notebooks/25sep.ipynb")
with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

code_cells = [
    "".join(cell.get("source", []))
    for cell in nb.get("cells", [])
    if cell.get("cell_type") == "code"
]

search_terms = [
    "test_loader",
    "class V5Dataset",
    "class",
    "AutoImageProcessor",
    "AutoTokenizer",
    "image_processor",
    "case_to_images",
    "pixel_values",
    "image_mask",
    "image_status",
    "v4_training_manifest.csv",
]

print("=" * 75)
print("R21H — ORIGINAL TEST PIPELINE SEARCH")
print("=" * 75)

for term in search_terms:
    matches = []
    for idx, source in enumerate(code_cells):
        if term.lower() in source.lower():
            matches.append(idx)
    print(f"{term}: {matches[-12:]}")

print("\n" + "=" * 75)
print("RELEVANT CELL CONTENTS")
print("=" * 75)

selected = set()
for idx, source in enumerate(code_cells):
    low = source.lower()
    score = sum(term.lower() in low for term in [
        "test_loader",
        "class V5Dataset",
        "AutoImageProcessor",
        "AutoTokenizer",
        "case_to_images",
        "image_status",
        "pixel_values",
        "image_mask",
    ])
    if score >= 2 or "test_loader" in low:
        selected.add(idx)

for idx in sorted(selected):
    source = code_cells[idx]
    print(f"\n{'=' * 25} CODE CELL {idx} {'=' * 25}")
    print(source[:9000])
    if len(source) > 9000:
        print("\n... CELL TRUNCATED ...")

print("\nR21H COMPLETE")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
R21H — ORIGINAL TEST PIPELINE SEARCH
test_loader: [25, 123, 139]
class V5Dataset: [139]
class: [104, 106, 115, 116, 119, 120, 122, 125, 131, 139, 144, 156]
AutoImageProcessor: [0, 1, 141, 142]
AutoTokenizer: [0, 1, 68, 120, 139, 141, 142]
image_processor: [60, 68, 69, 73, 119, 120, 122, 141, 142, 144, 156, 161]
case_to_images: [60, 65, 67, 69, 73, 119, 120, 122, 139, 144, 156, 161]
pixel_values: [120, 122, 123, 124, 126, 127, 131, 139, 140, 143, 144, 156]
image_mask: [119, 120, 122, 123, 124, 126, 127, 131, 139, 140, 143, 156]
image_status: [46, 66, 67, 69, 73, 113, 114, 119, 120, 139, 144, 156]
v4_training_manifest.csv: [119, 120, 139, 144, 145, 146, 147, 149, 153, 155, 156, 161]

RELEVANT CELL CONTENTS

========================= CODE CELL 0 =========================
import os
import json
import math
import random
from pathlib import Path
from collections im

In [13]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import json
from pathlib import Path

nb_path = Path("/content/drive/MyDrive/Colab Notebooks/25sep.ipynb")
assert nb_path.exists(), f"Không tìm thấy notebook: {nb_path}"

with nb_path.open("r", encoding="utf-8") as f:
    nb = json.load(f)

cells = nb.get("cells", [])
print("Notebook cells:", len(cells))

# In đúng các cell R21H đã xác định có test pipeline.
for idx in [25, 123, 139]:
    if idx >= len(cells):
        print(f"\nCELL {idx}: ngoài phạm vi notebook hiện tại")
        continue

    cell = cells[idx]
    src = cell.get("source", [])
    src = "".join(src) if isinstance(src, list) else str(src)

    print("\n" + "=" * 90)
    print(f"CELL INDEX {idx} | TYPE={cell.get('cell_type')} | CHARS={len(src)}")
    print("=" * 90)
    print(src)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Notebook cells: 162

CELL INDEX 25 | TYPE=code | CHARS=1266
# ============================================================
# AUDIT: CURRENT COLAB RUNTIME OBJECTS
# DO NOT MODIFY ANYTHING
# ============================================================

print("========== LOADER RUNTIME AUDIT ==========")

candidate_names = [
    "train_loader",
    "val_loader",
    "test_loader",
    "train_dataset",
    "val_dataset",
    "test_dataset",
    "train_dl",
    "val_dl",
    "dataset",
    "loader",
]

for name in candidate_names:
    if name in globals():
        obj = globals()[name]

        print(f"\n{name}")
        print(f"  type = {type(obj)}")

        try:
            print(f"  len  = {len(obj)}")
        except Exception:
            print("  len  = <not available>")


print("\n========== ALL *_LOADER / *_DATASET OBJECTS ==========")

for name, obj in gl

In [14]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import inspect
import json
from pathlib import Path

print("=" * 80)
print("V5 — FORWARD PATH AUDIT (READ-ONLY)")
print("=" * 80)

# 1. Find the restored model without assuming its variable name.
model = None
model_name = None

for name in ["model_v5", "v5", "model"]:
    obj = globals().get(name)
    if obj is not None and hasattr(obj, "state_dict"):
        model = obj
        model_name = name
        break

if model is None:
    print("MODEL: No restored model found under model_v5 / v5 / model")
else:
    print("MODEL variable:", model_name)
    print("MODEL class:", type(model).__name__)
    print("MODEL device:", next(model.parameters()).device)
    print("\n--- Exact runtime forward() ---")
    try:
        print(inspect.getsource(type(model).forward))
    except Exception as e:
        print("Runtime source unavailable:", repr(e))

# 2. Report existing test pipeline objects, without creating them.
print("\n--- Existing runtime objects ---")
for name in [
    "test_loader_v5", "test_dataset_v5",
    "test_cases", "selected_test",
    "structured_targets_v5", "processor", "tokenizer"
]:
    obj = globals().get(name)
    if obj is None:
        print(f"{name}: MISSING")
    else:
        try:
            detail = f"type={type(obj).__name__}, len={len(obj)}"
        except Exception:
            detail = f"type={type(obj).__name__}"
        print(f"{name}: {detail}")

# 3. Confirm relevant model modules.
if model is not None:
    print("\n--- Model modules ---")
    for name in [
        "vision", "projector", "conditioner",
        "concept_head", "negative_head",
        "attribute_head", "mt5"
    ]:
        obj = getattr(model, name, None)
        print(f"{name}: {type(obj).__name__ if obj is not None else 'MISSING'}")

print("\nREAD-ONLY AUDIT COMPLETE — no training or checkpoint writes.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
V5 — FORWARD PATH AUDIT (READ-ONLY)
MODEL variable: model_v5
MODEL class: V5Model
MODEL device: cuda:0

--- Exact runtime forward() ---
Runtime source unavailable: OSError('lineno is out of bounds')

--- Existing runtime objects ---
test_loader_v5: MISSING
test_dataset_v5: MISSING
test_cases: MISSING
selected_test: MISSING
structured_targets_v5: MISSING
processor: MISSING
tokenizer: MISSING

--- Model modules ---
vision: ViTModel
projector: Linear
conditioner: SoftStructuredConditioner
concept_head: Linear
negative_head: Linear
attribute_head: Linear
mt5: MT5ForConditionalGeneration

READ-ONLY AUDIT COMPLETE — no training or checkpoint writes.


In [15]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import ast
from pathlib import Path

nb_path = Path("/content/drive/MyDrive/Colab Notebooks/25sep.ipynb")
assert nb_path.exists(), f"Notebook not found: {nb_path}"

import json
with nb_path.open("r", encoding="utf-8") as f:
    nb = json.load(f)

found = False

for idx, cell in enumerate(nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    src = cell.get("source", [])
    src = "".join(src) if isinstance(src, list) else str(src)

    try:
        tree = ast.parse(src)
    except SyntaxError:
        continue

    for node in ast.walk(tree):
        if isinstance(node, ast.ClassDef) and node.name == "V5Model":
            methods = [
                n for n in node.body
                if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))
                and n.name == "forward"
            ]

            if methods:
                method = methods[0]
                print("=" * 80)
                print(f"V5Model.forward() — notebook cell {idx}")
                print("=" * 80)
                print(ast.get_source_segment(src, method))
                found = True
                break

    if found:
        break

assert found, "Could not locate V5Model.forward() in the original notebook."
print("\nREAD-ONLY: no model execution, training, or checkpoint writes.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
V5Model.forward() — notebook cell 81
def forward(
        self,
        pixel_values,
        image_mask,
        labels=None,
        concept_targets=None,
        negative_targets=None,
        attribute_targets=None,
    ):
        B, N, C, H, W = pixel_values.shape

        # ----------------------------------------------------------
        # Flatten images for ViT
        # ----------------------------------------------------------
        x = pixel_values.view(
            B * N,
            C,
            H,
            W
        )

        vision_outputs = self.vision(
            pixel_values=x
        )

        cls = vision_outputs.last_hidden_state[:, 0]
        cls = cls.view(B, N, -1)

        # ----------------------------------------------------------
        # Mask padded images
        # -----------------------------------------------------

In [16]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import ast
import json
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import ViTImageProcessor

print("=" * 80)
print("V5 STRUCTURED-HEAD TEST EVALUATION — READ ONLY")
print("=" * 80)

# ------------------------------------------------------------
# 1. Validate restored model; do not reload or modify checkpoint
# ------------------------------------------------------------
assert "model_v5" in globals(), "model_v5 is missing."
model = model_v5

for name in [
    "vision", "projector", "concept_head",
    "negative_head", "attribute_head"
]:
    assert hasattr(model, name), f"Missing model module: {name}"

device = next(model.parameters()).device
assert device.type == "cuda", f"Expected CUDA model, got {device}"

model.eval()
print("Model:", type(model).__name__)
print("Device:", device)
print("Mode: eval; no optimizer; no training")

# ------------------------------------------------------------
# 2. Load the exact persisted manifests and structured targets
# ------------------------------------------------------------
ROOT = Path("/content/drive/MyDrive/NoiSoi_Matching")

V4_MANIFEST = ROOT / "baseline_model_v4/v4_training_manifest.csv"
IMAGE_MANIFEST = ROOT / "final_manifest/final_dataset_manifest.csv"
STRUCTURED_CSV = ROOT / "v3_ontology_v2/v3_structured_targets_v2_2.csv"
ONTOLOGY_CSV = ROOT / "baseline_model_v4/diagnostics/frozen_concept_ontology_mapping.csv"

for p in [V4_MANIFEST, IMAGE_MANIFEST, STRUCTURED_CSV]:
    assert p.exists(), f"Required file missing: {p}"

v4_df = pd.read_csv(V4_MANIFEST)
image_df = pd.read_csv(IMAGE_MANIFEST)
structured_df = pd.read_csv(STRUCTURED_CSV)

assert len(v4_df) == 7606
assert len(structured_df) == 7606

test_cases = (
    v4_df.loc[v4_df["split"].astype(str).eq("test"), "case_id"]
    .astype(str).tolist()
)
assert len(test_cases) == 712
assert len(set(test_cases)) == 712

print("\nManifest rows:", len(v4_df))
print("Test cases:", len(test_cases))
print("Image manifest rows:", len(image_df))

# ------------------------------------------------------------
# 3. Reproduce original NORMAL-image selection for test
# ------------------------------------------------------------
assert "case_id" in image_df.columns

IMAGE_COL = (
    "image_path" if "image_path" in image_df.columns
    else "pdf_path" if "pdf_path" in image_df.columns
    else None
)
assert IMAGE_COL is not None, "No image_path/pdf_path column."

if "image_status" in image_df.columns:
    normal_df = image_df[
        image_df["image_status"].astype(str).str.upper().eq("NORMAL")
    ].copy()
else:
    normal_df = image_df.copy()

def numeric_order(x):
    try:
        return float(x)
    except Exception:
        return 0.0

case_to_images = {}

for case_id, group in normal_df.groupby("case_id", sort=False):
    case_id = str(case_id)
    rows = group.copy()

    if "image_order" in rows.columns:
        rows["_order"] = rows["image_order"].map(numeric_order)
        rows = rows.sort_values("_order")

    paths = rows[IMAGE_COL].dropna().astype(str).tolist()
    resolved = []

    for p in paths:
        pp = Path(p)

        if pp.exists():
            resolved.append(str(pp))
            continue

        candidates = [Path("/content") / p, ROOT / p]
        match = next((c for c in candidates if c.exists()), None)

        if match is not None:
            resolved.append(str(match))

    case_to_images[case_id] = resolved[:8]

missing_images = [
    c for c in test_cases
    if not case_to_images.get(c)
]

print("\nNORMAL images:", len(normal_df))
print("Test cases without usable image paths:", len(missing_images))

if missing_images:
    print("Examples:", missing_images[:10])
    raise RuntimeError(
        "Some test cases have no resolved NORMAL images. "
        "Stopping rather than silently changing the test set."
    )

# ------------------------------------------------------------
# 4. Load the same ViT image processor used by training
# ------------------------------------------------------------
processor = ViTImageProcessor.from_pretrained(
    "google/vit-base-patch16-224-in21k"
)

class StructuredTestDataset(Dataset):
    def __init__(self, case_ids, image_map, processor):
        self.case_ids = list(case_ids)
        self.image_map = image_map
        self.processor = processor

    def __len__(self):
        return len(self.case_ids)

    def __getitem__(self, idx):
        case_id = self.case_ids[idx]
        images = []

        for path in self.image_map[case_id]:
            try:
                with Image.open(path) as im:
                    images.append(im.convert("RGB").copy())
            except Exception as e:
                raise RuntimeError(
                    f"Image read failed for case={case_id}, "
                    f"path={path}: {e}"
                )

        if not images:
            raise RuntimeError(f"No images for case {case_id}")

        pixels = self.processor(
            images=images,
            return_tensors="pt"
        )["pixel_values"]

        return {"case_id": case_id, "pixel_values": pixels}

def collate_test(batch):
    B = len(batch)
    max_n = max(item["pixel_values"].shape[0] for item in batch)

    pixels = torch.zeros(B, max_n, 3, 224, 224, dtype=torch.float32)
    mask = torch.zeros(B, max_n, dtype=torch.float32)
    case_ids = []

    for i, item in enumerate(batch):
        n = item["pixel_values"].shape[0]
        pixels[i, :n] = item["pixel_values"]
        mask[i, :n] = 1.0
        case_ids.append(item["case_id"])

    return {
        "case_id": case_ids,
        "pixel_values": pixels,
        "image_mask": mask
    }

test_dataset = StructuredTestDataset(test_cases, case_to_images, processor)
test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_test
)

assert len(test_dataset) == 712
assert len(test_loader) == 178

print("Test dataset:", len(test_dataset))
print("Test batches:", len(test_loader))

# ------------------------------------------------------------
# 5. Parse structured ground truth in persisted vector order
# ------------------------------------------------------------
def parse_vector(value):
    if value is None:
        return []

    if isinstance(value, float) and np.isnan(value):
        return []

    s = str(value).strip()
    if not s:
        return []

    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, (list, tuple, np.ndarray)):
            return [float(x) for x in obj]
    except Exception:
        pass

    try:
        obj = json.loads(s)
        if isinstance(obj, list):
            return [float(x) for x in obj]
    except Exception:
        pass

    s = s.replace("[", "").replace("]", "")
    try:
        return [float(x.strip()) for x in s.split(",") if x.strip()]
    except Exception:
        return []

required = [
    "case_id", "concept_vector_str",
    "negative_vector_str", "attribute_vector_str"
]
assert all(c in structured_df.columns for c in required)

structured_df["case_id"] = structured_df["case_id"].astype(str)
assert structured_df["case_id"].is_unique

target_map = {}
for _, row in structured_df.iterrows():
    c = row["case_id"]
    concept = parse_vector(row["concept_vector_str"])
    negative = parse_vector(row["negative_vector_str"])
    attribute = parse_vector(row["attribute_vector_str"])

    if (len(concept), len(negative), len(attribute)) != (48, 5, 6):
        raise ValueError(
            f"Invalid target dimensions for {c}: "
            f"{len(concept)}, {len(negative)}, {len(attribute)}"
        )

    target_map[c] = (
        np.asarray(concept, dtype=np.float32),
        np.asarray(negative, dtype=np.float32),
        np.asarray(attribute, dtype=np.float32)
    )

assert all(c in target_map for c in test_cases)

y_concept = np.stack([target_map[c][0] for c in test_cases])
y_negative = np.stack([target_map[c][1] for c in test_cases])
y_attribute = np.stack([target_map[c][2] for c in test_cases])

assert y_concept.shape == (712, 48)
assert y_negative.shape == (712, 5)
assert y_attribute.shape == (712, 6)

# ------------------------------------------------------------
# 6. Exact structured path from V5Model.forward()
#    Deliberately bypass mT5; no labels/decoder required.
# ------------------------------------------------------------
pred_concept, pred_negative, pred_attribute = [], [], []
evaluated_ids = []

with torch.inference_mode():
    for batch_idx, batch in enumerate(test_loader):
        pixels = batch["pixel_values"].to(device, non_blocking=True)
        image_mask = batch["image_mask"].to(device, non_blocking=True)

        B, N, C, H, W = pixels.shape
        flat = pixels.reshape(B * N, C, H, W)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
            enabled=(device.type == "cuda")
        ):
            vision_outputs = model.vision(pixel_values=flat)
            cls = vision_outputs.last_hidden_state[:, 0]
            cls = cls.reshape(B, N, -1)

            mask = image_mask.unsqueeze(-1)
            cls = cls * mask
            pooled = cls.sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)

            visual_hidden = model.projector(pooled)

            concept_logits = model.concept_head(visual_hidden)
            negative_logits = model.negative_head(visual_hidden)
            attribute_logits = model.attribute_head(visual_hidden)

        pred_concept.append(torch.sigmoid(concept_logits.float()).cpu().numpy())
        pred_negative.append(torch.sigmoid(negative_logits.float()).cpu().numpy())
        pred_attribute.append(torch.sigmoid(attribute_logits.float()).cpu().numpy())
        evaluated_ids.extend(batch["case_id"])

        if (batch_idx + 1) % 25 == 0 or (batch_idx + 1) == len(test_loader):
            print(f"Evaluated batches: {batch_idx + 1}/{len(test_loader)}")

p_concept = np.concatenate(pred_concept)
p_negative = np.concatenate(pred_negative)
p_attribute = np.concatenate(pred_attribute)

assert evaluated_ids == test_cases, "Test case order/identity mismatch."
assert p_concept.shape == (712, 48)
assert p_negative.shape == (712, 5)
assert p_attribute.shape == (712, 6)

# ------------------------------------------------------------
# 7. Threshold 0.5 metrics: per-label + group summaries
# ------------------------------------------------------------
def metrics_for_group(y, p, names, group_name, threshold=0.5):
    y = y.astype(np.int32)
    pred = (p >= threshold).astype(np.int32)

    tp = ((y == 1) & (pred == 1)).sum(axis=0)
    fp = ((y == 0) & (pred == 1)).sum(axis=0)
    fn = ((y == 1) & (pred == 0)).sum(axis=0)
    support = y.sum(axis=0)

    precision = np.divide(
        tp, tp + fp, out=np.zeros_like(tp, dtype=float), where=(tp + fp) > 0
    )
    recall = np.divide(
        tp, tp + fn, out=np.zeros_like(tp, dtype=float), where=(tp + fn) > 0
    )
    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(precision),
        where=(precision + recall) > 0
    )

    total_tp, total_fp, total_fn = tp.sum(), fp.sum(), fn.sum()
    micro_p = total_tp / (total_tp + total_fp) if total_tp + total_fp else 0.0
    micro_r = total_tp / (total_tp + total_fn) if total_tp + total_fn else 0.0
    micro_f1 = (
        2 * micro_p * micro_r / (micro_p + micro_r)
        if micro_p + micro_r else 0.0
    )

    active = support > 0
    summary = {
        "group": group_name,
        "labels": y.shape[1],
        "positive_label_count": int(active.sum()),
        "micro_precision": micro_p,
        "micro_recall": micro_r,
        "micro_f1": micro_f1,
        "macro_f1_all_labels": float(f1.mean()),
        "macro_f1_labels_with_positives": (
            float(f1[active].mean()) if active.any() else 0.0
        ),
        "ground_truth_positives": int(y.sum()),
        "predicted_positives": int(pred.sum()),
    }

    per_label = pd.DataFrame({
        "group": group_name,
        "label_index": np.arange(y.shape[1]),
        "ground_truth_positives": support,
        "predicted_positives": pred.sum(axis=0),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "precision": precision,
        "recall": recall,
        "F1": f1,
    })

    return summary, per_label

# Keep vector indices explicit; ontology mapping is audited separately below.
concept_names = [f"concept_{i:02d}" for i in range(48)]
negative_names = [
    "no_abnormal_ent",
    "no_abnormal_external_middle_ear",
    "no_abnormal_nose_sinus",
    "no_bleeding",
    "no_foreign_body",
]
attribute_names = [
    "acute", "bilateral", "chronic", "left", "post_surgery", "right"
]

summaries, per_label_tables = [], []

for group_name, y, p, names in [
    ("concept", y_concept, p_concept, concept_names),
    ("negative", y_negative, p_negative, negative_names),
    ("attribute", y_attribute, p_attribute, attribute_names),
]:
    summary, per_label = metrics_for_group(y, p, names, group_name)
    summaries.append(summary)
    per_label["label"] = [names[i] for i in per_label["label_index"]]
    per_label_tables.append(per_label)

print("\n" + "=" * 80)
print("GROUP SUMMARY — threshold = 0.50")
print("=" * 80)
print(pd.DataFrame(summaries).round(4).to_string(index=False))

print("\n" + "=" * 80)
print("PER-LABEL METRICS")
print("=" * 80)
all_per_label = pd.concat(per_label_tables, ignore_index=True)
print(
    all_per_label[
        ["group", "label", "ground_truth_positives", "predicted_positives",
         "TP", "FP", "FN", "precision", "recall", "F1"]
    ].round(4).to_string(index=False)
)

# Show mapping schema so concept indices can be translated safely later.
if ONTOLOGY_CSV.exists():
    ontology = pd.read_csv(ONTOLOGY_CSV)
    print("\n" + "=" * 80)
    print("FROZEN ONTOLOGY MAPPING — schema/sample only")
    print("=" * 80)
    print("Shape:", ontology.shape)
    print("Columns:", ontology.columns.tolist())
    print(ontology.head(8).to_string(index=False))
else:
    print("\nOntology mapping file not found; concept labels remain indexed.")

print("\nPASS: 712 test cases evaluated.")
print("PASS: 48 concept + 5 negative + 6 attribute logits evaluated.")
print("PASS: No training, optimizer step, or checkpoint write.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
V5 STRUCTURED-HEAD TEST EVALUATION — READ ONLY
Model: V5Model
Device: cuda:0
Mode: eval; no optimizer; no training


/tmp/ipykernel_694/1696204890.py:53: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  image_df = pd.read_csv(IMAGE_MANIFEST)



Manifest rows: 7606
Test cases: 712
Image manifest rows: 76405

NORMAL images: 76216
Test cases without usable image paths: 0


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

Test dataset: 712
Test batches: 178


AssertionError: 

In [17]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import pandas as pd

print("=" * 80)
print("STRUCTURED TARGET SCHEMA CHECK — READ ONLY")
print("=" * 80)

print("STRUCTURED_CSV:", STRUCTURED_CSV)
print("File exists:", os.path.isfile(STRUCTURED_CSV))

if os.path.isfile(STRUCTURED_CSV):
    schema_df = pd.read_csv(STRUCTURED_CSV, nrows=5, low_memory=False)

    print("\nColumns actually present:")
    for i, col in enumerate(schema_df.columns):
        print(f"{i:02d}. {col!r}")

    print("\nColumns required by the evaluation cell:")
    required = [
        "case_id",
        "concept_vector_str",
        "negative_vector_str",
        "attribute_vector_str",
    ]
    for col in required:
        print(f"{col}: {'FOUND' if col in schema_df.columns else 'MISSING'}")

    print("\nFirst 3 rows:")
    display(schema_df.head(3))

    print("\nExisting structured_df in memory:")
    if "structured_df" in globals():
        print("Shape:", structured_df.shape)
        print("Columns:", structured_df.columns.tolist())
    else:
        print("No structured_df variable currently exists.")
else:
    print("ERROR: The configured structured-target CSV path does not exist.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
STRUCTURED TARGET SCHEMA CHECK — READ ONLY
STRUCTURED_CSV: /content/drive/MyDrive/NoiSoi_Matching/v3_ontology_v2/v3_structured_targets_v2_2.csv
File exists: True

Columns actually present:
00. 'case_id'
01. 'patient_group_id'
02. 'split'
03. 'pdf_path'
04. 'ho_ten'
05. 'nam_sinh'
06. 'gioi_tinh_clean'
07. 'ly_do_noi_soi'
08. 'tai'
09. 'hoc_mui'
10. 'hong_mui'
11. 'hong_thanh_quan'
12. 'hong_mieng'
13. 'ket_luan'
14. 'phan_biet'
15. 'de_nghi'
16. 'target_norm'
17. 'concepts'
18. 'attributes'
19. 'num_concepts'
20. 'num_attributes'
21. 'concept__viem_ong_tai_ngoai'
22. 'concept__nhot_ong_tai_ngoai'
23. 'concept__chan_thuong_ong_tai_ngoai'
24. 'concept__viem_tai_giua'
25. 'concept__viem_mang_nhi'
26. 'concept__thung_mang_nhi'
27. 'concept__xep_mang_nhi'
28. 'concept__ray_tai'
29. 'concept__viem_tai_xuong_chum'
30. 'concept__hau_phau_va_nhi'
31. 'concept__di_vat_

,case_id,patient_group_id,split,pdf_path,ho_ten,nam_sinh,gioi_tinh_clean,ly_do_noi_soi,tai,hoc_mui,...,negative__no_abnormal_ent,negative__no_bleeding,negative__no_foreign_body,concepts_v2_str,negative_findings_str,attributes_v2_str,num_concepts_v2,num_negative_findings,num_attributes_v2,has_structured_label
0,10000.10000.0.10014,NGUYEN TRUONG TAN SANG_2015,train,/content/drive/MyDrive/Hồ sơ bệnh/Hình nội soi...,NGUYỄN TRƯƠNG TẤN SANG,2015,NAM,HỈ MŨI LỎNG,NaN,"Cuốn mũi: hồng, xuất tiết nhầy trong. Khe mũi:...",...,0,0,0,viem_mui,NaN,chronic,1,0,1,True
1,10001.10001.0.10015,INARI ANH TUAN_2014,train,/content/drive/MyDrive/Hồ sơ bệnh/Hình nội soi...,INARI ANH TUẤN,2014,NAM,ĐAU TAI BÊN TRÁI,P: màng nhĩ bóng sáng. T: màng nhĩ bóng sáng.,Cuốn mũi: xuất tiết nhầy trắng Khe mũi: thoáng...,...,0,0,0,viem_mui,NaN,NaN,1,0,0,True
2,10002.10002.0.10016,BUI NGOC BAO CHAU_2016,train,/content/drive/MyDrive/Hồ sơ bệnh/Hình nội soi...,BÙI NGỌC BẢO CHÂU,2016,NỮ,CHẢY MŨI LỎNG,NaN,"Cuốn mũi: nhợt, gồ ghề nhẹ, xuất tiết nhầy trắ...",...,0,0,0,viem_mui,NaN,chronic,1,0,1,True



Existing structured_df in memory:
Shape: (7606, 102)
Columns: ['case_id', 'patient_group_id', 'split', 'pdf_path', 'ho_ten', 'nam_sinh', 'gioi_tinh_clean', 'ly_do_noi_soi', 'tai', 'hoc_mui', 'hong_mui', 'hong_thanh_quan', 'hong_mieng', 'ket_luan', 'phan_biet', 'de_nghi', 'target_norm', 'concepts', 'attributes', 'num_concepts', 'num_attributes', 'concept__viem_ong_tai_ngoai', 'concept__nhot_ong_tai_ngoai', 'concept__chan_thuong_ong_tai_ngoai', 'concept__viem_tai_giua', 'concept__viem_mang_nhi', 'concept__thung_mang_nhi', 'concept__xep_mang_nhi', 'concept__ray_tai', 'concept__viem_tai_xuong_chum', 'concept__hau_phau_va_nhi', 'concept__di_vat_tai', 'concept__viem_mui', 'concept__viem_mui_xoang', 'concept__viem_xoang', 'concept__polyp_mui', 'concept__chay_mau_mui', 'concept__dich_vat_mui', 'concept__hoc_xuong_ca', 'concept__di_vat_hong_thanh_quan', 'concept__di_vat', 'concept__hau_phau_mui_xoang', 'concept__tien_dinh_mui', 'concept__lech_vach_ngan', 'concept__viem_va', 'concept__qua_phat_

In [18]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import json, os, re

print("=" * 80)
print("V5 LABEL-ORDER AUDIT — READ ONLY")
print("=" * 80)

# 1. Actual head dimensions
for name in ["concept_head", "negative_head", "attribute_head"]:
    head = getattr(model_v5, name, None)
    if head is None:
        print(f"{name}: NOT FOUND on model_v5")
    else:
        print(f"{name}: out_features={getattr(head, 'out_features', 'unknown')}")

# 2. Candidate labels in the structured target CSV
for prefix in ["concept__", "negative__", "attribute__"]:
    cols = [c for c in structured_df.columns if c.startswith(prefix)]
    print(f"\n{prefix} ({len(cols)} columns):")
    print(cols)

# 3. Check whether the training label-order variables remain in memory
print("\n" + "=" * 80)
print("LABEL VARIABLES STILL IN MEMORY")
print("=" * 80)

for name in sorted(globals()):
    if re.search(r"(concept|negative|attribute).*(cols|columns|vocab|labels|names|keys|targets)", name, re.I):
        value = globals()[name]
        if isinstance(value, (list, tuple)):
            print(f"{name}: length={len(value)}; values={list(value)[:70]}")
        elif hasattr(value, "shape"):
            print(f"{name}: shape={value.shape}")
        else:
            print(f"{name}: type={type(value).__name__}")

# 4. Inspect the original notebook cell containing V5 training code
notebook_path = "/content/drive/MyDrive/Colab Notebooks/25sep.ipynb"
print("\nNotebook exists:", os.path.isfile(notebook_path))

if os.path.isfile(notebook_path):
    with open(notebook_path, "r", encoding="utf-8") as f:
        nb = json.load(f)

    for idx, cell in enumerate(nb.get("cells", [])):
        src = "".join(cell.get("source", []))
        if cell.get("cell_type") == "code" and (
            "class V5Dataset" in src or "class V5Model" in src
        ):
            print(f"\n--- Notebook cell {idx}: relevant label/target lines ---")
            lines = src.splitlines()
            patterns = re.compile(
                r"concept.*(col|vocab|label|target|vector)|"
                r"negative.*(col|vocab|label|target|vector)|"
                r"attribute.*(col|vocab|label|target|vector)|"
                r"structured_df|parse_vector|concept_head|negative_head|attribute_head",
                re.I
            )
            hits = [i for i, line in enumerate(lines) if patterns.search(line)]
            shown = set()
            for i in hits:
                for j in range(max(0, i-2), min(len(lines), i+3)):
                    if j not in shown:
                        print(f"{j+1:04d}: {lines[j]}")
                        shown.add(j)
                print("...")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
V5 LABEL-ORDER AUDIT — READ ONLY
concept_head: out_features=48
negative_head: out_features=5
attribute_head: out_features=6

concept__ (54 columns):
['concept__viem_ong_tai_ngoai', 'concept__nhot_ong_tai_ngoai', 'concept__chan_thuong_ong_tai_ngoai', 'concept__viem_tai_giua', 'concept__viem_mang_nhi', 'concept__thung_mang_nhi', 'concept__xep_mang_nhi', 'concept__ray_tai', 'concept__viem_tai_xuong_chum', 'concept__hau_phau_va_nhi', 'concept__di_vat_tai', 'concept__viem_mui', 'concept__viem_mui_xoang', 'concept__viem_xoang', 'concept__polyp_mui', 'concept__chay_mau_mui', 'concept__dich_vat_mui', 'concept__hoc_xuong_ca', 'concept__di_vat_hong_thanh_quan', 'concept__di_vat', 'concept__hau_phau_mui_xoang', 'concept__tien_dinh_mui', 'concept__lech_vach_ngan', 'concept__viem_va', 'concept__qua_phat_va', 'concept__viem_amidan', 'concept__amidan_qua_phat', 'concept__vi

In [19]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import json, re, os
import pandas as pd

NB_PATH = "/content/drive/MyDrive/Colab Notebooks/25sep.ipynb"
CSV_PATH = STRUCTURED_CSV

print("=" * 80)
print("TRACE V5 TARGET GENERATION — READ ONLY")
print("=" * 80)

with open(NB_PATH, "r", encoding="utf-8") as f:
    nb = json.load(f)

# Search every code cell for vector creation, label ordering, and target construction.
patterns = re.compile(
    r"concept_vector_str|negative_vector_str|attribute_vector_str|"
    r"structured_targets_v5|required_structured_cols|"
    r"concept_cols|negative_cols|attribute_cols|"
    r"concept_targets|negative_targets|attribute_targets|"
    r"concepts_v2_str|negative_findings_str|attributes_v2_str|"
    r"concept__viem_ong_tai_ngoai",
    re.I
)

for idx, cell in enumerate(nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue
    src = "".join(cell.get("source", []))
    lines = src.splitlines()
    hits = [i for i, line in enumerate(lines) if patterns.search(line)]
    if hits:
        print(f"\n{'='*25} CELL {idx} {'='*25}")
        shown = set()
        for i in hits:
            for j in range(max(0, i-3), min(len(lines), i+4)):
                if j not in shown:
                    print(f"{j+1:04d}: {lines[j]}")
                    shown.add(j)
            print("...")

# Check whether the CSV's existing structured columns can support the expected dimensions.
df = pd.read_csv(CSV_PATH, nrows=5, low_memory=False)
concept_cols = [c for c in df.columns if c.startswith("concept__")]
negative_cols = [c for c in df.columns if c.startswith("negative__")]
attribute_cols = [c for c in df.columns if c.startswith("attribute__")]

print("\n" + "=" * 80)
print("CURRENT CSV COUNTS")
print("=" * 80)
print("Concept columns:", len(concept_cols))
print("Negative columns:", len(negative_cols))
print("Attribute columns:", len(attribute_cols))
print("Model head dimensions: 48 / 5 / 6")
print("\nExisting serialized fields:")
for c in ["concepts_v2_str", "negative_findings_str", "attributes_v2_str"]:
    print(f"\n{c}:")
    if c in df.columns:
        print(df[c].head(5).tolist())
    else:
        print("MISSING")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TRACE V5 TARGET GENERATION — READ ONLY

========================= CELL 1 =========================
0742: 
0743:     def forward(
0744:         self,
0745:         concept_targets,
0746:         negative_targets,
0747:         attribute_targets,
0748:     ):
...
0749: 
...
0750:         device = concept_targets.device
...
0751: 
0752:         concept_ids = (
0753:             concept_targets
...
0754:             .long()
0755:             .clamp(0, 1)
0756:         )
...
0757: 
0758:         negative_ids = (
0759:             negative_targets
0760:             .long()
0761:             .clamp(0, 1)
0762:         )
...
0763: 
0764:         attribute_ids = (
0765:             attribute_targets
0766:             .long()
0767:             .clamp(0, 1)
0768:         )
...
0850:             structured_tokens
0851:         )
0852: 
0853:         B = concept_targets.s

In [20]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/NoiSoi_Matching")
V4_PATH = ROOT / "baseline_model_v4/v4_training_manifest.csv"
STRUCT_PATH = ROOT / "v3_ontology_v2/v3_structured_targets_v2_2.csv"
CKPT_PATH = ROOT / "baseline_model_v5/checkpoints/best.pt"

v4 = pd.read_csv(V4_PATH)
structured = pd.read_csv(STRUCT_PATH)

for name, df in [("V4 manifest", v4), ("Structured CSV", structured)]:
    print(f"\n===== {name} =====")
    print("Shape:", df.shape)
    print("Case IDs unique:", df["case_id"].nunique() if "case_id" in df else "MISSING")
    for col in ["concept_vector_str", "negative_vector_str",
                "attribute_vector_str", "structured_loss_mask"]:
        print(f"{col}: {'FOUND' if col in df.columns else 'MISSING'}")
    print("Concept columns:", sum(c.startswith("concept__") for c in df.columns))
    print("Negative columns:", sum(c.startswith("negative__") for c in df.columns))
    print("Attribute columns:", sum(c.startswith("attribute__") for c in df.columns))

print("\n===== CASE-ID OVERLAP =====")
if "case_id" in v4 and "case_id" in structured:
    a = set(v4["case_id"].astype(str))
    b = set(structured["case_id"].astype(str))
    print("V4 cases:", len(a))
    print("Structured cases:", len(b))
    print("Overlap:", len(a & b))
    print("Only V4:", len(a - b))
    print("Only Structured:", len(b - a))

print("\n===== CHECKPOINT FILE =====")
print("Exists:", CKPT_PATH.exists())
if CKPT_PATH.exists():
    print("Size (MB):", round(CKPT_PATH.stat().st_size / 1024**2, 2))
    print("Modified:", pd.Timestamp(CKPT_PATH.stat().st_mtime, unit="s"))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

===== V4 manifest =====
Shape: (7606, 15)
Case IDs unique: 7606
concept_vector_str: FOUND
negative_vector_str: FOUND
attribute_vector_str: FOUND
structured_loss_mask: FOUND
Concept columns: 0
Negative columns: 0
Attribute columns: 0

===== Structured CSV =====
Shape: (7606, 102)
Case IDs unique: 7606
concept_vector_str: MISSING
negative_vector_str: MISSING
attribute_vector_str: MISSING
structured_loss_mask: MISSING
Concept columns: 54
Negative columns: 5
Attribute columns: 6

===== CASE-ID OVERLAP =====
V4 cases: 7606
Structured cases: 7606
Overlap: 7606
Only V4: 0
Only Structured: 0

===== CHECKPOINT FILE =====
Exists: True
Size (MB): 4425.08
Modified: 2026-09-29 04:20:44


In [21]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import json
from pathlib import Path

nb_path = Path("/content/drive/MyDrive/Colab Notebooks/25sep.ipynb")
print("Notebook exists:", nb_path.exists())

if nb_path.exists():
    nb = json.loads(nb_path.read_text(encoding="utf-8"))
    cells = nb.get("cells", [])
    print("Total cells:", len(cells))

    for idx in [123, 139]:
        if idx >= len(cells):
            print(f"\nCell {idx}: OUT OF RANGE")
            continue

        cell = cells[idx]
        source = "".join(cell.get("source", []))
        outputs = cell.get("outputs", [])

        print(f"\n{'=' * 70}")
        print(f"CELL {idx}")
        print("Execution count:", cell.get("execution_count"))
        print("Output count:", len(outputs))
        print("--- Relevant source lines ---")

        for line_no, line in enumerate(source.splitlines(), start=1):
            if any(term in line.lower() for term in [
                "read_csv", "structured_df", "v4_df",
                "concept_vector_str", "structured_targets_v5",
                "assert col in", "epoch", "best.pt",
                "checkpoint", "training"
            ]):
                print(f"{line_no:04d}: {line[:240]}")

        print("--- Saved outputs (truncated) ---")
        for out_idx, out in enumerate(outputs):
            data = out.get("data", {})
            text_out = out.get("text", [])

            if "text/plain" in data:
                payload = data["text/plain"]
                text_out = payload if isinstance(payload, list) else [payload]

            text_out = "".join(text_out)
            if text_out.strip():
                print(f"[Output {out_idx}]")
                print(text_out[:2500])

            if out.get("output_type") == "error":
                print("ERROR:", out.get("ename"), out.get("evalue"))
else:
    print("Không tìm thấy notebook ở đường dẫn trên.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Notebook exists: True
Total cells: 162

CELL 123
Execution count: 15
Output count: 1
--- Relevant source lines ---
0089:             row["concept_vector_str"],
0091:             "concept_vector_str"
0127: structured_targets_v5 = build_structured_targets(v4_df)
0138:     len(structured_targets_v5)
0141: assert len(structured_targets_v5) == 7606
0144:     structured_targets_v5.items()
0172: for _, row in v4_df.iterrows():
0175:         row["concept_vector_str"],
0177:         "concept_vector_str"
0198:     v4_df["num_concepts_v2"].to_numpy()
0203:     v4_df["num_negative_findings"].to_numpy()
0208:     v4_df["num_attributes_v2"].to_numpy()
0279: # 8. Build one real training batch
0332:     structured_targets_v5[c]["concept_targets"]
0337:     structured_targets_v5[c]["negative_targets"]
0342:     structured_targets_v5[c]["attribute_targets"]
0347:     structure

In [22]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import json
from pathlib import Path
import torch

nb_path = Path("/content/drive/MyDrive/Colab Notebooks/25sep.ipynb")
ckpt_path = Path(
    "/content/drive/MyDrive/NoiSoi_Matching/"
    "baseline_model_v5/checkpoints/best.pt"
)

# 1. In đầy đủ loại và nội dung output của Cell 139
nb = json.loads(nb_path.read_text(encoding="utf-8"))
cell = nb["cells"][139]

print("=" * 70)
print("CELL 139")
print("Execution count:", cell.get("execution_count"))

for i, out in enumerate(cell.get("outputs", [])):
    print(f"\n--- OUTPUT {i}: {out.get('output_type')} ---")

    if out.get("output_type") == "error":
        print("Error:", out.get("ename"), out.get("evalue"))
        print("\n".join(out.get("traceback", []))[-5000:])

    data = out.get("data", {})
    for mime, content in data.items():
        if mime in ("text/plain", "text/html"):
            print(f"[{mime}]")
            print("".join(content) if isinstance(content, list) else content)

    if "text" in out:
        print("".join(out["text"]))

# 2. Đọc metadata checkpoint — không load model vào GPU
print("\n" + "=" * 70)
print("CHECKPOINT METADATA")

if not ckpt_path.exists():
    print("Checkpoint không tồn tại:", ckpt_path)
else:
    print("Exists:", True)
    print("Size (GB):", round(ckpt_path.stat().st_size / 1024**3, 3))
    print("Modified:", __import__("datetime").datetime.fromtimestamp(
        ckpt_path.stat().st_mtime
    ).isoformat())

    try:
        ckpt = torch.load(
            ckpt_path,
            map_location="cpu",
            weights_only=False
        )
        if isinstance(ckpt, dict):
            print("Top-level keys:", list(ckpt.keys()))
            for key in (
                "epoch", "global_step", "best_val_loss",
                "val_loss", "step", "last_epoch"
            ):
                if key in ckpt:
                    print(f"{key}:", ckpt[key])
        else:
            print("Checkpoint object type:", type(ckpt))
        del ckpt
    except Exception as e:
        print("Metadata read error:", repr(e))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CELL 139
Execution count: 15

--- OUTPUT 0: stream ---
V5 — FULL TRAINING
PASS: Drive mounted.
GPU: NVIDIA A100-SXM4-80GB

[1] MANIFESTS


--- OUTPUT 1: error ---
Error: AssertionError /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/v4_training_manifest.csv
---------------------------------------------------------------------------
AssertionError                            Traceback (most recent call last)
/tmp/ipykernel_6136/1670025182.py in <cell line: 0>()
    181 print("=" * 80)
    182 
--> 183 assert V4_MANIFEST.exists(), V4_MANIFEST
    184 assert IMAGE_MANIFEST.exists(), IMAGE_MANIFEST
    185 assert STRUCTURED_CSV.exists(), STRUCTURED_CSV

AssertionError: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/v4_training_manifest.csv

CHECKPOINT METADATA
Exists: True
Size (GB): 4.321
Modified: 2026-09-29T04:20:44
Top-level keys: ['epoch', 

In [23]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
from datetime import datetime
import torch

ROOT = Path("/content/drive/MyDrive/NoiSoi_Matching")
CKPT = ROOT / "baseline_model_v5/checkpoints/best.pt"

print("=" * 70)
print("1. KIỂM TRA CÁC ĐƯỜNG DẪN MANIFEST")
print("=" * 70)

expected_paths = [
    ROOT / "baseline_model_v4/v4_training_manifest.csv",
    ROOT / "final_manifest/final_dataset_manifest.csv",
    ROOT / "v3_ontology_v2/v3_structured_targets_v2_2.csv",
]

for p in expected_paths:
    print(f"{p}")
    print("  exists:", p.exists())
    if p.exists():
        print("  size (MB):", round(p.stat().st_size / 1024**2, 2))
        print("  modified:", datetime.fromtimestamp(p.stat().st_mtime).isoformat())

print("\n" + "=" * 70)
print("2. TÌM CÁC FILE MANIFEST LIÊN QUAN")
print("=" * 70)

for pattern in [
    "*training*manifest*.csv",
    "*dataset*manifest*.csv",
    "*structured*target*.csv",
    "*ontology*.csv",
]:
    print(f"\nPattern: {pattern}")
    matches = list(ROOT.rglob(pattern))
    if not matches:
        print("  Không tìm thấy.")
    for p in matches[:40]:
        print(
            p.relative_to(ROOT),
            "| modified:",
            datetime.fromtimestamp(p.stat().st_mtime).isoformat(),
            "| MB:",
            round(p.stat().st_size / 1024**2, 2),
        )

print("\n" + "=" * 70)
print("3. LỊCH SỬ HUẤN LUYỆN TRONG CHECKPOINT")
print("=" * 70)

if not CKPT.exists():
    print("Không tìm thấy checkpoint:", CKPT)
else:
    ckpt = torch.load(CKPT, map_location="cpu", weights_only=False)

    print("epoch:", ckpt.get("epoch"))
    print("global_step:", ckpt.get("global_step"))
    print("best_val:", ckpt.get("best_val"))

    history = ckpt.get("history")
    print("history type:", type(history).__name__)

    if isinstance(history, dict):
        for key, value in history.items():
            print(f"\nHistory key: {key}")
            print("Value:", repr(value)[:4000])
    elif isinstance(history, (list, tuple)):
        print("History entries:", len(history))
        for item in history[-10:]:
            print(repr(item)[:1000])
    else:
        print("History:", repr(history)[:4000])

    del ckpt

print("\nHOÀN TẤT — chỉ đọc, không huấn luyện, không lưu thay đổi.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1. KIỂM TRA CÁC ĐƯỜNG DẪN MANIFEST
/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/v4_training_manifest.csv
  exists: True
  size (MB): 1.83
  modified: 2026-09-22T16:26:41
/content/drive/MyDrive/NoiSoi_Matching/final_manifest/final_dataset_manifest.csv
  exists: True
  size (MB): 51.84
  modified: 2026-09-18T02:08:30
/content/drive/MyDrive/NoiSoi_Matching/v3_ontology_v2/v3_structured_targets_v2_2.csv
  exists: True
  size (MB): 5.94
  modified: 2026-09-21T16:51:22

2. TÌM CÁC FILE MANIFEST LIÊN QUAN

Pattern: *training*manifest*.csv
baseline_model_v3/v3_training_manifest.csv | modified: 2026-09-21T16:52:13 | MB: 1.83
baseline_model_v4/v4_training_manifest.csv | modified: 2026-09-22T16:26:41 | MB: 1.83

Pattern: *dataset*manifest*.csv
final_manifest/final_dataset_manifest.csv | modified: 2026-09-18T02:08:30 | MB: 51.84

Pattern: *structured*target*.c

In [24]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
from datetime import datetime
import pandas as pd
import json

ROOT = Path("/content/drive/MyDrive/NoiSoi_Matching")
V4_PATH = ROOT / "baseline_model_v4/v4_training_manifest.csv"
IMG_PATH = ROOT / "final_manifest/final_dataset_manifest.csv"
STRUCT_PATH = ROOT / "v3_ontology_v2/v3_structured_targets_v2_2.csv"
MAP_PATH = ROOT / "baseline_model_v4/diagnostics/frozen_concept_ontology_mapping.csv"
PRED_PATH = ROOT / "baseline_model_v5/test_predictions_712.csv"
OUT = ROOT / "baseline_model_v5/diagnostics/audit_20261015"
OUT.mkdir(parents=True, exist_ok=True)

report = []

def log(s=""):
    print(s)
    report.append(str(s))

def inspect_csv(path, name):
    log(f"\n{'=' * 75}\n{name}\n{'=' * 75}")
    log(f"Path: {path}")
    log(f"Exists: {path.exists()}")
    if not path.exists():
        return None
    log(f"Size MB: {path.stat().st_size / 1024**2:.3f}")
    log(f"Modified: {datetime.fromtimestamp(path.stat().st_mtime).isoformat()}")
    df = pd.read_csv(path, low_memory=False)
    log(f"Shape: {df.shape}")
    log(f"Columns ({len(df.columns)}): {df.columns.tolist()}")
    if "case_id" in df.columns:
        ids = df["case_id"].astype(str)
        log(f"Unique case_id: {ids.nunique()} / {len(df)}")
        log(f"Duplicate case_id rows: {int(ids.duplicated().sum())}")
    return df

v4 = inspect_csv(V4_PATH, "V4 TRAINING MANIFEST")
img = inspect_csv(IMG_PATH, "IMAGE MANIFEST")
st = inspect_csv(STRUCT_PATH, "STRUCTURED TARGET CSV")
mapping = inspect_csv(MAP_PATH, "FROZEN ONTOLOGY MAPPING")
pred = inspect_csv(PRED_PATH, "TEST PREDICTIONS")

if v4 is not None:
    log("\n[V4 vector/count/mask columns]")
    for c in v4.columns:
        if any(k in c.lower() for k in [
            "concept", "negative", "attribute", "structured", "split"
        ]):
            log(f"{c}: {v4[c].head(3).tolist()}")

if st is not None:
    log("\n[Structured CSV: column groups]")
    for prefix in ("concept__", "negative__", "attribute__"):
        cols = [c for c in st.columns if c.startswith(prefix)]
        log(f"{prefix}: {len(cols)}")
        log(f"Names: {cols}")

    log("\n[Potential serialized label columns]")
    for c in st.columns:
        if any(k in c.lower() for k in [
            "concept", "negative", "attribute", "finding"
        ]) and not c.startswith(("concept__", "negative__", "attribute__")):
            log(f"{c}: {st[c].head(3).tolist()}")

if mapping is not None:
    log("\n[Mapping table preview]")
    log(mapping.head(20).to_string(index=False))

if v4 is not None and st is not None:
    a, b = set(v4["case_id"].astype(str)), set(st["case_id"].astype(str))
    log("\n[CASE ID OVERLAP]")
    log(f"V4 IDs: {len(a)} | Structured IDs: {len(b)}")
    log(f"Overlap: {len(a & b)}")
    log(f"Only V4: {len(a - b)}")
    log(f"Only Structured: {len(b - a)}")

report_path = OUT / "cell1_schema_ontology_audit.txt"
report_path.write_text("\n".join(report), encoding="utf-8")
print("\nSaved report:", report_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

V4 TRAINING MANIFEST
Path: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/v4_training_manifest.csv
Exists: True
Size MB: 1.833
Modified: 2026-09-22T16:26:41
Shape: (7606, 15)
Columns (15): ['case_id', 'patient_group_id', 'split', 'ket_luan', 'concepts_v2_str', 'negative_findings_str', 'attributes_v2_str', 'concept_vector_str', 'negative_vector_str', 'attribute_vector_str', 'has_structured_label', 'structured_loss_mask', 'num_concepts_v2', 'num_negative_findings', 'num_attributes_v2']
Unique case_id: 7606 / 7606
Duplicate case_id rows: 0

IMAGE MANIFEST
Path: /content/drive/MyDrive/NoiSoi_Matching/final_manifest/final_dataset_manifest.csv
Exists: True
Size MB: 51.841
Modified: 2026-09-18T02:08:30
Shape: (76405, 26)
Columns (26): ['case_id', 'image_order', 'image_path', 'image_name', 'record_id', 'datetime', 'date', 'image_index', 'n_images_case', 'i

In [25]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import pandas as pd
import numpy as np
import ast, json, re

ROOT = Path("/content/drive/MyDrive/NoiSoi_Matching")
V4_PATH = ROOT / "baseline_model_v4/v4_training_manifest.csv"
OUT = ROOT / "baseline_model_v5/diagnostics/audit_20261015"
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(V4_PATH, low_memory=False)
print("Rows:", len(df), "| Columns:", len(df.columns))

def parse_vector(value):
    if pd.isna(value):
        return None
    if isinstance(value, (list, tuple, np.ndarray)):
        return np.asarray(value, dtype=float).reshape(-1)
    s = str(value).strip()
    if not s:
        return None
    for parser in (ast.literal_eval, json.loads):
        try:
            obj = parser(s)
            if isinstance(obj, (list, tuple, np.ndarray)):
                return np.asarray(obj, dtype=float).reshape(-1)
        except Exception:
            pass
    # Fallback only for simple numeric vectors.
    if s.startswith("[") and s.endswith("]"):
        vals = re.findall(r"[-+]?(?:\d*\.)?\d+(?:[eE][-+]?\d+)?", s)
        if vals:
            return np.asarray(vals, dtype=float)
    return None

expected = {
    "concept_vector_str": 48,
    "negative_vector_str": 5,
    "attribute_vector_str": 6,
}
count_cols = {
    "concept_vector_str": "num_concepts_v2",
    "negative_vector_str": "num_negative_findings",
    "attribute_vector_str": "num_attributes_v2",
}

issues = []
summary = []

for _, row in df.iterrows():
    cid = str(row.get("case_id", ""))
    record = {"case_id": cid, "split": row.get("split", None)}
    for col, dim in expected.items():
        if col not in df.columns:
            record[col + "_issue"] = "MISSING_COLUMN"
            continue

        vec = parse_vector(row[col])
        if vec is None:
            record[col + "_issue"] = "PARSE_FAILED_OR_NULL"
            continue

        record[col + "_length"] = len(vec)
        record[col + "_sum"] = float(np.nansum(vec))
        if len(vec) != dim:
            record[col + "_issue"] = f"WRONG_LENGTH_{len(vec)}_EXPECTED_{dim}"
        elif not np.isfinite(vec).all():
            record[col + "_issue"] = "NONFINITE_VALUE"
        elif not np.isin(vec, [0, 1]).all():
            record[col + "_issue"] = "NON_BINARY_VALUE"
        else:
            record[col + "_issue"] = ""

        cnt_col = count_cols[col]
        if cnt_col in df.columns and len(vec) == dim:
            try:
                count_value = float(row[cnt_col])
                if np.isfinite(count_value) and int(vec.sum()) != int(count_value):
                    record[col + "_count_issue"] = (
                        f"VECTOR_SUM_{int(vec.sum())}_COUNT_COL_{int(count_value)}"
                    )
                else:
                    record[col + "_count_issue"] = ""
            except Exception:
                record[col + "_count_issue"] = "COUNT_PARSE_FAILED"

    mask = row.get("structured_loss_mask", np.nan)
    try:
        mask = float(mask)
        record["mask_issue"] = "" if mask in (0.0, 1.0) else f"UNEXPECTED_MASK_{mask}"
    except Exception:
        record["mask_issue"] = "MASK_MISSING_OR_INVALID"

    all_issues = [
        (k, v) for k, v in record.items()
        if k.endswith("_issue") and v
    ]
    if all_issues:
        record["issue_details"] = "; ".join(f"{k}={v}" for k, v in all_issues)
        issues.append(record)

print("\n===== VECTOR AUDIT SUMMARY =====")
print("Cases:", len(df))
print("Cases with at least one issue:", len(issues))
print("Split counts:")
if "split" in df.columns:
    print(df["split"].value_counts(dropna=False).to_string())

for col, dim in expected.items():
    print(f"\n{col}: expected dimension {dim}")
    if col not in df.columns:
        print("  MISSING COLUMN")
        continue
    parsed = df[col].map(parse_vector)
    lengths = parsed.map(lambda x: len(x) if x is not None else -1)
    print("  Length distribution:", lengths.value_counts().sort_index().to_dict())
    valid = parsed[parsed.map(lambda x: x is not None and len(x) == dim)]
    if len(valid):
        sums = valid.map(lambda x: int(np.sum(x)))
        print("  Vector-sum distribution:", sums.value_counts().sort_index().to_dict())

issue_path = OUT / "cell2_vector_issues.csv"
summary_path = OUT / "cell2_split_summary.csv"
pd.DataFrame(issues).to_csv(issue_path, index=False)

if "split" in df.columns:
    split_rows = []
    for split, part in df.groupby("split", dropna=False):
        item = {"split": split, "cases": len(part)}
        for col in expected:
            if col in part.columns:
                parsed = part[col].map(parse_vector)
                item[col + "_parse_failures"] = int(parsed.isna().sum())
                item[col + "_expected_length"] = expected[col]
                item[col + "_wrong_length"] = int(
                    parsed.map(lambda x: x is None or len(x) != expected[col]).sum()
                )
        split_rows.append(item)
    pd.DataFrame(split_rows).to_csv(summary_path, index=False)

print("\nSaved issue file:", issue_path)
print("Saved split summary:", summary_path)
print("Read-only audit complete; no model or checkpoint changed.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Rows: 7606 | Columns: 15

===== VECTOR AUDIT SUMMARY =====
Cases: 7606
Cases with at least one issue: 0
Split counts:
split
train    6138
val       756
test      712

concept_vector_str: expected dimension 48
  Length distribution: {48: 7606}
  Vector-sum distribution: {0: 419, 1: 4708, 2: 2163, 3: 294, 4: 21, 5: 1}

negative_vector_str: expected dimension 5
  Length distribution: {5: 7606}
  Vector-sum distribution: {0: 6806, 1: 730, 2: 70}

attribute_vector_str: expected dimension 6
  Length distribution: {6: 7606}
  Vector-sum distribution: {0: 2337, 1: 3202, 2: 1669, 3: 316, 4: 70, 5: 12}

Saved issue file: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/diagnostics/audit_20261015/cell2_vector_issues.csv
Saved split summary: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/diagnostics/audit_20261015/cell2_split_summary.csv
Read-only audit

In [26]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import re, unicodedata

ROOT = Path("/content/drive/MyDrive/NoiSoi_Matching")
PRED_PATH = ROOT / "baseline_model_v5/test_predictions_712.csv"
OUT = ROOT / "baseline_model_v5/diagnostics/audit_20261015"
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(PRED_PATH, low_memory=False)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head(5).to_string(index=False))

def norm_text(x):
    if pd.isna(x):
        return ""
    s = unicodedata.normalize("NFKC", str(x)).lower().strip()
    s = re.sub(r"[^\w\s]", " ", s, flags=re.UNICODE)
    return re.sub(r"\s+", " ", s).strip()

def choose_col(columns, candidates):
    lower = {str(c).lower(): c for c in columns}
    for name in candidates:
        if name in lower:
            return lower[name]
    for c in columns:
        cl = str(c).lower()
        if any(name in cl for name in candidates):
            return c
    return None

id_col = choose_col(df.columns, ["case_id"])
pred_col = choose_col(df.columns, [
    "prediction", "predicted_text", "pred_text",
    "generated_text", "y_pred", "predicted", "pred"
])
true_col = choose_col(df.columns, [
    "ground_truth", "true_label", "target_text",
    "y_true", "ket_luan", "reference", "label", "target"
])

print("\nDetected case ID:", id_col)
print("Detected prediction:", pred_col)
print("Detected ground truth:", true_col)

lines = [
    f"Rows: {len(df)}",
    f"Columns: {df.columns.tolist()}",
    f"Detected ID: {id_col}",
    f"Detected prediction: {pred_col}",
    f"Detected ground truth: {true_col}",
]
case_report = []

if pred_col is None or true_col is None:
    lines.append(
        "Metrics not calculated because prediction/ground-truth columns "
        "could not be identified safely."
    )
else:
    d = df.copy()
    d["_pred_norm"] = d[pred_col].map(norm_text)
    d["_true_norm"] = d[true_col].map(norm_text)
    d["_exact_match"] = d["_pred_norm"] == d["_true_norm"]

    def token_f1(p, t):
        pt, tt = p.split(), t.split()
        if not pt or not tt:
            return float(pt == tt)
        pc, tc = Counter(pt), Counter(tt)
        overlap = sum((pc & tc).values())
        if overlap == 0:
            return 0.0
        precision = overlap / len(pt)
        recall = overlap / len(tt)
        return 2 * precision * recall / (precision + recall)

    d["_token_f1"] = [
        token_f1(p, t) for p, t in zip(d["_pred_norm"], d["_true_norm"])
    ]

    pred_counts = d["_pred_norm"].value_counts(dropna=False)
    lines.extend([
        f"Rows evaluated: {len(d)}",
        f"Unique normalized predictions: {d['_pred_norm'].nunique()}",
        f"Unique normalized ground truths: {d['_true_norm'].nunique()}",
        f"Exact match: {d['_exact_match'].mean():.4f}",
        f"Mean token F1: {d['_token_f1'].mean():.4f}",
        f"Top prediction share: {pred_counts.iloc[0] / len(d):.4f}",
        "\nTop 20 predictions:",
        pred_counts.head(20).to_string(),
        "\nTop 20 ground-truth labels:",
        d["_true_norm"].value_counts().head(20).to_string(),
    ])

    # Cases with mismatch, longest/shortest labels and case ID if available.
    mismatch = d.loc[~d["_exact_match"]].copy()
    cols = [c for c in [id_col, true_col, pred_col, "_token_f1"] if c is not None]
    mismatch[cols].to_csv(OUT / "cell3_prediction_mismatches.csv", index=False)

    d[cols + ["_exact_match"]].to_csv(
        OUT / "cell3_prediction_case_metrics.csv", index=False
    )

    print("\n===== PREDICTION METRICS =====")
    print("\n".join(lines))

report_path = OUT / "cell3_prediction_audit.txt"
report_path.write_text("\n".join(lines), encoding="utf-8")

print("\nSaved report:", report_path)
print("Saved mismatch CSV:", OUT / "cell3_prediction_mismatches.csv")
print("Saved per-case CSV:", OUT / "cell3_prediction_case_metrics.csv")
print("\nAudit complete. No training, no checkpoint overwrite.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Shape: (712, 3)
Columns: ['case_id', 'prediction', 'ground_truth']
            case_id                                   prediction                                         ground_truth
10017.10017.0.10031          VIÊM ỐNG TAI NGOÀI + MÀNG NHĨ T CẤP      VIÊM MÀNG NHĨ CẤP BÓNG NƯỚC P - NHIỀU RÁY TAI T
10057.10057.0.10071          VIÊM ỐNG TAI NGOÀI + MÀNG NHĨ T CẤP       VIÊM ỐNG TAI NGOÀI (T) MẠN / VIÊM MŨI ĐỢT CẤP.
10116.10116.0.10130          VIÊM ỐNG TAI NGOÀI + MÀNG NHĨ T CẤP                    VIÊM ỐNG TAI NGOÀI MÀNG NHĨ P CẤP
10125.10125.0.10139                                 VIÊM MŨI MẠN VIÊM MŨI MẠN VIÊM ỐNG TAI NGOÀI + MÀNG NHĨ 2 BÊN MẠN
10126.10126.0.10140 VIÊM HỌNG MẠN - THEO DÕI TRÀO NGƯỢC DỊCH VỊ.                                       VIÊM A-MI-ĐAN.

Detected case ID: case_id
Detected prediction: prediction
Detected ground truth: ground_truth



In [27]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import pandas as pd
import numpy as np
import ast, json

ROOT = Path("/content/drive/MyDrive/NoiSoi_Matching")
V4_PATH = ROOT / "baseline_model_v4/v4_training_manifest.csv"
STRUCT_PATH = ROOT / "v3_ontology_v2/v3_structured_targets_v2_2.csv"
MAP_PATH = ROOT / "baseline_model_v4/diagnostics/frozen_concept_ontology_mapping.csv"
OUT = ROOT / "baseline_model_v5/diagnostics/audit_20261015"
OUT.mkdir(parents=True, exist_ok=True)

v4 = pd.read_csv(V4_PATH, low_memory=False)
st = pd.read_csv(STRUCT_PATH, low_memory=False)
mp = pd.read_csv(MAP_PATH, low_memory=False)

def parse_vec(x):
    if pd.isna(x):
        return []
    s = str(x).strip()
    try:
        z = ast.literal_eval(s)
        if isinstance(z, (list, tuple)):
            return [int(float(v)) for v in z]
    except Exception:
        pass
    try:
        return [int(float(v.strip())) for v in s.split(",")]
    except Exception:
        return []

# Sort mapping by explicit dimension, never by current CSV column order.
mp["dimension"] = pd.to_numeric(mp["dimension"], errors="coerce")
mp = mp.sort_values("dimension").reset_index(drop=True)

print("Mapping rows:", len(mp))
print("Dimensions:", mp["dimension"].tolist())
print("Unique dimensions:", mp["dimension"].nunique())
print("Unique mapping columns:", mp["column"].nunique())
print("\nFull frozen mapping:")
print(mp.to_string(index=False))

mapping_labels = mp["label"].astype(str).tolist()
mapping_columns = mp["column"].astype(str).tolist()

# Structured wide columns are only a comparison source; don't assume their order.
wide_cols = [c for c in st.columns if c.startswith("concept__")]
wide_labels = {c.removeprefix("concept__"): c for c in wide_cols}

# Match each V4 serialized concept list against mapping labels.
# A one-hot vector should point to the corresponding serialized concept if
# the mapping and V4 vectors use the same ontology/order.
merged = v4[[
    "case_id", "split", "concepts_v2_str",
    "concept_vector_str", "num_concepts_v2"
]].merge(
    st[["case_id"] + wide_cols],
    on="case_id", how="left", validate="one_to_one",
    suffixes=("_v4", "_wide")
)

issues = []
dimension_rows = []

for _, row in merged.iterrows():
    vec = parse_vec(row["concept_vector_str"])
    if len(vec) != 48:
        issues.append({
            "case_id": row["case_id"],
            "split": row["split"],
            "issue": f"vector length {len(vec)}",
        })
        continue

    serialized = str(row["concepts_v2_str"])
    try:
        parsed = ast.literal_eval(serialized)
        if isinstance(parsed, list):
            source_labels = [str(x) for x in parsed if str(x) != "nan"]
        else:
            source_labels = [x.strip() for x in serialized.split(",") if x.strip()]
    except Exception:
        source_labels = [x.strip() for x in serialized.split(",") if x.strip()]

    active_dims = [i for i, val in enumerate(vec) if val == 1]
    mapped_labels = [mapping_labels[i] for i in active_dims if i < len(mapping_labels)]

    # Compare vector-active labels against serialized source labels.
    source_set = set(source_labels)
    mapped_set = set(mapped_labels)

    if source_set != mapped_set:
        issues.append({
            "case_id": row["case_id"],
            "split": row["split"],
            "issue": "vector labels differ from concepts_v2_str",
            "source_labels": "|".join(sorted(source_set)),
            "mapped_labels": "|".join(sorted(mapped_set)),
            "active_dims": "|".join(map(str, active_dims)),
        })

    for i, col in enumerate(mapping_columns):
        if i >= len(vec):
            continue
        label = mapping_labels[i]
        wide_col = wide_labels.get(label)
        if wide_col is None:
            continue
        expected = pd.to_numeric(pd.Series([row[wide_col]]), errors="coerce").iloc[0]
        if pd.notna(expected) and int(expected) != int(vec[i]):
            dimension_rows.append({
                "case_id": row["case_id"],
                "split": row["split"],
                "dimension": i,
                "label": label,
                "vector_value": int(vec[i]),
                "wide_csv_value": int(expected),
                "wide_column": wide_col,
            })

issues_df = pd.DataFrame(issues)
dim_df = pd.DataFrame(dimension_rows)

issues_path = OUT / "cell4_vector_vs_serialized_labels.csv"
dims_path = OUT / "cell4_vector_vs_wide_columns.csv"
issues_df.to_csv(issues_path, index=False)
dim_df.to_csv(dims_path, index=False)

print("\n===== RESULT =====")
print("Cases compared:", len(merged))
print("Cases with vector vs serialized-label discrepancy:", len(issues_df))
print("Per-dimension wide-column discrepancies:", len(dim_df))
print("Cases with any wide-column discrepancy:",
      dim_df["case_id"].nunique() if len(dim_df) else 0)

if len(issues_df):
    print("\nExamples of serialized-label discrepancies:")
    print(issues_df.head(20).to_string(index=False))

if len(dim_df):
    print("\nExamples of wide-column discrepancies:")
    print(dim_df.head(30).to_string(index=False))

print("\nSaved:")
print(issues_path)
print(dims_path)
print("\nRead-only audit completed. Checkpoint untouched.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Mapping rows: 48
Dimensions: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47]
Unique dimensions: 48
Unique mapping columns: 48

Full frozen mapping:
 dimension                             column                     label
         0           concept__amidan_qua_phat           amidan_qua_phat
         1      concept__chan_thuong_mang_nhi      chan_thuong_mang_nhi
         2 concept__chan_thuong_ong_tai_ngoai chan_thuong_ong_tai_ngoai
         3              concept__chay_mau_mui              chay_mau_mui
         4               concept__di_vat_hong               di_vat_hong
         5    concept__di_vat_hong_thanh_quan    di_vat_hong_thanh_quan
         6                concept__di_vat_tai                di_vat_tai
         7  

In [28]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import pandas as pd
import numpy as np
import ast

ROOT = Path("/content/drive/MyDrive/NoiSoi_Matching")
V4_PATH = ROOT / "baseline_model_v4/v4_training_manifest.csv"
OUT = ROOT / "baseline_model_v5/diagnostics/audit_20261015"
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(V4_PATH, low_memory=False)

def parse_vector(x):
    if pd.isna(x):
        return []
    s = str(x).strip()
    try:
        value = ast.literal_eval(s)
        if isinstance(value, (list, tuple)):
            return [int(float(v)) for v in value]
    except (ValueError, SyntaxError, TypeError):
        pass
    try:
        return [int(float(v.strip())) for v in s.split(",")]
    except (ValueError, TypeError):
        return []

def parse_labels(x):
    if pd.isna(x):
        return set()
    s = str(x).strip()
    if not s or s.lower() in {"nan", "none", "null"}:
        return set()
    try:
        value = ast.literal_eval(s)
        if isinstance(value, (list, tuple, set)):
            return {
                str(v).strip()
                for v in value
                if str(v).strip() and str(v).strip().lower() not in {"nan", "none", "null"}
            }
    except (ValueError, SyntaxError, TypeError):
        pass
    return {
        part.strip()
        for part in s.split("|")
        if part.strip() and part.strip().lower() not in {"nan", "none", "null"}
    }

mapping_path = ROOT / "baseline_model_v4/diagnostics/frozen_concept_ontology_mapping.csv"
mapping = pd.read_csv(mapping_path).sort_values("dimension")
labels = mapping["label"].astype(str).tolist()

rows = []
for _, r in df.iterrows():
    vec = parse_vector(r["concept_vector_str"])
    source = parse_labels(r["concepts_v2_str"])

    if len(vec) != len(labels):
        rows.append({
            "case_id": r["case_id"],
            "split": r["split"],
            "issue": "wrong_vector_length",
            "source_labels": "|".join(sorted(source)),
            "vector_labels": "",
        })
        continue

    vector_labels = {
        labels[i] for i, value in enumerate(vec)
        if value == 1
    }
    if source != vector_labels:
        rows.append({
            "case_id": r["case_id"],
            "split": r["split"],
            "issue": "label_set_mismatch",
            "source_labels": "|".join(sorted(source)),
            "vector_labels": "|".join(sorted(vector_labels)),
        })

audit = pd.DataFrame(rows)
audit_path = OUT / "cell5_normalized_label_comparison.csv"
audit.to_csv(audit_path, index=False)

print("Total cases:", len(df))
print("Cases with normalized discrepancies:", len(audit))
if len(audit):
    print("\nIssue types:")
    print(audit["issue"].value_counts().to_string())
    print("\nFirst 20 discrepancies:")
    print(audit.head(20).to_string(index=False))
else:
    print("PASS: All serialized label sets match their vectors.")

print("\nSaved:", audit_path)
print("Read-only audit; no model training or checkpoint modification.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total cases: 7606
Cases with normalized discrepancies: 0
PASS: All serialized label sets match their vectors.

Saved: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/diagnostics/audit_20261015/cell5_normalized_label_comparison.csv
Read-only audit; no model training or checkpoint modification.


In [29]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import json
import hashlib

ROOT = Path("/content/drive/MyDrive/NoiSoi_Matching")
NB_PATH = Path("/content/drive/MyDrive/Colab Notebooks/25sep.ipynb")
OUT = ROOT / "baseline_model_v5/diagnostics/audit_20261015"
OUT.mkdir(parents=True, exist_ok=True)

assert NB_PATH.exists(), f"Notebook not found: {NB_PATH}"

with open(NB_PATH, "r", encoding="utf-8") as f:
    nb = json.load(f)

cells = nb.get("cells", [])
print("Notebook:", NB_PATH.name)
print("Total cells:", len(cells))

keywords = [
    "def build_structured_targets",
    "V5 — FULL TRAINING",
    "V5 FULL TRAINING",
    "concept_vector_str",
    "negative_vector_str",
    "attribute_vector_str",
    "BCEWithLogitsLoss",
    "optimizer.step",
    "loss.backward",
    "best.pt",
    "load_state_dict",
]

report = []
for i, cell in enumerate(cells):
    source = "".join(cell.get("source", []))
    if any(k.lower() in source.lower() for k in keywords):
        report.append({
            "cell_index": i,
            "execution_count": cell.get("execution_count"),
            "source": source,
        })

for item in report:
    print("\n" + "=" * 90)
    print(
        f"CELL INDEX: {item['cell_index']} | "
        f"EXECUTION COUNT: {item['execution_count']}"
    )
    print("=" * 90)
    lines = item["source"].splitlines()
    hits = [
        j for j, line in enumerate(lines)
        if any(k.lower() in line.lower() for k in keywords)
    ]
    shown = set()
    for j in hits:
        for k in range(max(0, j - 3), min(len(lines), j + 5)):
            if k not in shown:
                print(f"{k+1:4}: {lines[k]}")
                shown.add(k)
        print("    ...")

out_path = OUT / "cell6_training_source_audit.txt"
with open(out_path, "w", encoding="utf-8") as f:
    for item in report:
        f.write(
            f"\n{'='*90}\nCELL INDEX: {item['cell_index']} | "
            f"EXECUTION COUNT: {item['execution_count']}\n"
        )
        f.write(item["source"])
        f.write("\n")

print("\nCells matched:", len(report))
print("Report saved:", out_path)
print("READ-ONLY: no training, no optimizer, no checkpoint writes.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Notebook: 25sep.ipynb
Total cells: 162

CELL INDEX: 0 | EXECUTION COUNT: None
  70: )
  71: 
  72: BEST_CKPT = (
  73:     f"{V4_DIR}/checkpoints/best.pt"
  74: )
    ...

CELL INDEX: 1 | EXECUTION COUNT: None
  87: )
  88: 
  89: BEST_CKPT = (
  90:     f"{V4_DIR}/checkpoints/best.pt"
  91: )
  92: 
  93: # ------------------------------------------------
  94: # Path checks
    ...
1014: # Strict load
1015: # ------------------------------------------------
1016: 
1017: load_result = model.load_state_dict(
1018:     E2["model_state"],
1019:     strict=True,
1020: )
1021: 
    ...
1028:     E2["best_val"]
1029: )
1030: 
1031: model.load_state_dict(
1032:     E3["model_state"],
1033:     strict=True,
1034: )
1035: 
    ...
1099: # 14. SNAPSHOT
1100: # ================================================================
1101: 
1102: model.load_state_dict(
1103:   

In [30]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import json
from pathlib import Path

NB_PATH = Path("/content/drive/MyDrive/Colab Notebooks/25sep.ipynb")
OUT_DIR = Path(
    "/content/drive/MyDrive/NoiSoi_Matching/"
    "baseline_model_v5/diagnostics/audit_20261015"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(NB_PATH, "r", encoding="utf-8") as f:
    nb = json.load(f)

matches = []
for idx, cell in enumerate(nb.get("cells", [])):
    source = "".join(cell.get("source", []))
    if "V5 — FULL TRAINING" in source or "V5 - FULL TRAINING" in source:
        matches.append((idx, source))

assert matches, "Không tìm thấy cell huấn luyện V5 trong notebook."

idx, source = matches[-1]
lines = source.splitlines()

print("Training cell index:", idx)
print("Total source lines:", len(lines))

# In các đoạn then chốt để xác định nguồn nhãn, loss và checkpoint.
needles = [
    "V4_MANIFEST",
    "STRUCTURED_CSV",
    "pd.read_csv",
    "required_structured_cols",
    "structured_df",
    "structured_lookup",
    "train_df",
    "build_structured_targets",
    "backward_loss",
    "loss.backward",
    "optimizer.step",
    "torch.save",
    "BEST_CKPT",
]

ranges = []
for i, line in enumerate(lines):
    if any(key.lower() in line.lower() for key in needles):
        ranges.append((max(0, i - 5), min(len(lines), i + 9)))

# Gộp các đoạn bị chồng lấn.
merged = []
for start, end in sorted(ranges):
    if merged and start <= merged[-1][1]:
        merged[-1] = (merged[-1][0], max(merged[-1][1], end))
    else:
        merged.append((start, end))

output = []
for start, end in merged:
    output.append(f"\n{'=' * 72}\nLINES {start+1}-{end}\n{'=' * 72}")
    output.extend(
        f"{i+1:4}: {lines[i]}"
        for i in range(start, end)
    )

text = "\n".join(output)
print(text)

report_path = OUT_DIR / "cell7_v5_training_flow.txt"
report_path.write_text(text, encoding="utf-8")
print("\nSaved:", report_path)
print("READ-ONLY: không train, không optimizer step, không ghi checkpoint.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Training cell index: 139
Total source lines: 2121

LINES 121-146
 121: ROOT = (
 122:     DRIVE /
 123:     "NoiSoi_Matching"
 124: )
 125: 
 126: V4_MANIFEST = (
 127:     ROOT /
 128:     "baseline_model_v4" /
 129:     "v4_training_manifest.csv"
 130: )
 131: 
 132: IMAGE_MANIFEST = (
 133:     ROOT /
 134:     "final_manifest" /
 135:     "final_dataset_manifest.csv"
 136: )
 137: 
 138: STRUCTURED_CSV = (
 139:     ROOT /
 140:     "v3_ontology_v2" /
 141:     "v3_structured_targets_v2_2.csv"
 142: )
 143: 
 144: V5_ROOT = (
 145:     ROOT /
 146:     "baseline_model_v5"

LINES 159-172
 159: LAST_CKPT = (
 160:     CKPT_DIR /
 161:     "last.pt"
 162: )
 163: 
 164: BEST_CKPT = (
 165:     CKPT_DIR /
 166:     "best.pt"
 167: )
 168: 
 169: ROLLING = [
 170:     CKPT_DIR / "resume_0.pt",
 171:     CKPT_DIR / "resume_1.pt",
 172: ]

LINES 178-204
 178: 
 

In [31]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import json
from pathlib import Path
from datetime import datetime

NB_PATH = Path("/content/drive/MyDrive/Colab Notebooks/25sep.ipynb")
ROOT = Path("/content/drive/MyDrive/NoiSoi_Matching")
CKPT = ROOT / "baseline_model_v5/checkpoints/best.pt"
OUT_DIR = ROOT / "baseline_model_v5/diagnostics/audit_20261015"
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(NB_PATH, "r", encoding="utf-8") as f:
    nb = json.load(f)

lines = []
for idx in [123, 126, 127, 139, 140]:
    if idx >= len(nb["cells"]):
        continue

    cell = nb["cells"][idx]
    source = "".join(cell.get("source", []))
    outputs = cell.get("outputs", [])

    lines.append("\n" + "=" * 85)
    lines.append(
        f"CELL {idx} | execution_count={cell.get('execution_count')} "
        f"| output_count={len(outputs)}"
    )
    lines.append("=" * 85)

    # Hiển thị các đoạn source liên quan đến dữ liệu và checkpoint.
    src = source.splitlines()
    keys = [
        "V4_MANIFEST", "STRUCTURED_CSV", "structured_df",
        "v4_df", "structured_targets", "save_ckpt",
        "BEST_CKPT", "backward_loss.backward",
        "optimizer.step", "PASS: V5 TRAINING COMPLETE",
    ]
    hit = [
        i for i, line in enumerate(src)
        if any(k.lower() in line.lower() for k in keys)
    ]
    shown = set()
    for i in hit:
        for j in range(max(0, i - 2), min(len(src), i + 4)):
            if j not in shown:
                lines.append(f"SOURCE {j+1:4}: {src[j]}")
                shown.add(j)

    # Chỉ trích xuất output văn bản đã lưu; không thực thi cell.
    for oi, out in enumerate(outputs):
        text_parts = []
        data = out.get("data", {})
        for mime in ("text/plain", "text/markdown"):
            value = data.get(mime)
            if value:
                text_parts.extend(value if isinstance(value, list) else [str(value)])
        if out.get("output_type") == "error":
            text_parts.append(
                f"ERROR: {out.get('ename')}: {out.get('evalue')}"
            )
        if text_parts:
            out_text = "\n".join(text_parts)
            lines.append(f"\nSAVED OUTPUT {oi}:\n{out_text[:5000]}")

if CKPT.exists():
    stat = CKPT.stat()
    lines.extend([
        "\n" + "=" * 85,
        "CHECKPOINT FILE METADATA",
        "=" * 85,
        f"Path: {CKPT}",
        f"Size (GB): {stat.st_size / (1024**3):.3f}",
        f"Modified: {datetime.fromtimestamp(stat.st_mtime).isoformat()}",
    ])
else:
    lines.append(f"\nCHECKPOINT NOT FOUND: {CKPT}")

report = "\n".join(lines)
print(report)

report_path = OUT_DIR / "cell8_saved_training_history.txt"
report_path.write_text(report, encoding="utf-8")
print("\nReport saved:", report_path)
print("READ-ONLY: không chạy cell trong notebook, không load model, không sửa checkpoint.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

CELL 123 | execution_count=15 | output_count=1
SOURCE   78: # ------------------------------------------------------------
SOURCE   79: 
SOURCE   80: def build_structured_targets(df):
SOURCE   81: 
SOURCE   82:     result = {}
SOURCE   83: 
SOURCE  125: 
SOURCE  126: 
SOURCE  127: structured_targets_v5 = build_structured_targets(v4_df)
SOURCE  128: 
SOURCE  129: 
SOURCE  130: # ------------------------------------------------------------
SOURCE  136: print(
SOURCE  137:     "Cases:",
SOURCE  138:     len(structured_targets_v5)
SOURCE  139: )
SOURCE  140: 
SOURCE  141: assert len(structured_targets_v5) == 7606
SOURCE  142: 
SOURCE  143: for case_id, target in list(
SOURCE  144:     structured_targets_v5.items()
SOURCE  145: )[:3]:
SOURCE  146: 
SOURCE  147:     print(
SOURCE  170: attribute_counts = []
SOURCE  171: 
SOURCE  172: for _, row in v4_df.iterrows()

In [32]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
from datetime import datetime
import torch

CKPT_DIR = Path(
    "/content/drive/MyDrive/NoiSoi_Matching/"
    "baseline_model_v5/checkpoints"
)

if not CKPT_DIR.exists():
    print("Không tìm thấy thư mục:", CKPT_DIR)
else:
    files = sorted(
        p for p in CKPT_DIR.iterdir()
        if p.is_file() and p.suffix.lower() in {".pt", ".pth"}
    )

    print("Checkpoint directory:", CKPT_DIR)
    print("Số file checkpoint:", len(files))

    for p in files:
        stat = p.stat()
        print(
            f"\n{p.name}\n"
            f"  Size: {stat.st_size / (1024**3):.3f} GB\n"
            f"  Modified: "
            f"{datetime.fromtimestamp(stat.st_mtime).isoformat()}"
        )

    # Chỉ đọc metadata của best.pt; không load model_state vào GPU.
    best = CKPT_DIR / "best.pt"
    if best.exists():
        try:
            obj = torch.load(
                best,
                map_location="cpu",
                weights_only=False,
                mmap=True,
            )
            print("\nBEST.PT METADATA")
            if isinstance(obj, dict):
                for key in obj:
                    if key not in {
                        "model_state", "optimizer_state",
                        "scheduler_state", "rng_state",
                        "cuda_rng_state", "python_rng_state",
                        "numpy_rng_state"
                    }:
                        value = obj[key]
                        if key == "history":
                            print("history:", value)
                        elif isinstance(value, (str, int, float, bool, type(None))):
                            print(f"{key}: {value}")
                print("Top-level keys:", list(obj.keys()))
            del obj
        except Exception as e:
            print("Không đọc được metadata:", repr(e))

print("\nREAD-ONLY: không train, không cập nhật checkpoint.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint directory: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/checkpoints
Số file checkpoint: 2

best.pt
  Size: 4.321 GB
  Modified: 2026-09-29T04:20:44

last.pt
  Size: 4.321 GB
  Modified: 2026-09-29T04:25:51

BEST.PT METADATA
epoch: 5
batch_idx: 0
global_step: 1920
best_val: 0.424518172663671
history: {'train_loss': [0.5731708970967303, 0.5046075203082849, 0.47171955251829634, 0.45422747778290645, 0.44715843725379206], 'val_loss': [0.46700948011622856, 0.44155955633946825, 0.42823387978095856, 0.4258775923539091, 0.424518172663671]}
Top-level keys: ['epoch', 'batch_idx', 'global_step', 'model_state', 'optimizer_state', 'scheduler_state', 'best_val', 'history', 'rng_state', 'cuda_rng_state', 'python_rng_state', 'numpy_rng_state']

READ-ONLY: không train, không cập nhật checkpoint.


In [33]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import json
from pathlib import Path
import ast

ROOT = Path("/content/drive/MyDrive/NoiSoi_Matching")
NB_PATH = Path("/content/drive/MyDrive/Colab Notebooks/25sep.ipynb")
V4_MANIFEST = ROOT / "baseline_model_v4/v4_training_manifest.csv"
STRUCTURED_CSV = ROOT / "v3_ontology_v2/v3_structured_targets_v2_2.csv"
MAPPING_CSV = ROOT / "baseline_model_v4/diagnostics/frozen_concept_ontology_mapping.csv"
CKPT_DIR = ROOT / "baseline_model_v5/checkpoints"
OUT_DIR = ROOT / "baseline_model_v5/diagnostics/audit_20261015"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("=== 1. INPUT FILES ===")
for p in [NB_PATH, V4_MANIFEST, STRUCTURED_CSV, MAPPING_CSV]:
    print(f"{p.name}: exists={p.exists()}")
    if p.exists():
        print(f"  bytes={p.stat().st_size}")

print("\n=== 2. CHECKPOINT FILES ===")
for p in sorted(CKPT_DIR.glob("*.pt")):
    print(f"{p.name}: {p.stat().st_size / (1024**3):.3f} GB")

print("\n=== 3. TRAINING-CELL HISTORY ===")
with open(NB_PATH, "r", encoding="utf-8") as f:
    nb = json.load(f)

for idx, cell in enumerate(nb.get("cells", [])):
    src = "".join(cell.get("source", []))
    if not any(k in src for k in [
        "V5 — FULL TRAINING",
        "V5 - FULL TRAINING",
        "V5 training completed",
        "PASS: V5 TRAINING COMPLETE",
    ]):
        continue

    print(
        f"\nCell {idx}: execution_count={cell.get('execution_count')}, "
        f"source_lines={len(src.splitlines())}"
    )
    for out in cell.get("outputs", []):
        if out.get("output_type") == "error":
            print("Saved error:", out.get("ename"), out.get("evalue"))
        data = out.get("data", {})
        text = data.get("text/plain", [])
        if text:
            s = "".join(text) if isinstance(text, list) else str(text)
            print("Saved text output:", s[:1200])

print("\n=== 4. SOURCE-CODE CLUES ===")
# Chỉ tìm các dòng khai báo nguồn targets trong cell huấn luyện.
for idx, cell in enumerate(nb.get("cells", [])):
    src = "".join(cell.get("source", []))
    if "V5 — FULL TRAINING" not in src and "V5 - FULL TRAINING" not in src:
        continue

    for n, line in enumerate(src.splitlines(), start=1):
        if any(k in line for k in [
            "STRUCTURED_CSV", "structured_df =", "v4_df =",
            "structured_targets_v5 =", "required_structured_cols",
            "concept_vector_str", "negative_vector_str",
            "attribute_vector_str",
        ]):
            print(f"Cell {idx}, line {n}: {line.strip()}")

report_path = OUT_DIR / "cell9_source_provenance.txt"
report_path.write_text(
    "Cell 9 ran read-only checks. See notebook output for details.\n",
    encoding="utf-8",
)
print("\nRead-only audit completed.")
print("Report marker:", report_path)
print("No training or checkpoint modification.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== 1. INPUT FILES ===
25sep.ipynb: exists=True
  bytes=2978538
v4_training_manifest.csv: exists=True
  bytes=1921940
v3_structured_targets_v2_2.csv: exists=True
  bytes=6233626
frozen_concept_ontology_mapping.csv: exists=True
  bytes=1974

=== 2. CHECKPOINT FILES ===
best.pt: 4.321 GB
last.pt: 4.321 GB

=== 3. TRAINING-CELL HISTORY ===

Cell 126: execution_count=1, source_lines=1348
Saved error: NameError name 'train_loader_v5' is not defined

Cell 139: execution_count=15, source_lines=2121
Saved error: AssertionError /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/v4_training_manifest.csv

=== 4. SOURCE-CODE CLUES ===
Cell 139, line 138: STRUCTURED_CSV = (
Cell 139, line 185: assert STRUCTURED_CSV.exists(), STRUCTURED_CSV
Cell 139, line 187: v4_df = pd.read_csv(
Cell 139, line 195: structured_df = pd.read_csv(
Cell 139, line 196: STRUCTURED_CSV
Cel

In [34]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import pandas as pd
import torch

ROOT = Path("/content/drive/MyDrive/NoiSoi_Matching")
CKPT_DIR = ROOT / "baseline_model_v5/checkpoints"
PRED_CSV = ROOT / "baseline_model_v5/test_predictions_712.csv"

for name in ["best.pt", "last.pt"]:
    path = CKPT_DIR / name
    print(f"\n=== {name} ===")
    if not path.exists():
        print("MISSING")
        continue

    try:
        obj = torch.load(
            path,
            map_location="cpu",
            weights_only=False,
            mmap=True,
        )
        for key in ["epoch", "batch_idx", "global_step", "best_val", "history"]:
            if key in obj:
                print(f"{key}: {obj[key]}")
        del obj
    except Exception as e:
        print("Metadata read error:", repr(e))

print("\n=== TEST PREDICTIONS ===")
if PRED_CSV.exists():
    pred = pd.read_csv(PRED_CSV)
    print("Shape:", pred.shape)
    print("Columns:", pred.columns.tolist())
    print("Unique case IDs:", pred["case_id"].nunique())
    print("Null counts:")
    print(pred[["case_id", "prediction", "ground_truth"]].isna().sum())
    print("\nFirst 5 rows:")
    print(pred.head().to_string(index=False))
else:
    print("MISSING:", PRED_CSV)

print("\nREAD-ONLY: không train, không sửa checkpoint hay predictions.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

=== best.pt ===
epoch: 5
batch_idx: 0
global_step: 1920
best_val: 0.424518172663671
history: {'train_loss': [0.5731708970967303, 0.5046075203082849, 0.47171955251829634, 0.45422747778290645, 0.44715843725379206], 'val_loss': [0.46700948011622856, 0.44155955633946825, 0.42823387978095856, 0.4258775923539091, 0.424518172663671]}

=== last.pt ===
epoch: 5
batch_idx: 0
global_step: 1920
best_val: 0.424518172663671
history: {'train_loss': [0.5731708970967303, 0.5046075203082849, 0.47171955251829634, 0.45422747778290645, 0.44715843725379206], 'val_loss': [0.46700948011622856, 0.44155955633946825, 0.42823387978095856, 0.4258775923539091, 0.424518172663671]}

=== TEST PREDICTIONS ===
Shape: (712, 3)
Columns: ['case_id', 'prediction', 'ground_truth']
Unique case IDs: 712
Null counts:
case_id         0
prediction      0
ground_truth    0
dtype: int64

First 5 rows:
  

In [35]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import pandas as pd
import numpy as np
import re
import unicodedata

ROOT = Path("/content/drive/MyDrive/NoiSoi_Matching")
PRED_PATH = ROOT / "baseline_model_v5/test_predictions_712.csv"
V4_PATH = ROOT / "baseline_model_v4/v4_training_manifest.csv"
OUT_DIR = ROOT / "baseline_model_v5/diagnostics/audit_20261015"
OUT_DIR.mkdir(parents=True, exist_ok=True)

pred = pd.read_csv(PRED_PATH, dtype={"case_id": str})
v4 = pd.read_csv(V4_PATH, dtype={"case_id": str})

def norm_text(x):
    if pd.isna(x):
        return ""
    x = unicodedata.normalize("NFKC", str(x)).lower().strip()
    x = re.sub(r"[.,;:!?()\[\]{}]", " ", x)
    x = re.sub(r"\s+", " ", x)
    return x

pred["prediction_norm"] = pred["prediction"].map(norm_text)
pred["ground_truth_norm"] = pred["ground_truth"].map(norm_text)
pred["exact_match"] = (
    pred["prediction_norm"] == pred["ground_truth_norm"]
)

print("=== BASIC METRICS ===")
print("Prediction rows:", len(pred))
print("Unique prediction IDs:", pred["case_id"].nunique())
print("Exact normalized match:", round(pred["exact_match"].mean(), 4))
print("Unique normalized predictions:", pred["prediction_norm"].nunique())
print("Unique normalized ground truth:", pred["ground_truth_norm"].nunique())

print("\n=== PREDICTION DISTRIBUTION: TOP 15 ===")
print(pred["prediction_norm"].value_counts().head(15).to_string())

print("\n=== GROUND-TRUTH DISTRIBUTION: TOP 15 ===")
print(pred["ground_truth_norm"].value_counts().head(15).to_string())

print("\n=== TEST SPLIT CROSS-CHECK ===")
test_ids = set(
    v4.loc[v4["split"].astype(str).str.lower().eq("test"), "case_id"]
      .astype(str)
)
pred_ids = set(pred["case_id"].astype(str))

print("V4 test IDs:", len(test_ids))
print("Prediction IDs:", len(pred_ids))
print("Overlap:", len(test_ids & pred_ids))
print("Only in predictions:", len(pred_ids - test_ids))
print("Missing from predictions:", len(test_ids - pred_ids))

mismatch = pred.loc[
    ~pred["exact_match"],
    ["case_id", "prediction", "ground_truth"]
].copy()

out_path = OUT_DIR / "cell10_prediction_audit.csv"
mismatch.to_csv(out_path, index=False)

print("\nMismatch rows:", len(mismatch))
print("Saved mismatch audit:", out_path)
print("READ-ONLY: không train, không sửa checkpoint.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== BASIC METRICS ===
Prediction rows: 712
Unique prediction IDs: 712
Exact normalized match: 0.184
Unique normalized predictions: 16
Unique normalized ground truth: 263

=== PREDICTION DISTRIBUTION: TOP 15 ===
prediction_norm
viêm mũi mạn                                    324
viêm ống tai ngoài + màng nhĩ t cấp             126
viêm họng mạn - theo dõi trào ngược dịch vị      81
viêm mũi xoang                                   52
viêm mũi + va                                    47
viêm tai giữa p mạn                              46
viêm tai giữa t cấp viêm mũi                     10
viêm ống tai ngoài + màng nhĩ t                   8
viêm mũi xoang mạn - polyp mũi                    6
viêm ống tai ngoài 2 bên mạn                      4
viêm ống tai ngoài p cấp                          2
viêm tai giữa t mạn                               2
viêm tai giữa t cấp 

In [36]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import json
from pathlib import Path

NB_PATH = Path("/content/drive/MyDrive/Colab Notebooks/25sep.ipynb")
ROOT = Path("/content/drive/MyDrive/NoiSoi_Matching")
OUT_DIR = ROOT / "baseline_model_v5/diagnostics/audit_20261015"
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(NB_PATH, "r", encoding="utf-8") as f:
    nb = json.load(f)

keywords = [
    "test_predictions_712.csv",
    "ground_truth",
    "generate(",
    "generate_text",
    "decode(",
    "batch_decode",
    "argmax(",
    "prediction",
    "test_loader",
    "model.generate",
]

report = []
for idx, cell in enumerate(nb.get("cells", [])):
    src = "".join(cell.get("source", []))
    lines = src.splitlines()

    hits = [
        i for i, line in enumerate(lines)
        if any(k.lower() in line.lower() for k in keywords)
    ]
    if not hits:
        continue

    report.append(
        f"\n{'='*78}\n"
        f"CELL {idx} | execution_count={cell.get('execution_count')} "
        f"| lines={len(lines)}\n{'='*78}"
    )

    # In đoạn quanh các dòng liên quan, tránh in trùng.
    shown = set()
    for i in hits:
        for j in range(max(0, i - 3), min(len(lines), i + 5)):
            if j not in shown:
                report.append(f"{j+1:4}: {lines[j]}")
                shown.add(j)

text = "\n".join(report)
print(text[:30000])

path = OUT_DIR / "cell11_prediction_generation_source.txt"
path.write_text(text, encoding="utf-8")
print("\nSaved:", path)
print("READ-ONLY: không chạy inference, không train, không sửa checkpoint.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

CELL 1 | execution_count=None | lines=1914
1116: # SAFE DECODE
1117: # ================================================================
1118: 
1119: def safe_decode(ids):
1120: 
1121:     if torch.is_tensor(ids):
1122:         ids = ids.detach().cpu().tolist()
1123: 
1128:         for x in ids
1129:     ]
1130: 
1131:     return tokenizer.decode(
1132:         ids,
1133:         skip_special_tokens=True,
1134:     ).strip()
1135: 
1278:     )
1279: 
1280:     # ------------------------------------------------
1281:     # HARD PREDICTIONS
1282:     # EXACT
1283:     # ------------------------------------------------
1284: 
1285:     concept_pred = (
1413: # ================================================================
1414: 
1415: @torch.no_grad()
1416: def exact_generate(
1417:     cond,
1418:     max_new_tokens=20,
1419: ):
1420: 
1456:             retur

In [37]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import json
from pathlib import Path

NB_PATH = Path("/content/drive/MyDrive/Colab Notebooks/25sep.ipynb")
ROOT = Path("/content/drive/MyDrive/NoiSoi_Matching")
PRED_PATH = ROOT / "baseline_model_v5/test_predictions_712.csv"
OUT_DIR = ROOT / "baseline_model_v5/diagnostics/audit_20261015"
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(NB_PATH, "r", encoding="utf-8") as f:
    nb = json.load(f)

terms = [
    "test_predictions_712.csv",
    "to_csv(", "write_csv(", "csv.writer",
    "torch.load(", "load_state_dict(",
    "BEST_CKPT", "best.pt", "last.pt",
    "model.generate(", "exact_generate(",
]

report = []
for i, cell in enumerate(nb.get("cells", [])):
    src = "".join(cell.get("source", []))
    outputs = cell.get("outputs", [])
    matches = [
        (n, line.strip()[:240])
        for n, line in enumerate(src.splitlines(), start=1)
        if any(term.lower() in line.lower() for term in terms)
    ]

    # Chỉ in cell có thao tác lưu/nạp checkpoint hoặc ghi CSV,
    # và cell có cả inference lẫn xuất kết quả.
    relevant = [
        item for item in matches
        if any(t in item[1].lower() for t in [
            "test_predictions_712.csv", "to_csv(", "write_csv(",
            "csv.writer", "torch.load(", "load_state_dict(",
            "best_ckpt", "best.pt", "last.pt"
        ])
    ]
    if not relevant:
        continue

    report.append(
        f"\n=== CELL {i} | execution_count={cell.get('execution_count')} "
        f"| outputs={len(outputs)} ==="
    )
    for line_no, line in relevant:
        report.append(f"{line_no}: {line}")

    for j, output in enumerate(outputs):
        text_parts = []
        for key in ("text",):
            val = output.get(key)
            if val:
                text_parts.append(
                    "".join(val) if isinstance(val, list) else str(val)
                )
        data = output.get("data", {})
        if "text/plain" in data:
            val = data["text/plain"]
            text_parts.append(
                "".join(val) if isinstance(val, list) else str(val)
            )
        summary = " | ".join(text_parts)
        if summary:
            report.append(f"OUTPUT {j}: {summary[:500]}")

result = "\n".join(report)
print(result[:24000] if result else "Không tìm thấy các thao tác lưu/nạp checkpoint trong source cell.")

print("\n=== CSV FILE METADATA ===")
if PRED_PATH.exists():
    stat = PRED_PATH.stat()
    print("Path:", PRED_PATH)
    print("Size bytes:", stat.st_size)
    print("Modified timestamp:", __import__("datetime").datetime.fromtimestamp(
        stat.st_mtime
    ).isoformat())
else:
    print("Prediction CSV không tồn tại tại đường dẫn dự kiến.")

save_path = OUT_DIR / "cell12_checkpoint_and_prediction_source.txt"
save_path.write_text(result, encoding="utf-8")
print("\nSaved audit:", save_path)
print("READ-ONLY: không nạp model, không inference, không train, không sửa checkpoint.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

=== CELL 0 | execution_count=None | outputs=1 ===
72: BEST_CKPT = (
73: f"{V4_DIR}/checkpoints/best.pt"
OUTPUT 0: ======================================================================
V4 NEW-NOTEBOOK BOOTSTRAP
Mounted at /content/drive


=== CELL 1 | execution_count=None | outputs=24 ===
89: BEST_CKPT = (
90: f"{V4_DIR}/checkpoints/best.pt"
104: "Best checkpoint": BEST_CKPT,
988: E2 = torch.load(
994: E3 = torch.load(
1017: load_result = model.load_state_dict(
1031: model.load_state_dict(
1102: model.load_state_dict(
1507: model.load_state_dict(
1902: model.load_state_dict(
OUTPUT 0: ======================================================================
V4 NEW-NOTEBOOK BOOTSTRAP
Google Drive: already mounted

PATH CHECK
  V4 manifest           : PASS -> /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/v4_training_manifest.csv
  Structured targets   

In [38]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import json
from pathlib import Path

NB_PATH = Path("/content/drive/MyDrive/Colab Notebooks/25sep.ipynb")
OUT_DIR = Path(
    "/content/drive/MyDrive/NoiSoi_Matching/"
    "baseline_model_v5/diagnostics/audit_20261015"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(NB_PATH, "r", encoding="utf-8") as f:
    nb = json.load(f)

needles = [
    "test_predictions_712.csv",
    "to_csv(",
    "test_predictions",
    "predictions_712",
    "prediction_path",
    "pred_path",
    "save_checkpoint",
    "best.pt",
    "last.pt",
    "model.generate(",
    "exact_generate(",
]

report = []

for i, cell in enumerate(nb.get("cells", [])):
    src = "".join(cell.get("source", []))
    output_text = []

    for output in cell.get("outputs", []):
        if "text" in output:
            val = output["text"]
            output_text.append(
                "".join(val) if isinstance(val, list) else str(val)
            )

        for mime in ("text/plain", "text/html"):
            val = output.get("data", {}).get(mime)
            if val:
                output_text.append(
                    "".join(val) if isinstance(val, list) else str(val)
                )

    out = "\n".join(output_text)
    src_hits = [
        (n, line.strip())
        for n, line in enumerate(src.splitlines(), 1)
        if any(term in line.lower() for term in needles)
    ]
    out_hits = [
        (n, line.strip()[:220])
        for n, line in enumerate(out.splitlines(), 1)
        if any(term in line.lower() for term in needles)
    ]

    if src_hits or out_hits:
        report.append(
            f"\n=== CELL {i} | execution_count={cell.get('execution_count')} ==="
        )
        if src_hits:
            report.append("-- SOURCE MATCHES --")
            report.extend(f"{n}: {line[:240]}" for n, line in src_hits[:35])
        if out_hits:
            report.append("-- SAVED OUTPUT MATCHES --")
            report.extend(f"output line {n}: {line}" for n, line in out_hits[:20])

result = "\n".join(report)
print(result[:26000] if result else "Không tìm thấy dấu vết khớp.")

save_path = OUT_DIR / "cell13_full_notebook_prediction_trace.txt"
save_path.write_text(result, encoding="utf-8")
print("\nSaved:", save_path)
print("READ-ONLY: không nạp model, không inference, không train, không sửa checkpoint.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

=== CELL 0 | execution_count=None ===
-- SOURCE MATCHES --
73: f"{V4_DIR}/checkpoints/best.pt"

=== CELL 1 | execution_count=None ===
-- SOURCE MATCHES --
90: f"{V4_DIR}/checkpoints/best.pt"
1416: def exact_generate(
1533: pred_full = exact_generate(
1551: exact_generate(
1570: exact_generate(
1589: exact_generate(
-- SAVED OUTPUT MATCHES --
output line 13: Best checkpoint       : PASS -> /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/checkpoints/best.pt
output line 43: source_v3_checkpoint: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v3/checkpoints/best.pt

=== CELL 7 | execution_count=None ===
-- SOURCE MATCHES --
350: sensitivity_df.to_csv(
355: summary_df.to_csv(

=== CELL 24 | execution_count=None ===
-- SOURCE MATCHES --
262: checkpoint_paths["last"] = os.path.join(CKPT_DIR, "last.pt")
263: checkpoint_paths["best"] = os.path.join(CK

In [39]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import torch

ROOT = Path("/content/drive/MyDrive/NoiSoi_Matching")
CKPT_DIR = ROOT / "baseline_model_v5/checkpoints"

for name in ["best.pt", "last.pt"]:
    path = CKPT_DIR / name
    print(f"\n{'='*65}\n{name}\n{'='*65}")

    if not path.exists():
        print("MISSING:", path)
        continue

    print("Size (GB):", round(path.stat().st_size / (1024**3), 3))
    print("Modified:", __import__("datetime").datetime.fromtimestamp(
        path.stat().st_mtime
    ).isoformat())

    # WARNING: checkpoint ~4.3 GB; reading it can take tens of seconds
    # and temporarily consume several GB of RAM.
    # weights_only=False is required for the checkpoint's optimizer/RNG metadata.
    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    print("Epoch:", ckpt.get("epoch"))
    print("Global step:", ckpt.get("global_step", "not stored"))
    print("Best validation loss:", ckpt.get("best_val"))
    print("Top-level keys:", list(ckpt.keys()))

    state = ckpt.get("model_state", {})
    print("Model tensor count:", len(state))

    for prefix in [
        "concept_head", "negative_head", "attribute_head",
        "structured_heads", "lm_head", "mt5"
    ]:
        matches = [
            (k, tuple(v.shape))
            for k, v in state.items()
            if k.startswith(prefix)
        ]
        if matches:
            print(f"\n{prefix}:")
            for key, shape in matches[:12]:
                print(" ", key, shape)

    del state, ckpt

print("\nREAD-ONLY: không train, không optimizer step, không lưu checkpoint.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

best.pt
Size (GB): 4.321
Modified: 2026-09-29T04:20:44
Epoch: 5
Global step: 1920
Best validation loss: 0.424518172663671
Top-level keys: ['epoch', 'batch_idx', 'global_step', 'model_state', 'optimizer_state', 'scheduler_state', 'best_val', 'history', 'rng_state', 'cuda_rng_state', 'python_rng_state', 'numpy_rng_state']
Model tensor count: 403

concept_head:
  concept_head.weight (48, 512)
  concept_head.bias (48,)

negative_head:
  negative_head.weight (5, 512)
  negative_head.bias (5,)

attribute_head:
  attribute_head.weight (6, 512)
  attribute_head.bias (6,)

mt5:
  mt5.shared.weight (250112, 512)
  mt5.encoder.embed_tokens.weight (250112, 512)
  mt5.encoder.block.0.layer.0.SelfAttention.q.weight (384, 512)
  mt5.encoder.block.0.layer.0.SelfAttention.k.weight (384, 512)
  mt5.encoder.block.0.layer.0.SelfAttention.v.weight (384, 512)
  mt5.encoder.block.

In [40]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import json

ROOT = Path("/content/drive/MyDrive/NoiSoi_Matching")
V5_DIR = ROOT / "baseline_model_v5"
NOTEBOOK = Path("/content/drive/MyDrive/Colab Notebooks/25sep.ipynb")

TERMS = [
    "test_predictions_712.csv",
    "test_predictions",
    "prediction_path",
    "model.generate",
    "exact_generate",
    "argmax",
    "best.pt",
    "last.pt",
]

print("=== NOTEBOOK: prediction-related source lines ===")
if NOTEBOOK.exists():
    nb = json.loads(NOTEBOOK.read_text(encoding="utf-8"))
    hits = 0

    for i, cell in enumerate(nb.get("cells", [])):
        source = "".join(cell.get("source", []))
        matches = [
            (line_no, line.strip()[:240])
            for line_no, line in enumerate(source.splitlines(), 1)
            if any(term.lower() in line.lower() for term in TERMS)
        ]

        if matches:
            print(f"\nCell {i} | type={cell.get('cell_type')}")
            for line_no, line in matches[:12]:
                print(f"  L{line_no}: {line}")
            hits += len(matches)

    print(f"\nNotebook source matches: {hits}")
else:
    print("Notebook not found at the expected path.")

print("\n=== V5: small code/log/config files ===")
allowed = {".py", ".txt", ".log", ".json", ".yaml", ".yml", ".md"}
skip_dirs = {
    "checkpoints", "__pycache__", ".git",
    "images", "image_cache", "data", "datasets"
}
scanned = 0
found = 0

if V5_DIR.exists():
    for folder, dirs, files in __import__("os").walk(V5_DIR):
        dirs[:] = [d for d in dirs if d not in skip_dirs]

        for filename in files:
            path = Path(folder) / filename
            if path.suffix.lower() not in allowed:
                continue
            if scanned >= 500:
                print("Stopped at 500 files to keep the audit bounded.")
                break

            scanned += 1
            try:
                if path.stat().st_size > 2_000_000:
                    continue
                content = path.read_text(encoding="utf-8", errors="ignore")
                matches = [
                    (line_no, line.strip()[:220])
                    for line_no, line in enumerate(content.splitlines(), 1)
                    if any(term.lower() in line.lower() for term in TERMS)
                ]
                if matches:
                    found += 1
                    print(f"\nFILE: {path}")
                    for line_no, line in matches[:10]:
                        print(f"  L{line_no}: {line}")
            except Exception as e:
                print(f"Skipped {path.name}: {type(e).__name__}")

        if scanned >= 500:
            break

print(f"\nFiles scanned: {scanned} | Files with matches: {found}")
print("READ-ONLY audit complete.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== NOTEBOOK: prediction-related source lines ===

Cell 0 | type=code
  L73: f"{V4_DIR}/checkpoints/best.pt"

Cell 1 | type=code
  L90: f"{V4_DIR}/checkpoints/best.pt"
  L1416: def exact_generate(
  L1459: next_token = torch.argmax(
  L1533: pred_full = exact_generate(
  L1551: exact_generate(
  L1570: exact_generate(
  L1589: exact_generate(

Cell 5 | type=code
  L214: next_token = torch.argmax(

Cell 6 | type=code
  L127: # Does argmax change?
  L128: full_argmax = int(torch.argmax(full))
  L129: other_argmax = int(torch.argmax(other))
  L131: argmax_changed = (
  L132: full_argmax != other_argmax
  L142: "argmax_changed": argmax_changed,
  L144: [full_argmax],
  L148: [other_argmax],
  L228: argmax_change_rate=(
  L229: "argmax_changed",

Cell 7 | type=code
  L96: full_top = full.argmax(dim=-1)
  L97: other_top = other.argmax(dim=-1)
  L132: "argmax_change

In [41]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import json

ROOT = Path("/content/drive/MyDrive/NoiSoi_Matching")
V5 = ROOT / "baseline_model_v5"

# 1. Training summary
summary_path = V5 / "diagnostics/v5_final_training_summary.json"
print("=" * 70)
print("1. V5 FINAL TRAINING SUMMARY")

if summary_path.exists():
    try:
        summary = json.loads(summary_path.read_text(encoding="utf-8"))
        print(json.dumps(summary, ensure_ascii=False, indent=2)[:12000])
    except Exception as e:
        print("Could not parse summary:", repr(e))
else:
    print("Summary file not found.")

# 2. Runtime paths
paths_path = V5 / "runtime_state/paths.json"
print("\n" + "=" * 70)
print("2. RUNTIME PATHS")

if paths_path.exists():
    try:
        paths = json.loads(paths_path.read_text(encoding="utf-8"))
        print(json.dumps(paths, ensure_ascii=False, indent=2))
    except Exception as e:
        print("Could not parse paths:", repr(e))
else:
    print("Runtime paths file not found.")

# 3. Relevant files in V5 (metadata only)
print("\n" + "=" * 70)
print("3. RELEVANT V5 FILES")

for folder in [V5, V5 / "diagnostics", V5 / "runtime_state",
               V5 / "checkpoints"]:
    print(f"\n[{folder}]")
    if not folder.exists():
        print("  Folder not found.")
        continue

    for p in sorted(folder.iterdir(), key=lambda x: x.name.lower()):
        if p.is_file():
            stat = p.stat()
            print(
                f"  FILE {p.name} | "
                f"{stat.st_size:,} bytes | "
                f"modified={__import__('datetime').datetime.fromtimestamp(stat.st_mtime).isoformat(timespec='seconds')}"
            )
        elif p.is_dir():
            print(f"  DIR  {p.name}/")

print("\nREAD-ONLY audit complete.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1. V5 FINAL TRAINING SUMMARY
{
  "model": "V5",
  "status": "TRAINING_COMPLETE",
  "num_epochs": 5,
  "global_step": 1920,
  "epoch": 5,
  "next_batch": 0,
  "best_val": 0.424518172663671,
  "train_loss": [
    0.5731708970967303,
    0.5046075203082849,
    0.47171955251829634,
    0.45422747778290645,
    0.44715843725379206
  ],
  "val_loss": [
    0.46700948011622856,
    0.44155955633946825,
    0.42823387978095856,
    0.4258775923539091,
    0.424518172663671
  ],
  "last_checkpoint": "/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/checkpoints/last.pt",
  "best_checkpoint": "/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/checkpoints/best.pt"
}

2. RUNTIME PATHS
{
  "v4_manifest": "/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/v4_training_manifest.csv",
  "v4_checkpoint": "/content/drive/MyDrive/NoiSoi_Matching/baseline_mo

In [42]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
from collections import Counter
import pandas as pd
import json
import os
import re

ROOT = Path("/content/drive/MyDrive/NoiSoi_Matching")
V5 = ROOT / "baseline_model_v5"
AUDIT = V5 / "diagnostics/audit_20261015"

def section(title):
    print("\n" + "=" * 88)
    print(title)
    print("=" * 88)

def norm_text(x):
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x).strip().lower())

# ------------------------------------------------------------
# 1. ORIGINAL PREDICTIONS: distribution and direct comparison
# ------------------------------------------------------------
section("1. TEST PREDICTIONS — COUNTS, LABEL DISTRIBUTION, EXACT MATCH")

pred_path = V5 / "test_predictions_712.csv"

if pred_path.exists():
    pred = pd.read_csv(pred_path)
    print("Path:", pred_path)
    print("Shape:", pred.shape)
    print("Columns:", pred.columns.tolist())

    if "case_id" in pred.columns:
        print("Unique case_id:", pred["case_id"].nunique())
        print("Duplicate case_id rows:", int(pred["case_id"].duplicated().sum()))

    if {"prediction", "ground_truth"}.issubset(pred.columns):
        pred["_p_norm"] = pred["prediction"].map(norm_text)
        pred["_g_norm"] = pred["ground_truth"].map(norm_text)

        print("Unique predictions:", pred["_p_norm"].nunique())
        print("Unique ground truths:", pred["_g_norm"].nunique())
        print("Exact normalized match rate:",
              round(float((pred["_p_norm"] == pred["_g_norm"]).mean()), 4))

        print("\nTop 15 predictions:")
        print(pred["prediction"].value_counts(dropna=False).head(15).to_string())

        print("\nTop 15 ground truths:")
        print(pred["ground_truth"].value_counts(dropna=False).head(15).to_string())

        print("\nFirst 8 mismatches:")
        mismatch = pred[pred["_p_norm"] != pred["_g_norm"]]
        print(mismatch[["case_id", "prediction", "ground_truth"]]
              .head(8).to_string(index=False))

        print("\nPrediction frequency quantiles:")
        freq = pred["prediction"].value_counts()
        print(freq.describe(percentiles=[.25, .5, .75, .9, .95]).to_string())
else:
    print("Prediction CSV not found.")

# ------------------------------------------------------------
# 2. EXISTING AUDIT CSV REPORTS
# ------------------------------------------------------------
section("2. EXISTING AUDIT CSV FILES")

audit_files = [
    V5 / "prediction_frequency_audit.csv",
    V5 / "prediction_quality_audit.csv",
    V5 / "distribution_shift_audit.csv",
    V5 / "text_overlap_audit.csv",
    V5 / "low_overlap_image_metadata.csv",
    V5 / "anatomy_error_stratification.csv",
    V5 / "tympanic_membrane_cases.csv",
    AUDIT / "cell10_prediction_audit.csv",
]

for path in audit_files:
    if not path.exists():
        print(f"\nMISSING: {path.name}")
        continue

    try:
        df = pd.read_csv(path)
        print(f"\n--- {path.name} | shape={df.shape} ---")
        print("Columns:", df.columns.tolist())

        # Show a compact sample of existing audit results.
        if len(df):
            print(df.head(4).to_string(index=False, max_cols=12, max_colwidth=90))

        # Useful numeric summary without dumping the full table.
        numeric = df.select_dtypes(include="number")
        if not numeric.empty:
            print("Numeric summary:")
            print(numeric.describe().round(4).to_string(max_rows=8, max_cols=8))

    except Exception as e:
        print(f"Could not read {path.name}: {type(e).__name__}: {e}")

# ------------------------------------------------------------
# 3. CHECK AUDIT TEXT REPORTS FOR CONCLUSIONS / EVIDENCE
# ------------------------------------------------------------
section("3. EXISTING TEXT AUDITS — RELEVANT LINES")

text_files = [
    AUDIT / "cell1_schema_ontology_audit.txt",
    AUDIT / "cell6_training_source_audit.txt",
    AUDIT / "cell7_v5_training_flow.txt",
    AUDIT / "cell8_saved_training_history.txt",
    AUDIT / "cell11_prediction_generation_source.txt",
    AUDIT / "cell12_checkpoint_and_prediction_source.txt",
    AUDIT / "cell13_full_notebook_prediction_trace.txt",
]

keywords = [
    "test_predictions_712.csv", "to_csv(", "to_csv ",
    "prediction_quality_audit", "prediction_frequency_audit",
    "exact_generate", "model.generate", "argmax",
    "ground_truth", "prediction_path", "save_path",
    "checkpoint", "best.pt", "last.pt",
]

for path in text_files:
    if not path.exists():
        continue

    print(f"\n--- {path.name} ---")
    try:
        lines = path.read_text(encoding="utf-8", errors="ignore").splitlines()
        hits = [
            (i + 1, line.strip()[:240])
            for i, line in enumerate(lines)
            if any(k.lower() in line.lower() for k in keywords)
        ]
        if not hits:
            print("No matching lines.")
        else:
            for line_no, line in hits[:12]:
                print(f"L{line_no}: {line}")
            if len(hits) > 12:
                print(f"... {len(hits) - 12} additional matching lines omitted.")
    except Exception as e:
        print("Read error:", repr(e))

# ------------------------------------------------------------
# 4. PROJECT-LEVEL CODE / LOG SEARCH, EXCLUDING LARGE DATA
# ------------------------------------------------------------
section("4. PROJECT CODE/LOG SEARCH FOR CSV GENERATION")

search_terms = [
    "test_predictions_712.csv",
    "prediction_quality_audit.csv",
    "prediction_frequency_audit.csv",
    "to_csv(",
    "to_csv ",
]
extensions = {".py", ".ipynb", ".txt", ".log", ".json", ".yaml", ".yml", ".md"}
skip_dirs = {
    "checkpoints", "images", "image_cache", "data", "datasets",
    "__pycache__", ".git"
}

scanned = 0
hits_found = 0
MAX_FILES = 700
MAX_MATCHES_PER_FILE = 5

for folder, dirs, files in os.walk(ROOT):
    dirs[:] = [d for d in dirs if d not in skip_dirs]

    for filename in files:
        path = Path(folder) / filename
        if path.suffix.lower() not in extensions:
            continue
        if scanned >= MAX_FILES:
            break

        scanned += 1
        try:
            if path.stat().st_size > 3_000_000:
                continue
            content = path.read_text(encoding="utf-8", errors="ignore")
            matches = []
            for i, line in enumerate(content.splitlines(), 1):
                if any(term.lower() in line.lower() for term in search_terms):
                    matches.append((i, line.strip()[:220]))
                    if len(matches) >= MAX_MATCHES_PER_FILE:
                        break

            if matches:
                hits_found += 1
                print(f"\nFILE: {path}")
                for line_no, line in matches:
                    print(f"  L{line_no}: {line}")
        except Exception:
            pass

    if scanned >= MAX_FILES:
        print(f"\nSearch stopped at the safety limit of {MAX_FILES} files.")
        break

print(f"\nScanned files: {scanned}")
print(f"Files with matches: {hits_found}")

section("AUDIT COMPLETE")
print("READ-ONLY: no training, no optimizer step, no checkpoint writes.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

1. TEST PREDICTIONS — COUNTS, LABEL DISTRIBUTION, EXACT MATCH
Path: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/test_predictions_712.csv
Shape: (712, 3)
Columns: ['case_id', 'prediction', 'ground_truth']
Unique case_id: 712
Duplicate case_id rows: 0
Unique predictions: 16
Unique ground truths: 267
Exact normalized match rate: 0.184

Top 15 predictions:
prediction
VIÊM MŨI MẠN                                    324
VIÊM ỐNG TAI NGOÀI + MÀNG NHĨ T CẤP             126
VIÊM HỌNG MẠN - THEO DÕI TRÀO NGƯỢC DỊCH VỊ.     81
VIÊM MŨI XOANG                                   52
VIÊM MŨI + VA                                    47
VIÊM TAI GIỮA P MẠN                              46
VIÊM TAI GIỮA T CẤP VIÊM MŨI                     10
VIÊM ỐNG TAI NGOÀI + MÀNG NHĨ T                   8
VIÊM MŨI XOANG MẠN - POLYP MŨI                    6
VIÊM ỐNG TAI NGOÀI 2 BÊ

In [43]:

# ============================================================
# V5 INFERENCE PROVENANCE AUDIT — READ ONLY
# Mục tiêu: truy vết code/checkpoint đã tạo test_predictions_712.csv
# Không training, không load checkpoint, không sửa file dự án
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import re
import json
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/NoiSoi_Matching')
NB_PATH = Path('/content/drive/MyDrive/Colab Notebooks/25sep.ipynb')
V5 = ROOT / 'baseline_model_v5'
AUDIT = V5 / 'diagnostics' / 'audit_20261015'

print('=' * 100)
print('V5 INFERENCE PROVENANCE AUDIT — READ ONLY')
print('=' * 100)
print('Notebook exists:', NB_PATH.exists())
print('Project exists :', ROOT.exists())
print('V5 directory   :', V5.exists())

# ------------------------------------------------------------
# 1. Read notebook and identify cells related to inference,
#    predictions, checkpoint loading, and CSV writing.
# ------------------------------------------------------------
if not NB_PATH.exists():
    print('\nERROR: Notebook not found:', NB_PATH)
else:
    with open(NB_PATH, 'r', encoding='utf-8') as f:
        nb = json.load(f)

    cells = nb.get('cells', [])
    print(f'\nNotebook cells: {len(cells)}')

    terms = [
        'test_predictions_712.csv',
        'prediction_path',
        'prediction_df',
        'predictions_df',
        'model.generate',
        'exact_generate',
        'generate(',
        'torch.load(',
        'load_state_dict',
        'from_pretrained',
        'best.pt',
        'last.pt',
        'checkpoint',
        'ground_truth',
        'prediction',
        'to_csv(',
        'argmax',
        'AutoModel',
        'T5ForConditionalGeneration',
        'MT5ForConditionalGeneration',
    ]

    hits = []
    for i, cell in enumerate(cells):
        source = ''.join(cell.get('source', []))
        outputs = cell.get('outputs', [])
        output_texts = []

        for out in outputs:
            if 'text' in out:
                value = out['text']
                output_texts.append(
                    ''.join(value) if isinstance(value, list) else str(value)
                )
            data = out.get('data', {})
            if 'text/plain' in data:
                value = data['text/plain']
                output_texts.append(
                    ''.join(value) if isinstance(value, list) else str(value)
                )

        out_text = '\n'.join(output_texts)
        combined = source + '\n' + out_text
        found = [t for t in terms if t.lower() in combined.lower()]

        if found:
            hits.append({
                'cell_index': i,
                'cell_type': cell.get('cell_type'),
                'execution_count': cell.get('execution_count'),
                'terms': found,
                'source': source,
                'output': out_text,
            })

    print(f'Cells with relevant terms: {len(hits)}')

    # First show cells mentioning the exact prediction CSV path/name.
    exact_hits = [
        h for h in hits
        if 'test_predictions_712.csv' in
        (h['source'] + '\n' + h['output']).lower()
    ]

    print('\n' + '=' * 100)
    print('SECTION 1 — EXACT CSV NAME IN NOTEBOOK')
    print('=' * 100)
    print('Exact-name hits:', len(exact_hits))

    if not exact_hits:
        print('No notebook cell source/output explicitly mentions test_predictions_712.csv.')
    else:
        for h in exact_hits:
            print(
                f"\n--- Cell {h['cell_index']} | type={h['cell_type']} "
                f"| execution_count={h['execution_count']} "
                f"| terms={h['terms']} ---"
            )
            print('SOURCE:')
            print(h['source'][:7000] or '[empty source]')
            print('OUTPUT:')
            print(h['output'][:2500] or '[no captured text output]')

    # Then show cells with model loading / generation / CSV writing.
    # Print compact snippets, prioritizing actual source over noisy output.
    print('\n' + '=' * 100)
    print('SECTION 2 — INFERENCE / CHECKPOINT / CSV-WRITING CELLS')
    print('=' * 100)

    priority_terms = [
        'torch.load(', 'load_state_dict', 'from_pretrained',
        'model.generate', 'exact_generate', 'generate(',
        'to_csv(', 'best.pt', 'last.pt'
    ]

    priority_hits = []
    for h in hits:
        src = h['source'].lower()
        if any(t.lower() in src for t in priority_terms):
            priority_hits.append(h)

    print('Relevant source cells:', len(priority_hits))

    for h in priority_hits:
        src = h['source']
        lines = src.splitlines()
        relevant_lines = set()

        for line_no, line in enumerate(lines):
            if any(t.lower() in line.lower() for t in priority_terms):
                for j in range(max(0, line_no - 3), min(len(lines), line_no + 5)):
                    relevant_lines.add(j)

        selected = sorted(relevant_lines)
        print(
            f"\n--- Cell {h['cell_index']} | execution_count={h['execution_count']} "
            f"| matches={h['terms']} | source_lines={len(lines)} ---"
        )

        if not selected:
            print(src[:2500])
        else:
            # Limit output per cell so the notebook audit remains readable.
            selected = selected[:100]
            for j in selected:
                print(f'{j + 1:4d}: {lines[j]}')

        # Show only a short error/result excerpt from captured output.
        if h['output']:
            out = h['output']
            markers = [
                'Traceback', 'Error', 'Exception', 'PASS',
                'FAIL', 'checkpoint', 'prediction', 'saved'
            ]
            out_lines = out.splitlines()
            chosen = [
                (j, line) for j, line in enumerate(out_lines)
                if any(m.lower() in line.lower() for m in markers)
            ][:15]
            if chosen:
                print('OUTPUT EXCERPT:')
                for j, line in chosen:
                    print(f'  OUT {j + 1}: {line[:350]}')

    # --------------------------------------------------------
    # 2. Inspect existing provenance/audit reports.
    # --------------------------------------------------------
    print('\n' + '=' * 100)
    print('SECTION 3 — EXISTING AUDIT REPORTS')
    print('=' * 100)

    report_names = [
        'cell1_schema_ontology_audit.txt',
        'cell6_training_source_audit.txt',
        'cell7_v5_training_flow.txt',
        'cell8_saved_training_history.txt',
        'cell11_prediction_generation_source.txt',
        'cell12_checkpoint_and_prediction_source.txt',
        'cell13_full_notebook_prediction_trace.txt',
    ]

    report_terms = [
        'test_predictions_712.csv',
        'prediction generation',
        'prediction source',
        'inference',
        'torch.load',
        'load_state_dict',
        'best.pt',
        'last.pt',
        'exact_generate',
        'to_csv',
        'missing',
        'error',
        'traceback',
        'execution_count',
        'not found',
    ]

    for name in report_names:
        path = AUDIT / name
        if not path.exists():
            print(f'\n--- MISSING REPORT: {name} ---')
            continue

        try:
            lines = path.read_text(encoding='utf-8', errors='replace').splitlines()
        except Exception as e:
            print(f'\n--- READ ERROR: {name}: {e} ---')
            continue

        matched = []
        for i, line in enumerate(lines):
            if any(term.lower() in line.lower() for term in report_terms):
                matched.append(i)

        # Add nearby lines around matched entries.
        selected = set()
        for i in matched:
            for j in range(max(0, i - 2), min(len(lines), i + 3)):
                selected.add(j)

        print(f'\n--- {name} | lines={len(lines)} | matched_lines={len(matched)} ---')
        if not selected:
            print('[No matching provenance terms found.]')
        else:
            # Avoid huge reports overwhelming the output.
            for j in sorted(selected)[:100]:
                print(f'L{j + 1}: {lines[j][:500]}')

    # --------------------------------------------------------
    # 3. Verify prediction file metadata without changing it.
    # --------------------------------------------------------
    print('\n' + '=' * 100)
    print('SECTION 4 — PREDICTION FILE METADATA')
    print('=' * 100)

    pred_path = V5 / 'test_predictions_712.csv'
    if pred_path.exists():
        stat = pred_path.stat()
        print('Path:', pred_path)
        print('Bytes:', stat.st_size)
        print('Modified time (local runtime):', __import__('datetime').datetime.fromtimestamp(stat.st_mtime))
        print('CSV file hash (SHA256):')
        import hashlib
        sha = hashlib.sha256()
        with open(pred_path, 'rb') as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b''):
                sha.update(chunk)
        print(sha.hexdigest())
    else:
        print('Prediction CSV does not exist:', pred_path)

    # List provenance-like small files near the V5 root.
    print('\nPotential provenance files:')
    for base in [V5, V5 / 'diagnostics', AUDIT, V5 / 'runtime_state']:
        if not base.exists():
            continue
        for p in sorted(base.iterdir()):
            if p.is_file() and any(
                word in p.name.lower()
                for word in ['summary', 'history', 'prediction', 'inference',
                             'source', 'runtime', 'path', 'log', 'audit']
            ):
                try:
                    print(f'{p} | {p.stat().st_size:,} bytes')
                except OSError:
                    print(p)

    print('\n' + '=' * 100)
    print('AUDIT FINISHED — READ ONLY')
    print('No checkpoint loaded; no training; no files written.')
    print('=' * 100)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
V5 INFERENCE PROVENANCE AUDIT — READ ONLY
Notebook exists: True
Project exists : True
V5 directory   : True

Notebook cells: 162
Cells with relevant terms: 100

SECTION 1 — EXACT CSV NAME IN NOTEBOOK
Exact-name hits: 0
No notebook cell source/output explicitly mentions test_predictions_712.csv.

SECTION 2 — INFERENCE / CHECKPOINT / CSV-WRITING CELLS
Relevant source cells: 47

--- Cell 0 | execution_count=None | matches=['best.pt', 'checkpoint', 'T5ForConditionalGeneration', 'MT5ForConditionalGeneration'] | source_lines=74 ---
  70: )
  71: 
  72: BEST_CKPT = (
  73:     f"{V4_DIR}/checkpoints/best.pt"
  74: )

--- Cell 1 | execution_count=None | matches=['exact_generate', 'generate(', 'torch.load(', 'load_state_dict', 'from_pretrained', 'best.pt', 'checkpoint', 'prediction', 'argmax', 'T5ForConditionalGeneration', 'MT5ForConditionalGeneration'] | source_lines

In [45]:
# ============================================================
# V5 CHECKPOINT ↔ PREDICTION FORENSIC AUDIT — READ ONLY
# Không train, không ghi file, không load toàn bộ checkpoint 4.3GB
# ============================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os
import re
import json
import hashlib
import ast
import pandas as pd
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/NoiSoi_Matching')
NB_PATH = Path('/content/drive/MyDrive/Colab Notebooks/25sep.ipynb')
V5 = ROOT / 'baseline_model_v5'
PRED = V5 / 'test_predictions_712.csv'
CKPT_DIR = V5 / 'checkpoints'

print('=' * 88)
print('V5 CHECKPOINT ↔ PREDICTION FORENSIC AUDIT — READ ONLY')
print('=' * 88)

# ------------------------------------------------------------
# 1. Existing artifact metadata; do not load checkpoint weights
# ------------------------------------------------------------
print('\n[1] FILE METADATA')

for p in [
    PRED,
    CKPT_DIR / 'best.pt',
    CKPT_DIR / 'last.pt',
    V5 / 'diagnostics' / 'v5_final_training_summary.json',
]:
    if p.exists():
        s = p.stat()
        print(f'{p.name:38s} exists=True size={s.st_size:,} bytes '
              f'mtime={pd.Timestamp(s.st_mtime, unit="s")}')
    else:
        print(f'{p.name:38s} exists=False')

# ------------------------------------------------------------
# 2. Prediction CSV fingerprint and schema
# ------------------------------------------------------------
print('\n[2] PREDICTION CSV')

if PRED.exists():
    df = pd.read_csv(PRED)
    print('shape:', df.shape)
    print('columns:', df.columns.tolist())
    print('dtypes:', df.dtypes.astype(str).to_dict())
    print('duplicate case_id:', int(df['case_id'].duplicated().sum())
          if 'case_id' in df else 'case_id missing')

    if {'prediction', 'ground_truth'}.issubset(df.columns):
        pred = df['prediction'].fillna('').astype(str).str.strip().str.lower()
        truth = df['ground_truth'].fillna('').astype(str).str.strip().str.lower()
        print('unique predictions:', pred.nunique())
        print('unique ground truths:', truth.nunique())
        print('exact normalized match rate:', round(float((pred == truth).mean()), 4))
        print('\nTop 12 predictions:')
        print(df['prediction'].value_counts(dropna=False).head(12).to_string())
else:
    df = None
    print('PREDICTION CSV NOT FOUND')

if PRED.exists():
    h = hashlib.sha256()
    with open(PRED, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    print('SHA256:', h.hexdigest())

# ------------------------------------------------------------
# 3. Training summary JSON; safe small metadata file only
# ------------------------------------------------------------
print('\n[3] V5 TRAINING SUMMARY')

summary_path = V5 / 'diagnostics' / 'v5_final_training_summary.json'
if summary_path.exists():
    try:
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        print(json.dumps(summary, ensure_ascii=False, indent=2)[:14000])
    except Exception as e:
        print('Could not parse summary:', repr(e))
else:
    print('Summary JSON not found.')

# ------------------------------------------------------------
# 4. Scan small text/code/config files for explicit inference provenance
# ------------------------------------------------------------
print('\n[4] SEARCH PROJECT TEXT/CONFIG FILES')

needles = [
    'test_predictions_712.csv',
    'prediction_path',
    'prediction_csv',
    'pd.DataFrame',
    'model.generate(',
    'generate(**',
    'argmax(',
    'best.pt',
    'last.pt',
    'load_state_dict(',
]

skip_parts = {
    'checkpoints', 'images', 'image', 'dataset', 'datasets',
    '.git', '__pycache__'
}
extensions = {'.py', '.ipynb', '.json', '.txt', '.yaml', '.yml', '.log', '.md'}
hits = 0
max_hits = 100

for base in [V5, ROOT / 'baseline_model_v4', ROOT / 'baseline_model_v3']:
    if not base.exists():
        continue
    for p in base.rglob('*'):
        if not p.is_file() or p.suffix.lower() not in extensions:
            continue
        if any(part in skip_parts for part in p.parts):
            continue
        try:
            if p.stat().st_size > 3_000_000:
                continue
            text = p.read_text(encoding='utf-8', errors='ignore')
        except Exception:
            continue

        lines = text.splitlines()
        matching = [
            (i + 1, line.strip()[:240])
            for i, line in enumerate(lines)
            if any(term.lower() in line.lower() for term in needles)
        ]
        if matching:
            print(f'\nFILE: {p}')
            for line_no, line in matching[:8]:
                print(f'  L{line_no}: {line}')
                hits += 1
                if hits >= max_hits:
                    break
        if hits >= max_hits:
            print('\nHit limit reached.')
            break
    if hits >= max_hits:
        break

print('\nText/config matches printed:', hits)

# ------------------------------------------------------------
# 5. Notebook: focus only on cells that might create predictions
# ------------------------------------------------------------
print('\n[5] NOTEBOOK INFERENCE SOURCE')

if NB_PATH.exists():
    try:
        nb = json.loads(NB_PATH.read_text(encoding='utf-8'))
        cells = nb.get('cells', [])
        print('Notebook cells:', len(cells))

        terms = [
            'test_predictions_712.csv',
            'prediction_path',
            'to_csv(',
            'model.generate(',
            'generate(**',
            'argmax(',
            'torch.load(',
            'load_state_dict(',
            'baseline_model_v5/checkpoints',
        ]

        found = 0
        for i, cell in enumerate(cells):
            source = ''.join(cell.get('source', []))
            matched = [
                (j + 1, line.strip()[:220])
                for j, line in enumerate(source.splitlines())
                if any(term.lower() in line.lower() for term in terms)
            ]
            if not matched:
                continue

            # Only print cells containing prediction-generation or CSV-write clues.
            print(f'\nCELL {i} | execution_count={cell.get("execution_count")}')
            for line_no, line in matched[:12]:
                print(f'  L{line_no}: {line}')
            found += 1
            if found >= 20:
                print('Cell display limit reached.')
                break

        print('\nRelevant source cells printed:', found)
    except Exception as e:
        print('Notebook parse error:', repr(e))
else:
    print('Notebook not found.')

# ------------------------------------------------------------
# 6. Existing diagnostics that might describe prediction collapse
# ------------------------------------------------------------
print('\n[6] EXISTING DIAGNOSTIC SCHEMAS')

for p in [
    V5 / 'diagnostics' / 'prediction_frequency_audit.csv',
    V5 / 'diagnostics' / 'prediction_quality_audit.csv',
    V5 / 'diagnostics' / 'distribution_shift_audit.csv',
]:
    if p.exists():
        try:
            x = pd.read_csv(p, nrows=5)
            print(f'\n{p.name}:')
            print('columns:', x.columns.tolist())
            print(x.to_string(index=False, max_cols=14))
        except Exception as e:
            print(p.name, 'read error:', repr(e))
    else:
        print(p.name, 'not found')

print('\n' + '=' * 88)
print('AUDIT FINISHED — NO TRAINING, NO CHECKPOINT WEIGHT LOAD, NO FILE WRITES')
print('=' * 88)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
V5 CHECKPOINT ↔ PREDICTION FORENSIC AUDIT — READ ONLY

[1] FILE METADATA
test_predictions_712.csv               exists=True size=62,941 bytes mtime=2026-09-30 16:34:27
best.pt                                exists=True size=4,640,037,647 bytes mtime=2026-09-29 04:20:44
last.pt                                exists=True size=4,640,037,647 bytes mtime=2026-09-29 04:25:51
v5_final_training_summary.json         exists=True size=649 bytes mtime=2026-09-29 04:23:10

[2] PREDICTION CSV
shape: (712, 3)
columns: ['case_id', 'prediction', 'ground_truth']
dtypes: {'case_id': 'object', 'prediction': 'object', 'ground_truth': 'object'}
duplicate case_id: 0
unique predictions: 16
unique ground truths: 267
exact normalized match rate: 0.184

Top 12 predictions:
prediction
VIÊM MŨI MẠN                                    324
VIÊM ỐNG TAI NGOÀI + MÀNG NHĨ T CẤP             126

In [46]:
# ============================================================
# V5 SOURCE TRACE — READ ONLY
# Mục tiêu: tìm luồng inference có thể liên quan đến predictions
# ============================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import json
from pathlib import Path

NB_PATH = Path('/content/drive/MyDrive/Colab Notebooks/25sep.ipynb')
AUDIT_DIR = Path(
    '/content/drive/MyDrive/NoiSoi_Matching/'
    'baseline_model_v5/diagnostics/audit_20261015'
)

print('=' * 90)
print('V5 SOURCE TRACE — READ ONLY')
print('=' * 90)

if not NB_PATH.exists():
    print('ERROR: Notebook not found:', NB_PATH)
else:
    nb = json.loads(NB_PATH.read_text(encoding='utf-8'))
    cells = nb.get('cells', [])
    print('Notebook cells:', len(cells))

    # Những dấu hiệu có thể giúp nhận diện luồng inference.
    inference_terms = [
        'test_predictions_712.csv',
        'prediction_path',
        'predictions_df',
        'predictions =',
        'model.generate(',
        'model.generate (',
        'generate(',
        'argmax(',
        'decode(',
        'batch_decode(',
        'prediction',
        'ground_truth',
        'to_csv(',
        'to_csv (',
    ]

    checkpoint_terms = [
        'torch.load(',
        'load_state_dict(',
        'best.pt',
        'last.pt',
        'V5_CKPT_DIR',
        'CKPT_DIR',
        'AutoModel',
        'from_pretrained(',
        'class ',
        'def collate_v5',
    ]

    # 1. Tìm mọi cell có dấu hiệu ghi dự đoán hoặc xử lý nhãn.
    print('\n' + '=' * 90)
    print('[1] CELLS WITH PREDICTION / CSV-WRITING CLUES')
    print('=' * 90)

    candidate_cells = []
    for idx, cell in enumerate(cells):
        src = ''.join(cell.get('source', []))
        low = src.lower()
        matched = [t for t in inference_terms if t.lower() in low]

        # Ưu tiên các cell vừa xử lý prediction vừa nạp checkpoint/model.
        if matched:
            candidate_cells.append((idx, cell, matched, src))

    print('Candidate cells:', len(candidate_cells))

    for idx, cell, matched, src in candidate_cells:
        lines = src.splitlines()
        low_lines = [line.lower() for line in lines]

        # Cell có đúng tên file là bằng chứng mạnh nhất.
        exact_name = 'test_predictions_712.csv' in src
        checkpoint_hits = [
            i for i, line in enumerate(low_lines)
            if any(t.lower() in line for t in checkpoint_terms)
        ]
        prediction_hits = [
            i for i, line in enumerate(low_lines)
            if any(t.lower() in line for t in inference_terms)
        ]

        print(
            f'\nCELL {idx} | execution_count={cell.get("execution_count")} '
            f'| exact_csv_name={exact_name}'
        )
        print('Matched terms:', matched)

        # In tất cả dòng khớp chính xác tên file; nếu không có,
        # in các dòng quan trọng về model/checkpoint/prediction.
        if exact_name:
            selected = set()
            for i, line in enumerate(lines):
                if 'test_predictions_712.csv' in line:
                    selected.update(range(max(0, i-8), min(len(lines), i+9)))
        else:
            selected = set()
            important = sorted(set(checkpoint_hits + prediction_hits))
            # Giữ output gọn, tối đa 30 dòng ngữ cảnh mỗi cell.
            for i in important:
                selected.update(range(max(0, i-2), min(len(lines), i+3)))
                if len(selected) >= 30:
                    break

        for i in sorted(selected)[:36]:
            print(f'  L{i+1}: {lines[i][:240]}')

        # Đọc output đã lưu của cell, chỉ in những dòng có liên quan.
        outputs = cell.get('outputs', [])
        output_matches = []
        for out in outputs:
            data = out.get('data', {})
            text_parts = []
            if 'text' in out:
                text_parts.extend(out['text'] if isinstance(out['text'], list)
                                  else [out['text']])
            if 'text/plain' in data:
                text_parts.extend(
                    data['text/plain'] if isinstance(data['text/plain'], list)
                    else [data['text/plain']]
                )
            for part in text_parts:
                for line in str(part).splitlines():
                    if any(k in line.lower() for k in [
                        'prediction', 'checkpoint', 'best.pt', 'last.pt',
                        'test_predictions', 'accuracy', 'exact match',
                        'saved', 'error:', 'exception',
                    ]):
                        output_matches.append(line.strip()[:240])

        if output_matches:
            print('  SAVED OUTPUT CLUES:')
            for line in output_matches[:8]:
                print('   ', line)

    # 2. Đọc báo cáo provenance nếu có.
    print('\n' + '=' * 90)
    print('[2] EXISTING PROVENANCE REPORTS')
    print('=' * 90)

    report_names = [
        'cell11_prediction_generation_source.txt',
        'cell12_checkpoint_and_prediction_source.txt',
        'cell13_full_notebook_prediction_trace.txt',
        'cell6_training_source_audit.txt',
        'cell7_v5_training_flow.txt',
    ]

    terms = [
        'test_predictions_712.csv',
        'prediction_path',
        'to_csv',
        'model.generate',
        'generate(',
        'best.pt',
        'last.pt',
        'load_state_dict',
        'prediction source',
        'inference',
    ]

    for name in report_names:
        p = AUDIT_DIR / name
        print(f'\n--- {name} ---')
        if not p.exists():
            print('Not found at:', p)
            continue

        lines = p.read_text(encoding='utf-8', errors='ignore').split

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
V5 SOURCE TRACE — READ ONLY
Notebook cells: 162

[1] CELLS WITH PREDICTION / CSV-WRITING CLUES
Candidate cells: 7

CELL 1 | execution_count=None | exact_csv_name=False
Matched terms: ['generate(', 'argmax(', 'decode(', 'prediction']
  L88: 
  L89: BEST_CKPT = (
  L90:     f"{V4_DIR}/checkpoints/best.pt"
  L91: )
  L92: 
  L171: print("\nLOADING TOKENIZER")
  L172: 
  L173: tokenizer = AutoTokenizer.from_pretrained(
  L174:     "google/mt5-small"
  L175: )
  L197: print("\nLOADING IMAGE PROCESSOR")
  L198: 
  L199: image_processor = AutoImageProcessor.from_pretrained(
  L200:     "google/vit-base-patch16-224"
  L201: )
  L495: # ================================================================
  L496: 
  L497: class V4ValidationDataset(Dataset):
  L498: 
  L499:     def __init__(
  L686: # ================================================================
  L687:

In [47]:
# ============================================================
# V5 INFERENCE TRACE — DEEP READ-ONLY SCAN
# Không train, không nạp checkpoint, không ghi/chỉnh sửa file
# ============================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import json
from pathlib import Path

NB_PATH = Path('/content/drive/MyDrive/Colab Notebooks/25sep.ipynb')
PROJECT = Path('/content/drive/MyDrive/NoiSoi_Matching')
V5 = PROJECT / 'baseline_model_v5'

nb = json.loads(NB_PATH.read_text(encoding='utf-8'))
cells = nb.get('cells', [])

print('=' * 90)
print('DEEP V5 INFERENCE TRACE')
print('=' * 90)

# A. Quét toàn bộ cell 70 trở đi, không chỉ dựa vào một từ khóa.
terms = [
    'torch.load', 'load_state_dict', 'model.eval', '.eval()',
    'model.generate', 'generate(', 'greedy_search',
    'argmax(', 'batch_decode', 'tokenizer.decode',
    'safe_decode', 'prediction', 'predictions',
    'ground_truth', 'to_csv', 'test_loader', 'test_dataset',
    'best.pt', 'last.pt', 'checkpoint', 'save_path',
    'prediction_path', 'csv_path', 'DataFrame(',
    'V5Model(', 'from_pretrained'
]

print('\n[A] SOURCE HITS — CELLS 70+')
for idx in range(70, len(cells)):
    cell = cells[idx]
    src = ''.join(cell.get('source', []))
    lines = src.splitlines()

    hits = [
        (i, line) for i, line in enumerate(lines)
        if any(term.lower() in line.lower() for term in terms)
    ]
    if not hits:
        continue

    print(f'\n--- CELL {idx} | execution_count={cell.get("execution_count")} '
          f'| lines={len(lines)} | hits={len(hits)} ---')

    # In các dòng khớp và một ít ngữ cảnh trước/sau.
    selected = set()
    for i, _ in hits:
        selected.update(range(max(0, i - 2), min(len(lines), i + 3)))

    for i in sorted(selected):
        print(f'L{i+1}: {lines[i][:260]}')

# B. Tìm chính xác mọi chuỗi liên quan đến CSV trong toàn bộ notebook.
print('\n' + '=' * 90)
print('[B] EXACT CSV / OUTPUT PATH REFERENCES')
print('=' * 90)

needles = [
    'test_predictions_712.csv',
    'test_predictions',
    'prediction_path',
    'predictions_path',
    'to_csv(',
    'to_csv (',
]
found = 0
for idx, cell in enumerate(cells):
    src = ''.join(cell.get('source', []))
    for i, line in enumerate(src.splitlines()):
        if any(n.lower() in line.lower() for n in needles):
            print(f'CELL {idx}, L{i+1}: {line[:300]}')
            found += 1

print('Total matching source lines:', found)

# C. Liệt kê tên các file nhỏ liên quan đến prediction/inference trong V5.
print('\n' + '=' * 90)
print('[C] V5 FILE INVENTORY — RELEVANT FILENAMES')
print('=' * 90)

if V5.exists():
    for p in sorted(V5.rglob('*')):
        if not p.is_file():
            continue
        rel = str(p.relative_to(V5))
        low = rel.lower()

        if any(k in low for k in [
            'predict', 'infer', 'generation', 'summary',
            'config', 'runtime', 'audit', 'history', 'log'
        ]):
            # Chỉ đọc metadata; bỏ qua nội dung checkpoint và file dữ liệu lớn.
            try:
                stat = p.stat()
                print(f'{rel} | {stat.st_size:,} bytes')
            except OSError:
                pass
else:
    print('V5 directory not found:', V5)

print('\nTRACE COMPLETE — READ ONLY')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DEEP V5 INFERENCE TRACE

[A] SOURCE HITS — CELLS 70+

--- CELL 70 | execution_count=None | lines=173 | hits=1 ---
L170: print("BLOCKED: collate_v4 must be restored from persisted V4 code.")
L171: print("V5 TRAINING: NOT STARTED")
L172: print("V4 CHECKPOINTS: UNTOUCHED")
L173: print("=" * 80)

--- CELL 71 | execution_count=None | lines=145 | hits=1 ---
L142: 
L143: print("V5 TRAINING: NOT STARTED")
L144: print("V4 CHECKPOINTS: UNTOUCHED")
L145: print("=" * 80)

--- CELL 72 | execution_count=None | lines=145 | hits=1 ---
L142: 
L143: print("V5 TRAINING: NOT STARTED")
L144: print("V4 CHECKPOINTS: UNTOUCHED")
L145: print("=" * 80)

--- CELL 73 | execution_count=None | lines=523 | hits=1 ---
L520: print("")
L521: print("V5 TRAINING: NOT STARTED")
L522: print("V4 CHECKPOINTS: UNTOUCHED")
L523: print("=" * 80)

--- CELL 74 | execution_count=None | lines=700 | hits=3

In [48]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import json
from pathlib import Path

nb_path = Path('/content/drive/MyDrive/Colab Notebooks/25sep.ipynb')
nb = json.loads(nb_path.read_text(encoding='utf-8'))
cells = nb['cells']

print('=' * 85)
print('V5 TRAINING TARGET / LOSS AUDIT — READ ONLY')
print('=' * 85)

keywords = [
    'labels', 'target_text', 'target_ids', 'decoder_input_ids',
    'label_ids', 'loss', 'cross_entropy', 'criterion',
    'tokenizer(', 'encode(', 'ground_truth', 'ket_luan',
    'concepts_v2_str', 'teacher_forcing', 'logits',
    'val_loss', 'validation', 'attention_mask', 'IGNORE_INDEX',
    'pad_token_id', 'eos_token_id', 'max_length',
]

for idx in [139, 140]:
    if idx >= len(cells):
        continue

    cell = cells[idx]
    lines = ''.join(cell.get('source', [])).splitlines()

    print(f'\n{"=" * 85}')
    print(f'CELL {idx} | {len(lines)} lines')
    print('=' * 85)

    hits = [
        i for i, line in enumerate(lines)
        if any(k.lower() in line.lower() for k in keywords)
    ]

    # Gộp các đoạn gần nhau để tránh in trùng lặp quá nhiều.
    ranges = []
    for i in hits:
        start, end = max(0, i - 3), min(len(lines), i + 5)
        if ranges and start <= ranges[-1][1] + 2:
            ranges[-1] = (ranges[-1][0], max(ranges[-1][1], end))
        else:
            ranges.append((start, end))

    printed = 0
    for start, end in ranges:
        print(f'\n--- Lines {start + 1}-{end} ---')
        for j in range(start, end):
            print(f'{j + 1:4d}: {lines[j][:220]}')
            printed += 1
            if printed >= 220:
                print('... output limit reached for this cell ...')
                break
        if printed >= 220:
            break

print('\nAUDIT COMPLETE — NO TRAINING OR FILE WRITES')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
V5 TRAINING TARGET / LOSS AUDIT — READ ONLY

CELL 139 | 2121 lines

--- Lines 392-399 ---
 392:                     str(found)
 393:                 )
 394: 
 395:     # V4 validation/test: first 8
 396:     # Training: random selection >8
 397:     case_to_images[
 398:         case_id
 399:     ] = resolved_paths

--- Lines 503-522 ---
 503: # 10. TARGET TEXT
 504: # ============================================================
 505: 
 506: assert "ket_luan" in v4_df.columns
 507: 
 508: report_map = dict(
 509:     zip(
 510:         v4_df["case_id"].astype(str),
 511:         v4_df["ket_luan"].fillna("").astype(str)
 512:     )
 513: )
 514: 
 515: assert len(report_map) == 7606
 516: 
 517: print(
 518:     "PASS: Report target = ket_luan."
 519: )
 520: 
 521: 
 522: # ============================================================

--- Lines 659-666 ---
 6

In [49]:

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import json
from pathlib import Path

# ============================================================
# READ-ONLY AUDIT: V5 FORWARD / VALIDATION / CHECKPOINT
# ============================================================
nb_path = Path('/content/drive/MyDrive/Colab Notebooks/25sep.ipynb')
assert nb_path.exists(), f'Không tìm thấy notebook: {nb_path}'

nb = json.loads(nb_path.read_text(encoding='utf-8'))
cells = nb['cells']

print('=' * 100)
print('V5 MODEL / VALIDATION / CHECKPOINT AUDIT — READ ONLY')
print('=' * 100)

# Các vùng code cần trích xuất
targets = {
    81: [
        'class V5Model',
        'def forward',
        'report_loss',
        'labels=',
        'logits',
        'generate(',
    ],
    139: [
        'class V5Model',
        'def forward',
        'report_loss',
        'val_loss',
        'best_val',
        'save',
        'torch.save',
        'load_state_dict',
        'Validation',
        'checkpoint',
    ],
    140: [
        'class V5Model',
        'def forward',
        'report_loss',
        'val_loss',
        'best_val',
        'save_v5_checkpoint',
        'load_v5_checkpoint',
        'torch.save',
        'load_state_dict',
        'Validation',
        'checkpoint',
    ],
}

MAX_LINES_PER_CELL = 280
CONTEXT_BEFORE = 5
CONTEXT_AFTER = 10

for cell_idx, keywords in targets.items():
    if cell_idx >= len(cells):
        print(f'\n[SKIP] Cell {cell_idx} không tồn tại.')
        continue

    cell = cells[cell_idx]
    source = ''.join(cell.get('source', [])).splitlines()

    print('\n' + '=' * 100)
    print(
        f'CELL {cell_idx} | '
        f'type={cell.get("cell_type")} | '
        f'lines={len(source)} | '
        f'execution_count={cell.get("execution_count")}'
    )
    print('=' * 100)

    # Tìm dòng chứa keyword, bỏ qua phân biệt chữ hoa/thường
    hits = [
        i for i, line in enumerate(source)
        if any(key.lower() in line.lower() for key in keywords)
    ]

    # Gom các vùng code gần nhau để hạn chế trùng lặp
    ranges = []
    for i in hits:
        start = max(0, i - CONTEXT_BEFORE)
        end = min(len(source), i + CONTEXT_AFTER + 1)

        if ranges and start <= ranges[-1][1] + 2:
            ranges[-1] = (
                ranges[-1][0],
                max(ranges[-1][1], end)
            )
        else:
            ranges.append((start, end))

    printed = 0
    for start, end in ranges:
        if printed >= MAX_LINES_PER_CELL:
            print('\n... Đã đạt giới hạn output của cell ...')
            break

        print(f'\n--- Source lines {start + 1}-{end} ---')
        for line_idx in range(start, end):
            print(f'{line_idx + 1:5d}: {source[line_idx]}')
            printed += 1
            if printed >= MAX_LINES_PER_CELL:
                print('... Đã đạt giới hạn output của cell ...')
                break

    if not hits:
        print('Không tìm thấy keyword phù hợp trong source cell này.')

    # Chỉ in metadata của output đã lưu, không thực thi lại cell
    outputs = cell.get('outputs', [])
    print('\n--- SAVED EXECUTION OUTPUT METADATA ---')
    print('Số output đã lưu:', len(outputs))
    for out_idx, out in enumerate(outputs):
        data = out.get('data', {})
        text_parts = data.get('text', [])
        if text_parts:
            text_value = ''.join(text_parts)
            print(f'Output {out_idx}:')
            print(text_value[:1200])
        elif out.get('ename'):
            print(
                f'Output {out_idx}: '
                f'{out.get("ename")}: {out.get("evalue", "")}'
            )

print('\n' + '=' * 100)
print('AUDIT COMPLETE — KHÔNG TRAINING / KHÔNG LOAD CHECKPOINT / KHÔNG GHI FILE')
print('=' * 100)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
V5 MODEL / VALIDATION / CHECKPOINT AUDIT — READ ONLY

CELL 81 | type=code | lines=373 | execution_count=None

--- Source lines 56-80 ---
   56: 
   57:         self.attribute_embeddings = nn.Parameter(
   58:             torch.randn(n_attributes, hidden_dim)
   59:         )
   60: 
   61:     def forward(
   62:         self,
   63:         concept_logits,
   64:         negative_logits,
   65:         attribute_logits,
   66:     ):
   67:         # Differentiable soft conditioning
   68:         concept_prob = torch.sigmoid(concept_logits)
   69:         negative_prob = torch.sigmoid(negative_logits)
   70:         attribute_prob = torch.sigmoid(attribute_logits)
   71: 
   72:         concept_tokens = torch.matmul(
   73:             concept_prob,
   74:             self.concept_embeddings
   75:         ).unsqueeze(1)
   76: 
   77:         negative_toke

In [50]:

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import json
from pathlib import Path
import pandas as pd

# ============================================================
# V5 EXISTING ARTIFACT AUDIT — READ ONLY
# ============================================================
ROOT = Path('/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5')
AUDIT = ROOT / 'diagnostics' / 'audit_20261015'
CKPT = ROOT / 'checkpoints'

print('=' * 90)
print('V5 ARTIFACT / PREDICTION COLLAPSE AUDIT — READ ONLY')
print('=' * 90)

# 1. Check file existence, size and modification time.
print('\n[1] FILE INVENTORY')
paths = [
    ROOT / 'test_predictions_712.csv',
    ROOT / 'diagnostics' / 'v5_final_training_summary.json',
    ROOT / 'prediction_frequency_audit.csv',
    ROOT / 'prediction_quality_audit.csv',
    ROOT / 'distribution_shift_audit.csv',
    ROOT / 'text_overlap_audit.csv',
    ROOT / 'anatomy_error_stratification.csv',
    AUDIT / 'prediction_frequency_audit.csv',
    AUDIT / 'prediction_quality_audit.csv',
    AUDIT / 'distribution_shift_audit.csv',
    AUDIT / 'text_overlap_audit.csv',
    AUDIT / 'anatomy_error_stratification.csv',
    CKPT / 'best.pt',
    CKPT / 'last.pt',
]
for p in paths:
    if p.exists():
        st = p.stat()
        print(f'FOUND | {p.name} | {st.st_size:,} bytes | mtime={pd.Timestamp(st.st_mtime, unit="s")}')
    else:
        print(f'NOT FOUND | {p}')

# 2. Read training summary JSON only.
print('\n[2] TRAINING SUMMARY JSON')
summary_path = ROOT / 'diagnostics' / 'v5_final_training_summary.json'
if summary_path.exists():
    try:
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        print(json.dumps(summary, ensure_ascii=False, indent=2, default=str)[:18000])
    except Exception as e:
        print('Không đọc được JSON:', repr(e))
else:
    print('Không tìm thấy training summary JSON.')

# 3. Inspect existing CSV diagnostics; print schema and compact summaries.
print('\n[3] EXISTING DIAGNOSTIC CSV FILES')
csv_names = [
    'prediction_frequency_audit.csv',
    'prediction_quality_audit.csv',
    'distribution_shift_audit.csv',
    'text_overlap_audit.csv',
    'anatomy_error_stratification.csv',
    'low_overlap_image_metadata.csv',
    'tympanic_membrane_cases.csv',
]
for name in csv_names:
    candidates = [ROOT / name, AUDIT / name]
    p = next((x for x in candidates if x.exists()), None)
    if p is None:
        print(f'\n--- {name}: NOT FOUND ---')
        continue

    print(f'\n--- {name} | {p} ---')
    try:
        df = pd.read_csv(p, low_memory=False)
        print('Shape:', df.shape)
        print('Columns:', list(df.columns))
        print(df.head(8).to_string(index=False, max_colwidth=100))
        if len(df) > 8:
            print(f'... còn {len(df) - 8} dòng')
    except Exception as e:
        print('Lỗi đọc CSV:', repr(e))

# 4. Independently summarize prediction CSV, without changing it.
print('\n[4] TEST PREDICTION DISTRIBUTION')
pred_path = ROOT / 'test_predictions_712.csv'
if pred_path.exists():
    df = pd.read_csv(pred_path, low_memory=False)
    print('Shape:', df.shape)
    print('Columns:', list(df.columns))

    required = {'case_id', 'prediction', 'ground_truth'}
    if required.issubset(df.columns):
        print('Rows:', len(df))
        print('Unique case_id:', df['case_id'].nunique(dropna=False))
        print('Duplicate case_id rows:', int(df['case_id'].duplicated().sum()))
        print('Missing predictions:', int(df['prediction'].isna().sum()))
        print('Missing ground truth:', int(df['ground_truth'].isna().sum()))

        pred = df['prediction'].fillna('').astype(str).str.strip()
        truth = df['ground_truth'].fillna('').astype(str).str.strip()

        print('Unique prediction strings:', pred.nunique())
        print('Unique ground-truth strings:', truth.nunique())
        print('Exact string match rate:', round(float((pred == truth).mean()), 4))
        print('\nTop 20 predictions:')
        print(pred.value_counts().head(20).to_string())
        print('\nTop 20 ground-truth labels:')
        print(truth.value_counts().head(20).to_string())

        print('\nPrediction frequency quantiles:')
        print(pred.value_counts().describe(percentiles=[.5, .9, .95, .99]).to_string())
    else:
        print('CSV thiếu cột bắt buộc:', sorted(required - set(df.columns)))
else:
    print('Không tìm thấy test_predictions_712.csv.')

# 5. Compare prediction case IDs against the V4 test split, if available.
print('\n[5] TEST SPLIT ID CONSISTENCY')
manifest = Path('/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v4/v4_training_manifest.csv')
if manifest.exists() and pred_path.exists():
    try:
        m = pd.read_csv(manifest, usecols=['case_id', 'split'])
        p = pd.read_csv(pred_path, usecols=['case_id'])

        expected = set(
            m.loc[m['split'].astype(str).str.lower().eq('test'), 'case_id']
            .astype(str).str.strip()
        )
        actual = set(p['case_id'].astype(str).str.strip())

        print('Manifest test IDs:', len(expected))
        print('Prediction unique IDs:', len(actual))
        print('Intersection:', len(expected & actual))
        print('Missing prediction IDs:', len(expected - actual))
        print('Unexpected prediction IDs:', len(actual - expected))
        if expected - actual:
            print('Sample missing IDs:', list(sorted(expected - actual))[:10])
        if actual - expected:
            print('Sample unexpected IDs:', list(sorted(actual - expected))[:10])
    except Exception as e:
        print('Lỗi đối chiếu ID:', repr(e))
else:
    print('Không đủ file để đối chiếu manifest và prediction CSV.')

print('\n' + '=' * 90)
print('AUDIT COMPLETE — KHÔNG LOAD CHECKPOINT / KHÔNG TRAINING / KHÔNG GHI FILE')
print('=' * 90)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
V5 ARTIFACT / PREDICTION COLLAPSE AUDIT — READ ONLY

[1] FILE INVENTORY
FOUND | test_predictions_712.csv | 62,941 bytes | mtime=2026-09-30 16:34:27
FOUND | v5_final_training_summary.json | 649 bytes | mtime=2026-09-29 04:23:10
FOUND | prediction_frequency_audit.csv | 710 bytes | mtime=2026-10-14 12:14:15
FOUND | prediction_quality_audit.csv | 79,960 bytes | mtime=2026-10-14 12:19:52
FOUND | distribution_shift_audit.csv | 1,669 bytes | mtime=2026-10-14 12:26:22
FOUND | text_overlap_audit.csv | 140,287 bytes | mtime=2026-10-14 12:28:11
FOUND | anatomy_error_stratification.csv | 331,383 bytes | mtime=2026-10-14 12:34:59
NOT FOUND | /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/diagnostics/audit_20261015/prediction_frequency_audit.csv
NOT FOUND | /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/diagnostics/audit_20261015/prediction_quality_audi

In [1]:

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path('/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5')

PRED_PATH = ROOT / 'test_predictions_712.csv'
QUALITY_PATH = ROOT / 'prediction_quality_audit.csv'
ANATOMY_PATH = ROOT / 'anatomy_error_stratification.csv'
SHIFT_PATH = ROOT / 'distribution_shift_audit.csv'

print('=' * 90)
print('V5 ERROR STRATIFICATION — READ ONLY')
print('=' * 90)

for p in [PRED_PATH, QUALITY_PATH, ANATOMY_PATH, SHIFT_PATH]:
    assert p.exists(), f'Thiếu file cần đọc: {p}'

pred = pd.read_csv(PRED_PATH, low_memory=False)
quality = pd.read_csv(QUALITY_PATH, low_memory=False)
anatomy = pd.read_csv(ANATOMY_PATH, low_memory=False)
shift = pd.read_csv(SHIFT_PATH, low_memory=False)

# Merge bằng case_id để tránh phụ thuộc thứ tự dòng.
df = pred.merge(
    quality[['case_id', 'token_f1', 'exact_match']],
    on='case_id',
    how='left',
    validate='one_to_one'
)

region_cols = [
    'tai', 'hoc_mui', 'hong_mui',
    'hong_thanh_quan', 'hong_mieng', 'ly_do_noi_soi'
]
available_regions = [c for c in region_cols if c in anatomy.columns]

anatomy_cols = [
    'case_id', 'gt_region_heuristic',
    'tai__present', 'hoc_mui__present',
    'hong_mui__present', 'hong_thanh_quan__present',
    'hong_mieng__present', 'ly_do_noi_soi__present'
]
anatomy_cols = [c for c in anatomy_cols if c in anatomy.columns]

df = df.merge(
    anatomy[anatomy_cols],
    on='case_id',
    how='left',
    validate='one_to_one'
)

print('\n[1] OVERALL METRICS')
print('Cases:', len(df))
print('Exact match:', round(df['exact_match'].fillna(False).astype(bool).mean(), 4))
print('Mean token F1:', round(df['token_f1'].mean(), 4))
print('Median token F1:', round(df['token_f1'].median(), 4))
print('Cases token F1 = 0:', int((df['token_f1'] == 0).sum()))
print('Cases token F1 < 0.25:', int((df['token_f1'] < 0.25).sum()))

print('\n[2] PERFORMANCE BY GROUND-TRUTH ANATOMICAL REGION')
if 'gt_region_heuristic' in df.columns:
    region_summary = (
        df.groupby('gt_region_heuristic', dropna=False)
        .agg(
            cases=('case_id', 'count'),
            exact_match_rate=('exact_match', lambda x: x.fillna(False).astype(bool).mean()),
            mean_token_f1=('token_f1', 'mean'),
            zero_f1_cases=('token_f1', lambda x: int((x == 0).sum()))
        )
        .sort_values('cases', ascending=False)
    )
    print(region_summary.round(3).to_string())
else:
    print('Không có cột gt_region_heuristic.')

print('\n[3] MOST COMMON PREDICTIONS AND THEIR ERROR PROFILE')
pred_summary = (
    df.groupby('prediction', dropna=False)
    .agg(
        cases=('case_id', 'count'),
        exact_match_rate=('exact_match', lambda x: x.fillna(False).astype(bool).mean()),
        mean_token_f1=('token_f1', 'mean'),
        distinct_ground_truth=('ground_truth', 'nunique')
    )
    .sort_values('cases', ascending=False)
)
print(pred_summary.head(16).round(3).to_string())

print('\n[4] TOP GROUND-TRUTH LABELS AND MOST COMMON PREDICTION')
top_truth = df['ground_truth'].value_counts().head(15).index
for gt in top_truth:
    sub = df[df['ground_truth'] == gt]
    common_pred = sub['prediction'].value_counts().head(3).to_dict()
    print(
        f'\nGT ({len(sub)} cases): {gt}\n'
        f'  Top predictions: {common_pred}\n'
        f'  Exact match: {sub["exact_match"].fillna(False).astype(bool).mean():.3f}'
        f' | Mean token F1: {sub["token_f1"].mean():.3f}'
    )

print('\n[5] PREDICTION FREQUENCY VS TRAIN/TEST LABEL COUNTS')
show_cols = [
    'prediction', 'predicted_cases', 'test_GT_exact_label_cases',
    'train_exact_label_cases', 'prediction_vs_GT_count_ratio',
    'seen_exactly_in_train'
]
show_cols = [c for c in show_cols if c in shift.columns]
print(
    shift.sort_values('predicted_cases', ascending=False)[show_cols]
    .head(16).to_string(index=False)
)

print('\n[6] DOES PREDICTION VARY WITH ANATOMICAL EVIDENCE?')
for col in available_regions:
    values = df[col].fillna('').astype(str).str.strip()
    valid = values.ne('') & values.str.lower().ne('nan')
    if valid.sum() == 0:
        continue
    sub = df.loc[valid].copy()
    sub['_region_text'] = values[valid]
    print(
        f'{col}: {len(sub)} cases with text; '
        f'{sub["_region_text"].nunique()} unique region descriptions; '
        f'{sub["prediction"].nunique()} unique predictions'
    )
    print('  Frequent predictions:', sub['prediction'].value_counts().head(3).to_dict())

print('\n' + '=' * 90)
print('COMPLETE — CHỈ ĐỌC CSV CÓ SẴN, KHÔNG LOAD MODEL / KHÔNG GHI FILE')
print('=' * 90)


Mounted at /content/drive
V5 ERROR STRATIFICATION — READ ONLY

[1] OVERALL METRICS
Cases: 712
Exact match: 0.1896
Mean token F1: 0.6097
Median token F1: 0.6667
Cases token F1 = 0: 19
Cases token F1 < 0.25: 82

[2] PERFORMANCE BY GROUND-TRUTH ANATOMICAL REGION
                                             cases  exact_match_rate  mean_token_f1  zero_f1_cases
gt_region_heuristic                                                                               
Mũi/xoang/VA                                   345             0.229          0.701              1
Tai + Mũi/xoang/VA                             152             0.000          0.410              0
Tai                                            110             0.045          0.506             11
Họng/thanh quản/dị vật                          79             0.646          0.805              7
Mũi/xoang/VA + Họng/thanh quản/dị vật           14             0.000          0.500              0
Khác/chưa rõ                                   

KeyError: 'tai'

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path('/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5')
PRED = ROOT / 'test_predictions_712.csv'
QUALITY = ROOT / 'prediction_quality_audit.csv'
ANATOMY = ROOT / 'anatomy_error_stratification.csv'
SHIFT = ROOT / 'distribution_shift_audit.csv'

for p in [PRED, QUALITY, ANATOMY, SHIFT]:
    assert p.exists(), f'Thiếu file: {p}'

pred = pd.read_csv(PRED, low_memory=False)
quality = pd.read_csv(QUALITY, low_memory=False)
anatomy = pd.read_csv(ANATOMY, low_memory=False)
shift = pd.read_csv(SHIFT, low_memory=False)

# Chỉ lấy các cột cần thiết; tránh merge trùng tên cột.
qcols = [c for c in ['case_id', 'token_f1', 'exact_match'] if c in quality.columns]
acols = [
    c for c in [
        'case_id', 'gt_region_heuristic',
        'tai', 'hoc_mui', 'hong_mui',
        'hong_thanh_quan', 'hong_mieng', 'ly_do_noi_soi',
        'tai__present', 'hoc_mui__present', 'hong_mui__present',
        'hong_thanh_quan__present', 'hong_mieng__present',
        'ly_do_noi_soi__present'
    ] if c in anatomy.columns
]

df = pred.merge(
    quality[qcols], on='case_id', how='left', validate='one_to_one'
).merge(
    anatomy[acols], on='case_id', how='left', validate='one_to_one'
)

print('=' * 85)
print('CELL 1 — FIXED ERROR STRATIFICATION')
print('=' * 85)
print('Rows:', len(df), '| Unique case IDs:', df['case_id'].nunique())
print('Columns successfully joined:', acols)
print('Exact match rate:', round(df['exact_match'].fillna(False).astype(bool).mean(), 4))
print('Mean token F1:', round(df['token_f1'].mean(), 4))
print('Zero token-F1 cases:', int((df['token_f1'] == 0).sum()))

if 'gt_region_heuristic' in df.columns:
    print('\n[Performance by ground-truth region]')
    print(
        df.groupby('gt_region_heuristic', dropna=False)
        .agg(
            cases=('case_id', 'size'),
            exact_match_rate=('exact_match',
                              lambda s: s.fillna(False).astype(bool).mean()),
            mean_token_f1=('token_f1', 'mean'),
            zero_f1=('token_f1', lambda s: int((s == 0).sum())),
            unique_predictions=('prediction', 'nunique')
        )
        .sort_values('cases', ascending=False)
        .round(3).to_string()
    )

print('\n[Prediction counts and performance]')
print(
    df.groupby('prediction', dropna=False)
    .agg(
        cases=('case_id', 'size'),
        exact_match_rate=('exact_match',
                          lambda s: s.fillna(False).astype(bool).mean()),
        mean_token_f1=('token_f1', 'mean'),
        unique_ground_truth=('ground_truth', 'nunique')
    )
    .sort_values('cases', ascending=False)
    .head(16).round(3).to_string()
)

print('\n[Anatomical text coverage and prediction diversity]')
for col in ['tai', 'hoc_mui', 'hong_mui',
            'hong_thanh_quan', 'hong_mieng', 'ly_do_noi_soi']:
    if col not in df.columns:
        print(f'{col}: column absent')
        continue
    s = df[col].fillna('').astype(str).str.strip()
    valid = s.ne('') & s.str.lower().ne('nan')
    sub = df.loc[valid]
    print(
        f'{col}: text_cases={int(valid.sum())}, '
        f'unique_texts={s[valid].nunique()}, '
        f'unique_predictions={sub["prediction"].nunique()}'
    )
    if len(sub):
        print('  Top predictions:', sub['prediction'].value_counts().head(4).to_dict())

print('\nREAD ONLY — no model loaded; no files written.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CELL 1 — FIXED ERROR STRATIFICATION
Rows: 712 | Unique case IDs: 712
Columns successfully joined: ['case_id', 'gt_region_heuristic', 'tai', 'hoc_mui', 'hong_mui', 'hong_thanh_quan', 'hong_mieng', 'ly_do_noi_soi', 'tai__present', 'hoc_mui__present', 'hong_mui__present', 'hong_thanh_quan__present', 'hong_mieng__present', 'ly_do_noi_soi__present']
Exact match rate: 0.1896
Mean token F1: 0.6097
Zero token-F1 cases: 19

[Performance by ground-truth region]
                                             cases  exact_match_rate  mean_token_f1  zero_f1  unique_predictions
gt_region_heuristic                                                                                             
Mũi/xoang/VA                                   345             0.229          0.701        1                  10
Tai + Mũi/xoang/VA                             152             0.000        

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path
import pandas as pd

ROOT = Path('/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5')
PRED = ROOT / 'test_predictions_712.csv'
QUALITY = ROOT / 'prediction_quality_audit.csv'
ANATOMY = ROOT / 'anatomy_error_stratification.csv'

for p in [PRED, QUALITY, ANATOMY]:
    assert p.exists(), f'Thiếu file: {p}'

pred = pd.read_csv(PRED, low_memory=False)
quality = pd.read_csv(QUALITY, low_memory=False)
anatomy = pd.read_csv(ANATOMY, low_memory=False)

keep_q = [c for c in ['case_id', 'token_f1', 'exact_match'] if c in quality.columns]
keep_a = [
    c for c in [
        'case_id', 'gt_region_heuristic',
        'tai', 'hoc_mui', 'hong_mui',
        'hong_thanh_quan', 'hong_mieng', 'ly_do_noi_soi'
    ] if c in anatomy.columns
]

df = pred.merge(
    quality[keep_q], on='case_id', how='left', validate='one_to_one'
).merge(
    anatomy[keep_a], on='case_id', how='left', validate='one_to_one'
)

region_cols = [
    c for c in [
        'tai', 'hoc_mui', 'hong_mui',
        'hong_thanh_quan', 'hong_mieng', 'ly_do_noi_soi'
    ] if c in df.columns
]

print('=' * 85)
print('CELL 2 — LOW-OVERLAP CASE REVIEW')
print('=' * 85)

# Ưu tiên ca token F1 thấp nhất; dùng case_id để có thể truy lại dữ liệu.
df['token_f1'] = pd.to_numeric(df['token_f1'], errors='coerce')
low = df.sort_values(['token_f1', 'case_id'], ascending=[True, True]).head(30)

print('\n[30 ca có token F1 thấp nhất]')
show = ['case_id', 'token_f1', 'prediction', 'ground_truth', 'gt_region_heuristic']
show += region_cols
show = [c for c in show if c in low.columns]
print(low[show].to_string(index=False, max_colwidth=100))

print('\n[Nhóm dự đoán thường gặp trong các ca token F1 < 0.25]')
bad = df[df['token_f1'] < 0.25].copy()
print('Số ca:', len(bad))
if len(bad):
    print(
        bad.groupby('prediction')
        .agg(
            cases=('case_id', 'size'),
            mean_token_f1=('token_f1', 'mean'),
            unique_ground_truth=('ground_truth', 'nunique')
        )
        .sort_values('cases', ascending=False)
        .head(12).round(3).to_string()
    )

    print('\n[Phân bố vùng giải phẫu ở nhóm F1 thấp]')
    if 'gt_region_heuristic' in bad.columns:
        print(bad['gt_region_heuristic'].fillna('(missing)').value_counts().to_string())

print('\n[Đối chiếu mô tả giải phẫu với dự đoán]')
for col in region_cols:
    s = bad[col].fillna('').astype(str).str.strip()
    valid = s.ne('') & s.str.lower().ne('nan')
    subset = bad.loc[valid]
    print(f'\n{col}: {len(subset)} ca F1 thấp có mô tả')
    if len(subset):
        print('Dự đoán phổ biến:', subset['prediction'].value_counts().head(5).to_dict())
        print('Ví dụ mô tả:')
        for value in subset[col].dropna().astype(str).head(3):
            print(' -', value[:250])

print('\nREAD ONLY — no model loaded; no files written.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CELL 2 — LOW-OVERLAP CASE REVIEW

[30 ca có token F1 thấp nhất]
            case_id  token_f1                                   prediction                                                                 ground_truth    gt_region_heuristic                                                                                                 tai                                                                                              hoc_mui                      hong_mui                                                                                      hong_thanh_quan                                                                            hong_mieng        ly_do_noi_soi
10783.10783.0.10798  0.000000 VIÊM HỌNG MẠN - THEO DÕI TRÀO NGƯỢC DỊCH VỊ.                 HIỆN KHÔNG THẤY ĐIỂM CHẢY MÁU TRÊN NỘI SOI A-MI-ĐAN QUÁ PHÁT Họng/thanh quản/dị vật                   

In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path
import json

PROJECT = Path('/content/drive/MyDrive/NoiSoi_Matching')
ROOT = PROJECT / 'baseline_model_v5'
NB_PATH = Path('/content/drive/MyDrive/Colab Notebooks/25sep.ipynb')
SUMMARY = ROOT / 'diagnostics/v5_final_training_summary.json'
PRED = ROOT / 'test_predictions_712.csv'

print('=' * 85)
print('CELL 3 — NOTEBOOK / ARTIFACT PROVENANCE AUDIT')
print('=' * 85)

# 1. Metadata only: no checkpoint deserialization.
for p in [PRED, SUMMARY, ROOT / 'checkpoints/best.pt', ROOT / 'checkpoints/last.pt']:
    if p.exists():
        st = p.stat()
        print(f'\nFILE: {p}')
        print(f'  size_bytes={st.st_size}')
        print(f'  modified={__import__("datetime").datetime.fromtimestamp(st.st_mtime).isoformat()}')
    else:
        print(f'\nMISSING: {p}')

if SUMMARY.exists():
    try:
        meta = json.loads(SUMMARY.read_text(encoding='utf-8'))
        print('\n[Training summary JSON]')
        for key in [
            'model', 'status', 'num_epochs', 'global_step', 'epoch',
            'best_val', 'last_checkpoint', 'best_checkpoint'
        ]:
            if key in meta:
                print(f'{key}: {meta[key]}')
        print('train_loss:', meta.get('train_loss'))
        print('val_loss:', meta.get('val_loss'))
    except Exception as e:
        print('Could not parse summary:', repr(e))

# 2. Search saved notebook source AND saved cell outputs.
if not NB_PATH.exists():
    print('\nNOTEBOOK NOT FOUND:', NB_PATH)
else:
    nb = json.loads(NB_PATH.read_text(encoding='utf-8'))
    cells = nb.get('cells', [])
    terms = [
        'test_predictions_712.csv',
        'test_predictions',
        'prediction_path',
        'model.generate',
        'exact_generate',
        'generate(',
        'best.pt',
        'last.pt',
        'torch.save',
        'to_csv(',
        'load_state_dict',
        'TRAINING_COMPLETE'
    ]

    print(f'\nNotebook cells: {len(cells)}')
    matches = []
    for i, cell in enumerate(cells):
        source = ''.join(cell.get('source', []))
        output_texts = []
        for out in cell.get('outputs', []):
            if 'text' in out:
                output_texts.append(''.join(out['text']))
            data = out.get('data', {})
            if 'text/plain' in data:
                value = data['text/plain']
                output_texts.append(''.join(value) if isinstance(value, list) else str(value))
            if 'ename' in out:
                output_texts.append(f"{out.get('ename')}: {out.get('evalue', '')}")

        output = '\n'.join(output_texts)
        found_source = [t for t in terms if t.lower() in source.lower()]
        found_output = [t for t in terms if t.lower() in output.lower()]

        if found_source or found_output:
            matches.append((i, cell.get('execution_count'), found_source, found_output, source, output))

    for i, execution_count, src_terms, out_terms, source, output in matches:
        print(f'\n--- CELL INDEX {i} | execution_count={execution_count} ---')
        print('Source matches:', src_terms)
        print('Output matches:', out_terms)

        if src_terms:
            lines = source.splitlines()
            shown = 0
            for j, line in enumerate(lines):
                if any(t.lower() in line.lower() for t in src_terms):
                    lo, hi = max(0, j-2), min(len(lines), j+3)
                    print(f'  Lines {lo+1}-{hi}:')
                    for k in range(lo, hi):
                        print(f'    {k+1}: {lines[k][:240]}')
                    shown += 1
                    if shown >= 5:
                        print('  ... source excerpts truncated')
                        break

        if output:
            relevant = [
                line for line in output.splitlines()
                if any(t.lower() in line.lower() for t in terms)
                or 'Error' in line or 'Traceback' in line
            ]
            if relevant:
                print('Relevant saved output:')
                for line in relevant[:15]:
                    print(' ', line[:240])

    exact_source_hits = [
        i for i, cell in enumerate(cells)
        if 'test_predictions_712.csv' in ''.join(cell.get('source', []))
    ]
    print('\n[Direct source reference to prediction CSV]')
    print('Cell indices:', exact_source_hits if exact_source_hits else 'NONE FOUND')

print('\nREAD ONLY — checkpoint weights were NOT loaded; no files written.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CELL 3 — NOTEBOOK / ARTIFACT PROVENANCE AUDIT

FILE: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/test_predictions_712.csv
  size_bytes=62941
  modified=2026-09-30T16:34:27

FILE: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/diagnostics/v5_final_training_summary.json
  size_bytes=649
  modified=2026-09-29T04:23:10

FILE: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/checkpoints/best.pt
  size_bytes=4640037647
  modified=2026-09-29T04:20:44

FILE: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/checkpoints/last.pt
  size_bytes=4640037647
  modified=2026-09-29T04:25:51

[Training summary JSON]
model: V5
status: TRAINING_COMPLETE
num_epochs: 5
global_step: 1920
epoch: 5
best_val: 0.424518172663671
last_checkpoint: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/checkpoints/last.pt
best_checkpoint: /content/dri

In [6]:
# CELL 2 — Review low-overlap cases (read-only)
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

ROOT = '/content/drive/MyDrive/NoiSoi_Matching'
AUDIT = os.path.join(ROOT, 'baseline_model_v5', 'diagnostics', 'audit_20261015')

candidates = [
    os.path.join(AUDIT, 'low_overlap_image_metadata.csv'),
    os.path.join(AUDIT, 'text_overlap_audit.csv'),
    os.path.join(AUDIT, 'anatomy_error_stratification.csv'),
]

for path in candidates:
    print(f'{os.path.basename(path)}: {"FOUND" if os.path.exists(path) else "NOT FOUND"}')

low_path = candidates[0]
overlap_path = candidates[1]
anatomy_path = candidates[2]

if os.path.exists(overlap_path):
    df = pd.read_csv(overlap_path)
elif os.path.exists(anatomy_path):
    df = pd.read_csv(anatomy_path)
elif os.path.exists(low_path):
    df = pd.read_csv(low_path)
else:
    raise FileNotFoundError('Không tìm thấy các file audit đã dự kiến.')

print('\nShape:', df.shape)
print('Columns:', df.columns.tolist())

# Sort the weakest-overlap cases first, when the score exists.
score_col = next(
    (c for c in ['token_f1', 'f1', 'token_F1'] if c in df.columns),
    None
)
if score_col:
    df[score_col] = pd.to_numeric(df[score_col], errors='coerce')
    df = df.sort_values(score_col, ascending=True, na_position='last')

# Show only useful clinical/text fields; keep available anatomy fields.
preferred = [
    'case_id', 'prediction', 'ground_truth',
    'precision_tokens', 'recall_tokens', 'token_f1', 'exact_match',
    'gt_region_heuristic',
    'tai', 'hoc_mui', 'hong_mui',
    'hong_thanh_quan', 'hong_mieng', 'ly_do_noi_soi'
]
show_cols = [c for c in preferred if c in df.columns]

print('\n=== Lowest-overlap cases (up to 25) ===')
if show_cols:
    display(df[show_cols].head(25).fillna(''))
else:
    display(df.head(25).fillna(''))

print('\n=== Summary ===')
if score_col:
    print(df[score_col].describe().to_string())
if 'exact_match' in df.columns:
    print('\nExact-match distribution:')
    print(df['exact_match'].value_counts(dropna=False).to_string())

print('\nRead-only audit completed; no model training or checkpoint loading.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
low_overlap_image_metadata.csv: NOT FOUND
text_overlap_audit.csv: NOT FOUND
anatomy_error_stratification.csv: NOT FOUND


FileNotFoundError: Không tìm thấy các file audit đã dự kiến.

In [7]:

# CELL 2A — Locate audit artifacts (read-only; no training)
from google.colab import drive
drive.mount('/content/drive')

import os

MYDRIVE = '/content/drive/MyDrive'
PROJECT = os.path.join(MYDRIVE, 'NoiSoi_Matching')

target_names = {
    'low_overlap_image_metadata.csv',
    'text_overlap_audit.csv',
    'anatomy_error_stratification.csv',
    'prediction_quality_audit.csv',
    'test_predictions_712.csv',
    'v5_final_training_summary.json',
}
skip_dirs = {
    '.ipynb_checkpoints', '__pycache__', '.Trash',
    'node_modules'
}

print('MyDrive exists:', os.path.isdir(MYDRIVE))
print('Project path exists:', os.path.isdir(PROJECT))

if os.path.isdir(PROJECT):
    search_roots = [PROJECT]
else:
    search_roots = [MYDRIVE]
    print('WARNING: Project folder not found at expected path; searching MyDrive.')

found = []
audit_dirs = []

for search_root in search_roots:
    for current, dirs, files in os.walk(search_root):
        dirs[:] = [d for d in dirs if d not in skip_dirs]

        if os.path.basename(current).startswith('audit_'):
            audit_dirs.append(current)

        for filename in files:
            if filename in target_names:
                path = os.path.join(current, filename)
                found.append(path)
                try:
                    stat = os.stat(path)
                    print(
                        f'FOUND: {path}\n'
                        f'  Size: {stat.st_size:,} bytes | '
                        f'Modified: {stat.st_mtime}'
                    )
                except OSError:
                    print('FOUND but metadata unavailable:', path)

print('\n=== Audit directories ===')
for path in sorted(set(audit_dirs)):
    print(path)

print('\n=== Result ===')
if found:
    print(f'Found {len(found)} target file(s).')
else:
    print('No target files found under the searched root.')
    print('No files were created, modified, or deleted.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MyDrive exists: True
Project path exists: True
FOUND: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/test_predictions_712.csv
  Size: 62,941 bytes | Modified: 1790786067.0
FOUND: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/prediction_quality_audit.csv
  Size: 79,960 bytes | Modified: 1791980392.0
FOUND: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/text_overlap_audit.csv
  Size: 140,287 bytes | Modified: 1791980891.0
FOUND: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/low_overlap_image_metadata.csv
  Size: 4,948 bytes | Modified: 1791980991.0
FOUND: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/anatomy_error_stratification.csv
  Size: 331,383 bytes | Modified: 1791981299.0
FOUND: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/diagnostics/v5_final_training_summary.json
  Size: 649 bytes | Modif

In [9]:

# CELL 2C — Fixed low-overlap review (read-only)
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

ROOT = '/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5'

overlap_path = os.path.join(ROOT, 'text_overlap_audit.csv')
anatomy_path = os.path.join(ROOT, 'anatomy_error_stratification.csv')
low_path = os.path.join(ROOT, 'low_overlap_image_metadata.csv')

# Reload source files from disk; do not depend on partially modified variables.
df = pd.read_csv(overlap_path)
print('Overlap:', df.shape, '| columns:', df.columns.tolist())

if os.path.isfile(anatomy_path):
    anatomy = pd.read_csv(anatomy_path)
    print('\nAnatomy:', anatomy.shape, '| columns:', anatomy.columns.tolist())

    # Only join if both tables have the key.
    if 'case_id' in df.columns and 'case_id' in anatomy.columns:
        extra_cols = [
            c for c in [
                'gt_region_heuristic', 'tai', 'hoc_mui', 'hong_mui',
                'hong_thanh_quan', 'hong_mieng', 'ly_do_noi_soi'
            ]
            if c in anatomy.columns and c not in df.columns
        ]

        if extra_cols:
            anatomy_subset = anatomy[['case_id'] + extra_cols].copy()
            anatomy_subset = anatomy_subset.drop_duplicates(subset=['case_id'])
            df = df.merge(
                anatomy_subset,
                on='case_id',
                how='left',
                validate='one_to_one'
            )
            print('\nJoined anatomy fields:', extra_cols)
        else:
            print('\nNo additional anatomy columns available to join.')
    else:
        print('\nWARNING: case_id is missing from one table; skipped join.')

if 'token_f1' in df.columns:
    df['token_f1'] = pd.to_numeric(df['token_f1'], errors='coerce')
    df = df.sort_values('token_f1', ascending=True, na_position='last')

cols = [
    'case_id', 'prediction', 'ground_truth',
    'precision_tokens', 'recall_tokens', 'token_f1', 'exact_match',
    'gt_region_heuristic', 'tai', 'hoc_mui', 'hong_mui',
    'hong_thanh_quan', 'hong_mieng', 'ly_do_noi_soi'
]
cols = [c for c in cols if c in df.columns]

print('\n=== 25 LOWEST-OVERLAP CASES ===')
display(df[cols].head(25).fillna(''))

print('\n=== SUMMARY ===')
if 'token_f1' in df.columns:
    print(df['token_f1'].describe().to_string())
    print('Zero-F1 cases:', int(df['token_f1'].eq(0).sum()))

if 'exact_match' in df.columns:
    print('\nExact-match distribution:')
    print(df['exact_match'].value_counts(dropna=False).to_string())

if os.path.isfile(low_path):
    low = pd.read_csv(low_path)
    print('\n=== LOW-OVERLAP METADATA ===')
    print('Shape:', low.shape)
    print('Columns:', low.columns.tolist())
    display(low.head(10).fillna(''))

print('\nRead-only audit completed; no checkpoint loaded or files changed.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Overlap: (712, 11) | columns: ['case_id', 'prediction', 'ground_truth', 'precision_tokens', 'recall_tokens', 'token_f1', 'pred_norm', 'gt_norm', 'exact_match', 'pred_token_count', 'gt_token_count']

Anatomy: (712, 24) | columns: ['case_id', 'prediction', 'ground_truth', 'precision_tokens', 'recall_tokens', 'token_f1', 'pred_norm', 'gt_norm', 'exact_match', 'pred_token_count', 'gt_token_count', 'tai', 'hoc_mui', 'hong_mui', 'hong_thanh_quan', 'hong_mieng', 'ly_do_noi_soi', 'tai__present', 'hoc_mui__present', 'hong_mui__present', 'hong_thanh_quan__present', 'hong_mieng__present', 'ly_do_noi_soi__present', 'gt_region_heuristic']

Joined anatomy fields: ['gt_region_heuristic', 'tai', 'hoc_mui', 'hong_mui', 'hong_thanh_quan', 'hong_mieng', 'ly_do_noi_soi']

=== 25 LOWEST-OVERLAP CASES ===


,case_id,prediction,ground_truth,precision_tokens,recall_tokens,token_f1,exact_match,gt_region_heuristic,tai,hoc_mui,hong_mui,hong_thanh_quan,hong_mieng,ly_do_noi_soi
665,17557.17557.0.17582,VIÊM HỌNG MẠN - THEO DÕI TRÀO NGƯỢC DỊCH VỊ.,HÓC DỊ VẬT N1.,0.000000,0.000000,0.000000,False,Họng/thanh quản/dị vật,,,,"Thanh thiệt: hồng, láng. Dây thanh: trơn, láng...",Hai a-mi-đan: bên P có dị vật. Thành sau họng:...,Nuốt vướng sau ăn N1
672,17686.17686.0.17711,VIÊM HỌNG MẠN - THEO DÕI TRÀO NGƯỢC DỊCH VỊ.,HÓC XƯƠNG CÁ N1 (ĐÃ LẤY).,0.000000,0.000000,0.000000,False,Họng/thanh quản/dị vật,,,,"Thanh thiệt: hồng, láng. Vùng đáy lưỡi: có xươ...",Hai a-mi-đan: không to. Thành sau họng: có hạt...,Nuốt vướng sau ăn N1
412,14927.14927.0.14946,VIÊM MŨI MẠN,"HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ TAI NGOÀI, TA...",0.000000,0.000000,0.000000,False,Tai,,"Cuốn mũi: hông, gồ ghề. Khe mũi: thoáng, sạch ...",thoáng sạch,,,nặng hàm T
408,14900.14900.0.14919,VIÊM MŨI XOANG,"HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ TAI NGOÀI, TA...",0.000000,0.000000,0.000000,False,Tai,P: màng nhĩ bóng sáng. T: màng nhĩ bóng sáng.,"Cuốn mũi: hồng, láng. Khe mũi: thoáng, sạch Sà...",lỗ vòi nhĩ 2 bên thoáng sạch,,,ĐAU TAI P
112,11630.11630.0.11647,VIÊM HỌNG MẠN - THEO DÕI TRÀO NGƯỢC DỊCH VỊ.,LIỆT DÂY THANH P,0.000000,0.000000,0.000000,False,Họng/thanh quản/dị vật,,,,"Thanh thiệt: hồng, láng. Dây thanh: trơn, láng...",Hai a-mi-đan: không to. Thành sau họng: trơn l...,NUỐT VƯỚNG
426,15036.15036.0.15055,VIÊM MŨI MẠN,"HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ TAI NGOÀI, TA...",0.000000,0.000000,0.000000,False,Tai,,"Cuốn mũi: hông, gồ ghề. Khe mũi: thoáng, sạch ...",thoáng sạch,,,ĐAU HÀM T
461,15356.15356.0.15376,VIÊM HỌNG MẠN - THEO DÕI TRÀO NGƯỢC DỊCH VỊ.,HÓC XƯƠNG CÁ (ĐÃ LẤY).,0.000000,0.000000,0.000000,False,Họng/thanh quản/dị vật,,,,"Đáy lưỡi: có xương cá. Thanh thiệt: hồng, láng...",Hai a-mi-đan: không to. Thành sau họng: có hạt...,NUỐT VƯỚNG SAU ĂN CÁ
337,14009.14009.0.14028,VIÊM MŨI MẠN,VA,0.000000,0.000000,0.000000,False,Mũi/xoang/VA,P: màng nhĩ bóng sáng. T: màng nhĩ bóng sáng.,"Cuốn mũi: hồng, láng. Các khe mũi: thoáng, sạc...",VA độ II,,,Kiểm tra VA
506,16027.16027.0.16048,VIÊM MŨI MẠN,"HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ TAI NGOÀI, TA...",0.000000,0.000000,0.000000,False,Tai,"P: ống tai ngoài thoáng sạch, màng nhĩ bóng sá...","Cuốn mũi: hồng, láng. Các khe mũi: thoáng, sạc...",lỗ vòi nhĩ 2 bên thoáng sạch,,,Ù tai 2 bên
586,16806.16806.0.16828,VIÊM MŨI MẠN,"HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ TAI NGOÀI, TA...",0.000000,0.000000,0.000000,False,Tai,"P: ống tai ngoài thoáng sạch, màng nhĩ bóng sá...",,,,,NGỨA TAI 2 BÊN



=== SUMMARY ===
count    712.000000
mean       0.609702
std        0.284208
min        0.000000
25%        0.375000
50%        0.666667
75%        0.800000
max        1.000000
Zero-F1 cases: 19

Exact-match distribution:
exact_match
False    577
True     135

=== LOW-OVERLAP METADATA ===
Shape: (20, 11)
Columns: ['case_id', 'prediction', 'ground_truth', 'precision_tokens', 'recall_tokens', 'token_f1', 'pred_norm', 'gt_norm', 'exact_match', 'pred_token_count', 'gt_token_count']


,case_id,prediction,ground_truth,precision_tokens,recall_tokens,token_f1,pred_norm,gt_norm,exact_match,pred_token_count,gt_token_count
0,17557.17557.0.17582,VIÊM HỌNG MẠN - THEO DÕI TRÀO NGƯỢC DỊCH VỊ.,HÓC DỊ VẬT N1.,0.0,0.0,0.0,VIÊM HỌNG MẠN THEO DÕI TRÀO NGƯỢC DỊCH VỊ,HÓC DỊ VẬT N1,False,9,4
1,17686.17686.0.17711,VIÊM HỌNG MẠN - THEO DÕI TRÀO NGƯỢC DỊCH VỊ.,HÓC XƯƠNG CÁ N1 (ĐÃ LẤY).,0.0,0.0,0.0,VIÊM HỌNG MẠN THEO DÕI TRÀO NGƯỢC DỊCH VỊ,HÓC XƯƠNG CÁ N1 ĐÃ LẤY,False,9,6
2,14927.14927.0.14946,VIÊM MŨI MẠN,"HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ TAI NGOÀI, TA...",0.0,0.0,0.0,VIÊM MŨI MẠN,HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ TAI NGOÀI TAI...,False,3,14
3,14900.14900.0.14919,VIÊM MŨI XOANG,"HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ TAI NGOÀI, TA...",0.0,0.0,0.0,VIÊM MŨI XOANG,HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ TAI NGOÀI TAI...,False,3,14
4,11630.11630.0.11647,VIÊM HỌNG MẠN - THEO DÕI TRÀO NGƯỢC DỊCH VỊ.,LIỆT DÂY THANH P,0.0,0.0,0.0,VIÊM HỌNG MẠN THEO DÕI TRÀO NGƯỢC DỊCH VỊ,LIỆT DÂY THANH P,False,9,4
5,15036.15036.0.15055,VIÊM MŨI MẠN,"HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ TAI NGOÀI, TA...",0.0,0.0,0.0,VIÊM MŨI MẠN,HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ TAI NGOÀI TAI...,False,3,14
6,15356.15356.0.15376,VIÊM HỌNG MẠN - THEO DÕI TRÀO NGƯỢC DỊCH VỊ.,HÓC XƯƠNG CÁ (ĐÃ LẤY).,0.0,0.0,0.0,VIÊM HỌNG MẠN THEO DÕI TRÀO NGƯỢC DỊCH VỊ,HÓC XƯƠNG CÁ ĐÃ LẤY,False,9,5
7,14009.14009.0.14028,VIÊM MŨI MẠN,VA,0.0,0.0,0.0,VIÊM MŨI MẠN,VA,False,3,1
8,16027.16027.0.16048,VIÊM MŨI MẠN,"HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ TAI NGOÀI, TA...",0.0,0.0,0.0,VIÊM MŨI MẠN,HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ TAI NGOÀI TAI...,False,3,14
9,16806.16806.0.16828,VIÊM MŨI MẠN,"HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ TAI NGOÀI, TA...",0.0,0.0,0.0,VIÊM MŨI MẠN,HIỆN KHÔNG THẤY HÌNH ẢNH BỆNH LÝ TAI NGOÀI TAI...,False,3,14



Read-only audit completed; no checkpoint loaded or files changed.


In [10]:

# CELL 3A — Trace inference source and prediction export (read-only)
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import re

ROOT = '/content/drive/MyDrive/NoiSoi_Matching'
NB_PATH = '/content/drive/MyDrive/Colab Notebooks/25sep.ipynb'

patterns = [
    'test_predictions_712.csv',
    'test_predictions',
    'prediction_path',
    'model.generate',
    'generate(',
    'best.pt',
    'last.pt',
    'torch.load',
    'load_state_dict',
    'save_pretrained',
    'prediction.to_csv',
    'predictions.to_csv',
]

def scan_text(label, text, max_hits=8):
    hits = []
    lines = text.splitlines()
    for i, line in enumerate(lines, start=1):
        if any(p.lower() in line.lower() for p in patterns):
            hits.append((i, line.strip()[:240]))
    if hits:
        print(f'\n--- {label}: {len(hits)} matching lines ---')
        for line_no, line in hits[:max_hits]:
            print(f'{line_no}: {line}')
        if len(hits) > max_hits:
            print(f'... {len(hits) - max_hits} additional matches omitted')
    return len(hits)

print('=== 1. Notebook source ===')
if os.path.isfile(NB_PATH):
    with open(NB_PATH, 'r', encoding='utf-8') as f:
        nb = json.load(f)

    total_hits = 0
    for idx, cell in enumerate(nb.get('cells', [])):
        source = cell.get('source', [])
        if isinstance(source, list):
            source = ''.join(source)
        if source:
            total_hits += scan_text(
                f'Cell {idx} (execution_count={cell.get("execution_count")})',
                source,
                max_hits=5
            )
    print('\nTotal matching source lines across notebook cells:', total_hits)
else:
    print('Notebook not found at expected path:', NB_PATH)

print('\n=== 2. Project Python scripts ===')
script_count = 0
script_hits = 0

for current, dirs, files in os.walk(ROOT):
    dirs[:] = [
        d for d in dirs
        if d not in {'.ipynb_checkpoints', '__pycache__', '.git'}
    ]
    for filename in files:
        if filename.endswith('.py'):
            script_count += 1
            path = os.path.join(current, filename)
            try:
                with open(path, 'r', encoding='utf-8', errors='replace') as f:
                    text = f.read()
                hits = scan_text(path, text, max_hits=8)
                script_hits += hits
            except OSError as e:
                print('Could not read:', path, str(e))

print('\nPython scripts scanned:', script_count)
print('Matching source lines in scripts:', script_hits)

print('\n=== 3. Training summary metadata ===')
summary_path = os.path.join(
    ROOT, 'baseline_model_v5', 'diagnostics', 'v5_final_training_summary.json'
)
if os.path.isfile(summary_path):
    with open(summary_path, 'r', encoding='utf-8') as f:
        summary = json.load(f)
    print(json.dumps(summary, ensure_ascii=False, indent=2))
else:
    print('Training summary not found.')

print('\nRead-only trace finished; no checkpoint loaded and no files modified.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== 1. Notebook source ===

--- Cell 0 (execution_count=None): 1 matching lines ---
73: f"{V4_DIR}/checkpoints/best.pt"

--- Cell 1 (execution_count=None): 13 matching lines ---
90: f"{V4_DIR}/checkpoints/best.pt"
988: E2 = torch.load(
994: E3 = torch.load(
1017: load_result = model.load_state_dict(
1031: model.load_state_dict(
... 8 additional matches omitted

--- Cell 2 (execution_count=None): 2 matching lines ---
88: model.load_state_dict(E2["model_state"], strict=True)
95: model.load_state_dict(E3["model_state"], strict=True)

--- Cell 5 (execution_count=None): 1 matching lines ---
280: model.load_state_dict(

--- Cell 6 (execution_count=None): 1 matching lines ---
59: model.load_state_dict(

--- Cell 7 (execution_count=None): 1 matching lines ---
40: model.load_state_dict(E3["model_state"], strict=True)

--- Cell 9 (execution_count=None): 1 matching line

In [11]:

# CELL 3B — Inspect likely inference/export code (read-only)
from google.colab import drive
drive.mount('/content/drive')

import json
import os

NB_PATH = '/content/drive/MyDrive/Colab Notebooks/25sep.ipynb'

if not os.path.isfile(NB_PATH):
    raise FileNotFoundError(f'Notebook not found: {NB_PATH}')

with open(NB_PATH, 'r', encoding='utf-8') as f:
    nb = json.load(f)

# Terms that can identify inference, decoding, evaluation, and CSV export.
terms = [
    'test_predictions', 'to_csv', 'test_loader', 'val_loader',
    'model.generate', 'batch_decode', 'tokenizer.decode',
    'model.eval()', 'torch.no_grad', 'prediction',
    'ground_truth', 'argmax', 'generate(', 'decode(',
    'test_predictions_712'
]

# Prioritize cells that appear relevant from the earlier scan,
# while also searching every cell for prediction/export terms.
priority_cells = {
    0, 1, 2, 23, 24, 54, 75, 82, 84,
    126, 127, 139, 140
}

def get_source(cell):
    source = cell.get('source', [])
    return ''.join(source) if isinstance(source, list) else str(source)

def matching_lines(source):
    lines = source.splitlines()
    return [
        i for i, line in enumerate(lines)
        if any(term.lower() in line.lower() for term in terms)
    ]

shown = set()
print('Notebook cells:', len(nb.get('cells', [])))

# First: show concise context around relevant lines in priority cells.
print('\n=== PRIORITY CELLS ===')
for idx in sorted(priority_cells):
    cells = nb.get('cells', [])
    if idx >= len(cells):
        continue

    cell = cells[idx]
    source = get_source(cell)
    hits = matching_lines(source)
    if not hits:
        continue

    print(
        f'\n--- CELL {idx} | type={cell.get("cell_type")} | '
        f'execution_count={cell.get("execution_count")} | '
        f'{len(source.splitlines())} source lines ---'
    )

    printed = set()
    for hit in hits[:12]:
        start = max(0, hit - 3)
        end = min(len(source.splitlines()), hit + 4)
        lines = source.splitlines()
        for line_idx in range(start, end):
            if line_idx not in printed:
                print(f'{line_idx + 1}: {lines[line_idx][:220]}')
                printed.add(line_idx)
        print('...')
    shown.add(idx)

# Second: report other cells with relevant inference/export code.
print('\n=== OTHER CELLS WITH INFERENCE/EXPORT TERMS ===')
for idx, cell in enumerate(nb.get('cells', [])):
    if idx in shown:
        continue
    source = get_source(cell)
    hits = matching_lines(source)
    if hits:
        print(
            f'Cell {idx} | type={cell.get("cell_type")} | '
            f'execution_count={cell.get("execution_count")} | '
            f'lines={len(source.splitlines())} | '
            f'hits={len(hits)}'
        )
        for hit in hits[:5]:
            print(f'  {hit + 1}: {source.splitlines()[hit].strip()[:200]}')

# Finally: inspect saved outputs only for clues about prediction/export.
print('\n=== SAVED OUTPUT CLUES ===')
output_terms = [
    'test_predictions_712.csv', 'to_csv', 'prediction saved',
    'predictions saved', 'test exact match', 'generating predictions',
    'inference complete'
]
output_count = 0

for idx, cell in enumerate(nb.get('cells', [])):
    for output in cell.get('outputs', []):
        text_parts = []
        if 'text' in output:
            value = output['text']
            text_parts.append(''.join(value) if isinstance(value, list) else str(value))
        data = output.get('data', {})
        if 'text/plain' in data:
            value = data['text/plain']
            text_parts.append(''.join(value) if isinstance(value, list) else str(value))

        output_text = '\n'.join(text_parts)
        matching = [
            line for line in output_text.splitlines()
            if any(term.lower() in line.lower() for term in output_terms)
        ]
        if matching:
            output_count += len(matching)
            print(f'\nCell {idx} saved output:')
            for line in matching[:5]:
                print(' ', line[:240])

print('\nSaved-output clue lines:', output_count)
print('\nInspection complete. No notebook cells executed and no files modified.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Notebook cells: 162

=== PRIORITY CELLS ===

--- CELL 1 | type=code | execution_count=None | 1914 source lines ---
656:     max_target_length=96,
657: )
658: 
659: val_loader = DataLoader(
660:     val_dataset,
661:     batch_size=4,
662:     shuffle=False,
...
673: )
674: 
675: print(
676:     "val_loader batches:",
677:     len(val_loader)
678: )
679: 
...
680: assert len(val_dataset) == 756
...
681: assert len(val_loader) == 189
682: 
683: 
684: # ================================================================
...
937: 
938: model = V4Model().to(device)
939: 
940: model.eval()
941: 
942: print(
943:     "Model device:",
...
1053: 
1054: print("\nDATASET SANITY CHECK")
1055: 
1056: batch0 = next(iter(val_loader))
1057: 
1058: print(
1059:     "pixel_values:",
...
1104:     strict=True,
1105: )
1106: 
1107: model.eval()
1108: 
1109: model_snapshot = {
1110:

In [12]:

from google.colab import drive
drive.mount('/content/drive')

import json, re
from pathlib import Path

NB_PATH = Path('/content/drive/MyDrive/Colab Notebooks/25sep.ipynb')
assert NB_PATH.exists(), f"Không tìm thấy notebook: {NB_PATH}"

with NB_PATH.open('r', encoding='utf-8') as f:
    nb = json.load(f)

cells = nb.get('cells', [])

# Từ khóa phân biệt nạp model, inference và xuất file
patterns = {
    "checkpoint_load": re.compile(
        r'torch\.load|load_state_dict|from_pretrained|'
        r'best\.pt|last\.pt|checkpoint|ckpt_path|resume_from',
        re.I
    ),
    "inference": re.compile(
        r'\.generate\(|model\.eval\(|generate\(|'
        r'test_loader|test_dataset|argmax|beam_search|'
        r'greedy|safe_decode|tokenizer\.decode|'
        r'prediction\s*=|predictions\.append',
        re.I
    ),
    "export": re.compile(
        r'test_predictions_712|to_csv|to_json|'
        r'write_csv|prediction_path|output_path|'
        r'csv_path|DataFrame\(',
        re.I
    )
}

priority = [116, 118, 123, 124, 130, 138, 144]
print(f"Notebook: {NB_PATH}")
print(f"Tổng số cells: {len(cells)}")
print("\n" + "=" * 90)
print("PHẦN A — CÁC CELL CẦN KIỂM TRA")
print("=" * 90)

for idx in priority:
    if idx >= len(cells):
        continue

    cell = cells[idx]
    source = ''.join(cell.get('source', []))
    lines = source.splitlines()
    print(f"\n--- CELL {idx} | type={cell.get('cell_type')} "
          f"| execution_count={cell.get('execution_count')} "
          f"| source_lines={len(lines)} ---")

    # In các dòng liên quan và ngữ cảnh xung quanh
    hits = set()
    for line_no, line in enumerate(lines):
        if any(p.search(line) for p in patterns.values()):
            for j in range(max(0, line_no - 2),
                           min(len(lines), line_no + 3)):
                hits.add(j)

    if hits:
        prev = -2
        for j in sorted(hits):
            if j > prev + 1:
                print("   ...")
            print(f"{j+1:4}: {lines[j]}")
            prev = j
    else:
        print("(Không thấy từ khóa liên quan trong source.)")

    # Output đã lưu của cell — không thực thi lại
    outputs = cell.get('outputs', [])
    print(f"\n[SAVED OUTPUTS: {len(outputs)}]")
    for oi, out in enumerate(outputs):
        typ = out.get('output_type', '')
        text_parts = []
        if 'text' in out:
            text_parts.append(''.join(out['text']))
        data = out.get('data', {})
        for mime in ('text/plain', 'text/markdown'):
            if mime in data:
                value = data[mime]
                text_parts.append(''.join(value) if isinstance(value, list)
                                  else str(value))
        if out.get('ename'):
            text_parts.append(
                f"ERROR: {out.get('ename')}: {out.get('evalue', '')}"
            )
        saved_text = '\n'.join(text_parts).strip()
        if saved_text:
            print(f"[output {oi} | {typ}]")
            print(saved_text[:1800])
            if len(saved_text) > 1800:
                print("... [đã rút gọn]")
        elif typ == 'execute_result':
            print(f"[output {oi} | {typ}] (không có text hiển thị)")

print("\n" + "=" * 90)
print("PHẦN B — CÁC CELL KHÁC CÓ DẤU HIỆU NẠP CHECKPOINT + INFERENCE/EXPORT")
print("=" * 90)

for idx, cell in enumerate(cells):
    if idx in priority or cell.get('cell_type') != 'code':
        continue

    source = ''.join(cell.get('source', []))
    has_load = bool(patterns["checkpoint_load"].search(source))
    has_infer = bool(patterns["inference"].search(source))
    has_export = bool(patterns["export"].search(source))

    # Ưu tiên cell có ít nhất hai nhóm tín hiệu
    if sum([has_load, has_infer, has_export]) >= 2:
        lines = source.splitlines()
        print(f"\n--- CELL {idx} | execution_count={cell.get('execution_count')} "
              f"| lines={len(lines)} | load={has_load} "
              f"| inference={has_infer} | export={has_export} ---")
        count = 0
        for line_no, line in enumerate(lines):
            if any(p.search(line) for p in patterns.values()):
                print(f"{line_no+1:4}: {line[:240]}")
                count += 1
                if count >= 18:
                    print("... [giới hạn 18 dòng khớp]")
                    break

        # Chỉ lấy output lỗi hoặc output nhắc đến dự đoán/checkpoint
        for oi, out in enumerate(cell.get('outputs', [])):
            parts = []
            if 'text' in out:
                parts.append(''.join(out['text']))
            data = out.get('data', {})
            if 'text/plain' in data:
                v = data['text/plain']
                parts.append(''.join(v) if isinstance(v, list) else str(v))
            if out.get('ename'):
                parts.append(
                    f"ERROR: {out.get('ename')}: {out.get('evalue', '')}"
                )
            s = '\n'.join(parts).strip()
            if s and (
                out.get('ename') or
                re.search(r'predict|checkpoint|\.pt|saved|export|test_predictions',
                          s, re.I)
            ):
                print(f"[saved output {oi}] {s[:900]}")

print("\nHoàn tất: chỉ đọc notebook và outputs đã lưu; không chạy cell notebook.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Notebook: /content/drive/MyDrive/Colab Notebooks/25sep.ipynb
Tổng số cells: 162

PHẦN A — CÁC CELL CẦN KIỂM TRA

--- CELL 116 | type=code | execution_count=8 | source_lines=257 ---
   ...
   4: #
   5: # IMPORTANT:
   6: # - V4 checkpoint loaded on CPU
   7: # - NO deepcopy of V5
   8: # - NO optimizer
   ...
  17: V4_CKPT = (
  18:     "/content/drive/MyDrive/NoiSoi_Matching/"
  19:     "baseline_model_v4/checkpoints/epoch_03.pt"
  20: )
  21: 
   ...
  25: 
  26: assert os.path.isfile(V4_CKPT), \
  27:     "FAIL: V4 Epoch 3 checkpoint not found."
  28: 
  29: # ------------------------------------------------------------
   ...
  41: 
  42: # ------------------------------------------------------------
  43: # Load V4 checkpoint on CPU
  44: # ------------------------------------------------------------
  45: print("\n[LOADING]")
  46: print("Loading V4 Epo

In [3]:

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime
import os, re, json

ROOT = Path('/content/drive/MyDrive/NoiSoi_Matching')
V5 = ROOT / 'baseline_model_v5'
TARGET = 'test_predictions_712.csv'

print('=' * 80)
print('A. FILE METADATA — PREDICTIONS, CHECKPOINTS, SUMMARY')
print('=' * 80)

paths = [
    V5 / TARGET,
    V5 / 'checkpoints' / 'best.pt',
    V5 / 'checkpoints' / 'last.pt',
    V5 / 'diagnostics' / 'v5_final_training_summary.json',
    V5 / 'runtime_state' / 'environment.json',
    V5 / 'runtime_state' / 'gpu_state.json',
    V5 / 'runtime_state' / 'paths.json',
    V5 / 'runtime_state' / 'runtime_info.json',
]

for p in paths:
    print(f'\n{p}')
    if not p.exists():
        print('  STATUS: NOT FOUND')
        continue
    st = p.stat()
    print('  size_bytes:', st.st_size)
    print('  modified:', datetime.fromtimestamp(st.st_mtime).isoformat(
        sep=' ', timespec='seconds'))
    print('  created/changed metadata time:', datetime.fromtimestamp(
        st.st_ctime).isoformat(sep=' ', timespec='seconds'))

    if p.suffix.lower() == '.json' and st.st_size < 2_000_000:
        try:
            obj = json.loads(p.read_text(encoding='utf-8'))
            print('  JSON content:')
            print(json.dumps(obj, ensure_ascii=False, indent=2)[:5000])
        except Exception as e:
            print('  JSON read error:', repr(e))

print('\n' + '=' * 80)
print('B. SEARCH SMALL TEXT FILES FOR EXACT CSV NAME / EXPORT REFERENCES')
print('=' * 80)

# Search source, logs, configs and notebooks only.
# Skip large binaries, images, datasets, and checkpoint files.
extensions = {
    '.py', '.ipynb', '.json', '.jsonl', '.txt', '.log',
    '.md', '.yaml', '.yml', '.sh'
}
terms = [
    TARGET,
    'test_predictions',
    'to_csv(',
    'best.pt',
    'last.pt',
    'load_state_dict',
    'model.generate(',
    'tokenizer.decode(',
]

max_file_bytes = 20_000_000
max_hits_per_file = 12
files_scanned = 0
matches_found = 0

for dirpath, dirnames, filenames in os.walk(ROOT):
    # Avoid irrelevant bulky directories
    dirnames[:] = [
        d for d in dirnames
        if d not in {
            '.git', '__pycache__', 'node_modules',
            'images', 'image_cache', 'raw_images'
        }
    ]

    for filename in filenames:
        p = Path(dirpath) / filename
        if p.suffix.lower() not in extensions:
            continue
        try:
            if p.stat().st_size > max_file_bytes:
                continue
            content = p.read_text(encoding='utf-8', errors='ignore')
            files_scanned += 1
        except Exception:
            continue

        lines = content.splitlines()
        hits = []
        for i, line in enumerate(lines):
            if any(term.lower() in line.lower() for term in terms):
                hits.append((i + 1, line.strip()[:240]))

        if hits:
            matches_found += 1
            print(f'\n--- {p} | {len(hits)} matching lines ---')
            for line_no, line in hits[:max_hits_per_file]:
                print(f'{line_no:5}: {line}')
            if len(hits) > max_hits_per_file:
                print('  ... additional matches omitted')

print('\n' + '=' * 80)
print('C. CSV STRUCTURE AND ORDER CHECK — READ ONLY')
print('=' * 80)

csv_path = V5 / TARGET
manifest_path = ROOT / 'baseline_model_v4' / 'v4_training_manifest.csv'

if csv_path.exists():
    import pandas as pd
    pred = pd.read_csv(csv_path)
    print('CSV shape:', pred.shape)
    print('CSV columns:', pred.columns.tolist())
    print('Unique case IDs:', pred['case_id'].nunique()
          if 'case_id' in pred.columns else 'case_id column absent')
    print('First 8 rows:')
    print(pred.head(8).to_string(index=False))

    if manifest_path.exists():
        manifest = pd.read_csv(manifest_path, usecols=['case_id', 'split'])
        expected = manifest.loc[manifest['split'].eq('test'), 'case_id'].astype(str)
        actual = pred['case_id'].astype(str) if 'case_id' in pred.columns else pd.Series([], dtype=str)

        print('\nTest manifest rows:', len(expected))
        print('Prediction rows:', len(actual))
        print('Same ID set:', set(expected) == set(actual))
        print('Same ID order:', list(expected) == list(actual))
        print('First 10 manifest test IDs:', expected.head(10).tolist())
        print('First 10 prediction IDs:', actual.head(10).tolist())
    else:
        print('V4 manifest missing; skipped order comparison.')
else:
    print('Prediction CSV not found:', csv_path)

print('\nFiles scanned:', files_scanned)
print('Files with matching references:', matches_found)
print('READ-ONLY CHECK COMPLETE — no files modified; no model loaded.')


Mounted at /content/drive
A. FILE METADATA — PREDICTIONS, CHECKPOINTS, SUMMARY

/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/test_predictions_712.csv
  size_bytes: 62941
  modified: 2026-09-30 16:34:27
  created/changed metadata time: 2026-09-30 16:34:27

/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/checkpoints/best.pt
  size_bytes: 4640037647
  modified: 2026-09-29 04:20:44
  created/changed metadata time: 2026-09-29 04:20:44

/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/checkpoints/last.pt
  size_bytes: 4640037647
  modified: 2026-09-29 04:25:51
  created/changed metadata time: 2026-09-29 04:25:51

/content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/diagnostics/v5_final_training_summary.json
  size_bytes: 649
  modified: 2026-09-29 04:23:10
  created/changed metadata time: 2026-09-29 04:23:10
  JSON content:
{
  "model": "V5",
  "status": "TRAINING_COMPLETE",
  "num_epochs": 5,
  "global_step": 1920,
  "epoch": 5,
  "next_batch": 0,
  "best_va

In [4]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import json
from pathlib import Path

NOTEBOOK_PATH = Path(
    "/content/drive/MyDrive/Colab Notebooks/25sep.ipynb"
)

assert NOTEBOOK_PATH.exists(), f"Notebook not found: {NOTEBOOK_PATH}"

with NOTEBOOK_PATH.open("r", encoding="utf-8") as f:
    nb = json.load(f)

cells = nb.get("cells", [])
print("Notebook:", NOTEBOOK_PATH)
print("Total cells:", len(cells))

# Print the current source cell 139 exactly as stored.
TARGET_INDEX = 139
if TARGET_INDEX < len(cells):
    cell = cells[TARGET_INDEX]
    source = "".join(cell.get("source", []))
    print("\n" + "=" * 90)
    print(f"CELL {TARGET_INDEX} | type={cell.get('cell_type')}")
    print("=" * 90)
    print(source)
else:
    print(f"Cell {TARGET_INDEX} does not exist.")

# Search all cell source and saved output for inference/provenance clues.
terms = [
    "test_predictions_712.csv",
    "test_predictions",
    "generate(",
    "generate_one",
    "generate_batch",
    "model.generate",
    "mt5.generate",
    "tokenizer.decode",
    "batch_decode",
    "max_new_tokens",
    "num_beams",
    "do_sample",
    "best.pt",
    "last.pt",
    "load_state_dict",
    "to_csv(",
]

print("\n" + "=" * 90)
print("MATCHING NOTEBOOK CELLS")
print("=" * 90)

for i, cell in enumerate(cells):
    source = "".join(cell.get("source", []))
    outputs = cell.get("outputs", [])
    output_text = []
    for out in outputs:
        output_text.extend(out.get("text", []))
        for data in out.get("data", {}).values():
            if isinstance(data, list):
                output_text.extend(data)
    searchable = source + "\n" + "".join(output_text)

    hits = [term for term in terms if term.lower() in searchable.lower()]
    if hits:
        print(f"\n--- CELL {i} | hits: {hits} ---")
        # Print source only here; output may be large.
        print(source[:12000])
        if len(source) > 12000:
            print("[SOURCE TRUNCATED]")

Streaming output truncated to the last 5000 lines.
)

original_rng_state = capture_rng_state()

print(
    "Model tensors:",
    len(original_model_state)
)

print(
    "Optimizer state entries:",
    len(
        original_optimizer_state["state"]
    )
)

print(
    "PASS: Original V5 state captured"
)

# ------------------------------------------------------------------------------
# 8. TEMP CHECKPOINT
# ------------------------------------------------------------------------------

temp_checkpoint = os.path.join(
    V5_CHECKPOINT_DIR,
    "_audit_temp_checkpoint.pt",
)

if os.path.exists(
    temp_checkpoint
):

    os.remove(
        temp_checkpoint
    )

print("\n[TEMP CHECKPOINT]")

save_v5_checkpoint(
    path=temp_checkpoint,
    epoch=test_epoch,
    model=v5,
    optimizer=optimizer_v5,
    scheduler=scheduler_v5,
    best_val=test_best_val,
    history=test_history,
)

if not os.path.exists(
    temp_checkpoint
):

    raise RuntimeError(
        "Temporary checkpoint was 

In [5]:

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import json
from pathlib import Path

NOTEBOOK_PATH = Path(
    "/content/drive/MyDrive/Colab Notebooks/25sep.ipynb"
)

assert NOTEBOOK_PATH.exists(), f"Notebook not found: {NOTEBOOK_PATH}"

with NOTEBOOK_PATH.open("r", encoding="utf-8") as f:
    nb = json.load(f)

terms = [
    "test_predictions_712.csv",
    "test_predictions",
    "prediction_path",
    "to_csv(",
    "model.generate",
    "mt5.generate",
    "generate_one",
    "generate_batch",
    "max_new_tokens",
    "max_length",
    "num_beams",
    "do_sample",
    "early_stopping",
    "load_state_dict",
    "torch.load(",
    "best.pt",
    "last.pt",
]

print("Notebook cells:", len(nb.get("cells", [])))
print("Searching source only; no model/checkpoint will be loaded.")

for i, cell in enumerate(nb.get("cells", [])):
    source = "".join(cell.get("source", []))
    lines = source.splitlines()

    hits = [
        j for j, line in enumerate(lines)
        if any(term.lower() in line.lower() for term in terms)
    ]

    if not hits:
        continue

    # Merge nearby line ranges to avoid dumping entire training cells.
    ranges = []
    for j in hits:
        start, end = max(0, j - 4), min(len(lines), j + 6)
        if ranges and start <= ranges[-1][1]:
            ranges[-1] = (ranges[-1][0], max(ranges[-1][1], end))
        else:
            ranges.append((start, end))

    print(f"\n{'=' * 30} CELL {i} {'=' * 30}")
    for start, end in ranges:
        print(f"\n--- source lines {start + 1}-{end} ---")
        for k in range(start, end):
            print(f"{k + 1:4}: {lines[k]}")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Notebook cells: 162
Searching source only; no model/checkpoint will be loaded.

============================== CELL 0 ==============================

--- source lines 69-74 ---
  69:     f"{V4_DIR}/checkpoints/epoch_03.pt"
  70: )
  71: 
  72: BEST_CKPT = (
  73:     f"{V4_DIR}/checkpoints/best.pt"
  74: )

============================== CELL 1 ==============================

--- source lines 86-95 ---
  86:     f"{V4_DIR}/checkpoints/epoch_03.pt"
  87: )
  88: 
  89: BEST_CKPT = (
  90:     f"{V4_DIR}/checkpoints/best.pt"
  91: )
  92: 
  93: # ------------------------------------------------
  94: # Path checks
  95: # ------------------------------------------------

--- source lines 564-575 ---
 564:         )
 565: 
 566:         labels = self.tokenizer(
 567:             report,
 568:             padding="max_length",
 569:             truncation=True,


In [6]:

from google.colab import drive
drive.mount("/content/drive")

import os
import re
import json
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/NoiSoi_Matching")
NB = Path("/content/drive/MyDrive/Colab Notebooks/25sep.ipynb")
TARGET = "test_predictions_712.csv"

print("=" * 80)
print("1. CSV METADATA — NO CONTENT MODIFICATION")
print("=" * 80)

csv_path = ROOT / "baseline_model_v5" / TARGET
if csv_path.exists():
    st = csv_path.stat()
    print("Path:", csv_path)
    print("Size:", st.st_size, "bytes")
    print("Modified:", __import__("datetime").datetime.fromtimestamp(
        st.st_mtime).isoformat())
    with open(csv_path, "r", encoding="utf-8-sig") as f:
        print("Header:", f.readline().strip())
else:
    print("CSV not found at expected path:", csv_path)

# Search for source that creates predictions, invokes generation,
# loads a model for inference, or writes the target CSV.
patterns = re.compile(
    r"test_predictions_712|test_predictions|prediction_path|"
    r"predictions.*to_csv|to_csv.*predictions|"
    r"generate_one|generate_batch|model\.generate|mt5\.generate|"
    r"tokenizer\.decode|safe_decode|num_beams|max_new_tokens|"
    r"do_sample|early_stopping|v5\.load_state_dict|"
    r"BEST_CKPT|LAST_CKPT|best\.pt|last\.pt",
    re.IGNORECASE
)

print("\n" + "=" * 80)
print("2. NOTEBOOK SOURCE MATCHES")
print("=" * 80)

if NB.exists():
    with open(NB, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    hits = 0
    for i, cell in enumerate(notebook.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue
        src = "".join(cell.get("source", []))
        lines = src.splitlines()
        matches = [
            j for j, line in enumerate(lines)
            if patterns.search(line)
        ]
        if not matches:
            continue

        # Merge nearby matching lines into compact context ranges.
        ranges = []
        for j in matches:
            a, b = max(0, j - 3), min(len(lines), j + 4)
            if ranges and a <= ranges[-1][1]:
                ranges[-1] = (ranges[-1][0], max(ranges[-1][1], b))
            else:
                ranges.append((a, b))

        print(f"\n--- CELL {i} | {len(matches)} matching lines ---")
        for a, b in ranges[:12]:
            for j in range(a, b):
                print(f"{j+1:4}: {lines[j][:300]}")
            print("...")
        hits += len(matches)

    print("\nTotal matching source lines:", hits)
else:
    print("Notebook not found:", NB)

print("\n" + "=" * 80)
print("3. PROJECT SOURCE FILES")
print("=" * 80)

source_exts = {".py", ".ipynb", ".sh", ".json", ".yaml", ".yml", ".md", ".txt"}
file_count = 0
match_count = 0
max_files = 5000

for dirpath, dirnames, filenames in os.walk(ROOT):
    # Skip large/non-source directories.
    dirnames[:] = [
        d for d in dirnames
        if d not in {".git", "__pycache__", "node_modules"}
    ]

    for name in filenames:
        p = Path(dirpath) / name
        if p.suffix.lower() not in source_exts:
            continue
        file_count += 1
        if file_count > max_files:
            print("Stopped at file limit:", max_files)
            break

        try:
            if p.stat().st_size > 5 * 1024 * 1024:
                continue
            if p.suffix.lower() == ".ipynb":
                with open(p, "r", encoding="utf-8") as f:
                    obj = json.load(f)
                chunks = [
                    "".join(c.get("source", []))
                    for c in obj.get("cells", [])
                    if c.get("cell_type") == "code"
                ]
                text = "\n".join(chunks)
            else:
                text = p.read_text(encoding="utf-8", errors="ignore")

            lines = text.splitlines()
            found = [
                j for j, line in enumerate(lines)
                if patterns.search(line)
            ]
            if found:
                match_count += len(found)
                print(f"\nFILE: {p}")
                for j in found[:12]:
                    a, b = max(0, j-2), min(len(lines), j+3)
                    print(f"  line {j+1}:")
                    for k in range(a, b):
                        print(f"    {k+1}: {lines[k][:250]}")
                if len(found) > 12:
                    print(f"  ... {len(found)-12} additional matching lines")
        except Exception as e:
            print("Skipped unreadable file:", p, "|", type(e).__name__)

    if file_count > max_files:
        break

print("\nSource files scanned:", min(file_count, max_files))
print("Matching lines in project sources:", match_count)
print("\nREAD-ONLY AUDIT COMPLETE — no checkpoint loaded; no files written.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1. CSV METADATA — NO CONTENT MODIFICATION
Path: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/test_predictions_712.csv
Size: 62941 bytes
Modified: 2026-09-30T16:34:27
Header: case_id,prediction,ground_truth

2. NOTEBOOK SOURCE MATCHES

--- CELL 0 | 2 matching lines ---
  69:     f"{V4_DIR}/checkpoints/epoch_03.pt"
  70: )
  71: 
  72: BEST_CKPT = (
  73:     f"{V4_DIR}/checkpoints/best.pt"
  74: )
...

--- CELL 1 | 9 matching lines ---
  86:     f"{V4_DIR}/checkpoints/epoch_03.pt"
  87: )
  88: 
  89: BEST_CKPT = (
  90:     f"{V4_DIR}/checkpoints/best.pt"
  91: )
  92: 
  93: # ------------------------------------------------
...
 101:     "V4 config": CONFIG_PATH,
 102:     "E2 checkpoint": E2_CKPT,
 103:     "E3 checkpoint": E3_CKPT,
 104:     "Best checkpoint": BEST_CKPT,
 105: }
 106: 
 107: print("\nPATH CHECK")
...
1116: # SAFE DECODE
1117: 

In [7]:

# ============================================================
# ENT PROJECT — WIDE READ-ONLY PROVENANCE SEARCH
# Không load checkpoint; không train; không tạo/sửa/xóa file.
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os, re, json, glob, sqlite3, hashlib
from pathlib import Path
from datetime import datetime

ROOT = Path("/content/drive/MyDrive/NoiSoi_Matching")
NB_DIR = Path("/content/drive/MyDrive/Colab Notebooks")
TARGET = "test_predictions_712.csv"
MAX_TEXT_BYTES = 8 * 1024 * 1024
MAX_CONTEXT_PER_FILE = 16
MAX_LINE_LEN = 260

# Nhóm từ khóa rộng, bao gồm trường hợp tên file không xuất hiện trực tiếp.
PATTERNS = {
    "TARGET / PREDICTION": re.compile(
        r"test_predictions_712|test_predictions|predictions?_712|"
        r"prediction|preds?\b|ground_truth|case_id.*prediction|"
        r"prediction.*case_id|inference|infer\b|evaluation|eval_results",
        re.I
    ),
    "GENERATION / DECODING": re.compile(
        r"\.generate\s*\(|generate_one|generate_batch|generate_text|"
        r"max_new_tokens|max_length|num_beams|do_sample|"
        r"early_stopping|greedy|argmax|tokenizer\.decode|"
        r"batch_decode|safe_decode|decoder_input_ids",
        re.I
    ),
    "FILE WRITING / EXPORT": re.compile(
        r"\.to_csv\s*\(|csv\.writer|DictWriter|"
        r"open\s*\(.*[\"']w|write_text|writelines|"
        r"json\.dump|torch\.save|pickle\.dump|"
        r"shutil\.(copy|move)|os\.rename|Path\(.*\)\.rename|"
        r"cp\s+|mv\s+",
        re.I
    ),
    "CHECKPOINT / MODEL LOADING": re.compile(
        r"best\.pt|last\.pt|checkpoint|state_dict|"
        r"torch\.load|load_from_checkpoint|from_pretrained|"
        r"glob\s*\(|os\.listdir|safetensors|"
        r"strict\s*=\s*False|load_state_dict",
        re.I
    ),
    "V5 MODEL / DATA PIPELINE": re.compile(
        r"baseline_model_v5|test_loader|v5_test|"
        r"pixel_values|image_mask|concept_targets|"
        r"MAX_IMAGES|MAX_TARGET_LENGTH|mt5|vit_model|"
        r"visual_projector|structured_conditioner",
        re.I
    ),
}

SOURCE_EXTS = {
    ".py", ".ipynb", ".sh", ".md", ".txt", ".json",
    ".yaml", ".yml", ".log", ".out", ".err"
}
SKIP_DIRS = {
    ".git", "__pycache__", "node_modules", ".cache",
    "site-packages", "dist-packages"
}

def stamp(ts):
    try:
        return datetime.fromtimestamp(ts).isoformat(sep=" ", timespec="seconds")
    except Exception:
        return "unknown"

def short(s, limit=MAX_LINE_LEN):
    s = str(s).replace("\t", "    ")
    return s if len(s) <= limit else s[:limit] + " ...[truncated]"

def safe_read_text(path):
    try:
        if path.stat().st_size > MAX_TEXT_BYTES:
            return None
        return path.read_text(encoding="utf-8", errors="replace")
    except Exception:
        return None

def inspect_notebook(path):
    """Search notebook code AND saved cell outputs."""
    try:
        obj = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return []

    found = []
    for idx, cell in enumerate(obj.get("cells", [])):
        source = "".join(cell.get("source", []))
        exec_count = cell.get("execution_count", None)

        # Code source
        for line_no, line in enumerate(source.splitlines(), 1):
            cats = [k for k, p in PATTERNS.items() if p.search(line)]
            if cats:
                found.append({
                    "cell": idx, "where": "SOURCE",
                    "line": line_no, "cats": cats,
                    "text": short(line), "exec": exec_count
                })

        # Saved outputs may reveal a path or generation parameters
        outputs = cell.get("outputs", [])
        for oi, out in enumerate(outputs):
            chunks = []
            data = out.get("data", {})
            for key in ("text/plain", "text/html", "application/json"):
                value = data.get(key)
                if value is not None:
                    chunks.append(
                        "\n".join(value) if isinstance(value, list)
                        else str(value)
                    )
            if out.get("text"):
                value = out["text"]
                chunks.append(
                    "\n".join(value) if isinstance(value, list)
                    else str(value)
                )
            if out.get("ename"):
                chunks.append(
                    f"EXCEPTION {out.get('ename')}: {out.get('evalue')}"
                )

            output_text = "\n".join(chunks)
            for line_no, line in enumerate(output_text.splitlines(), 1):
                cats = [k for k, p in PATTERNS.items() if p.search(line)]
                if cats:
                    found.append({
                        "cell": idx, "where": f"OUTPUT[{oi}]",
                        "line": line_no, "cats": cats,
                        "text": short(line), "exec": exec_count
                    })

    return found

print("\n" + "=" * 90)
print("A. ENVIRONMENT / EXPECTED ARTIFACT")
print("=" * 90)

csv_path = ROOT / "baseline_model_v5" / TARGET
if csv_path.exists():
    st = csv_path.stat()
    print("CSV:", csv_path)
    print("Bytes:", st.st_size, "| Modified:", stamp(st.st_mtime))
    with open(csv_path, "rb") as f:
        head = f.read(4096)
    print("SHA256(first 4KB):", hashlib.sha256(head).hexdigest())
    print("Header:", head.splitlines()[0][:500].decode("utf-8", "replace")
          if head else "(empty)")
else:
    print("Expected CSV not present at:", csv_path)

print("\n" + "=" * 90)
print("B. NOTEBOOKS — SEARCH SOURCE + SAVED OUTPUTS")
print("=" * 90)

notebooks = []
for base in (NB_DIR, ROOT):
    if base.exists():
        for p in base.rglob("*.ipynb"):
            if not any(part in SKIP_DIRS for part in p.parts):
                notebooks.append(p)

# De-duplicate paths
notebooks = sorted(set(notebooks))
print("Notebook files found:", len(notebooks))

notebook_hits = []
for p in notebooks:
    hits = inspect_notebook(p)
    if hits:
        notebook_hits.append((p, hits))

for p, hits in notebook_hits:
    print(f"\nNOTEBOOK: {p}")
    print("Modified:", stamp(p.stat().st_mtime), "| matching lines:", len(hits))

    # Highest-priority matches first: literal target, file writing, generation.
    def priority(h):
        s = h["text"].lower()
        if TARGET.lower() in s: return 0
        if "to_csv" in s or "csv.writer" in s or "open(" in s: return 1
        if "generate" in s or "decode" in s: return 2
        if "best.pt" in s or "last.pt" in s or "torch.load" in s: return 3
        return 4

    chosen = sorted(hits, key=priority)[:MAX_CONTEXT_PER_FILE]
    for h in chosen:
        print(
            f"  cell={h['cell']} exec={h['exec']} "
            f"{h['where']}:{h['line']} [{','.join(h['cats'])}]"
        )
        print("   ", h["text"])
    if len(hits) > len(chosen):
        print(f"  ... {len(hits)-len(chosen)} more matching lines omitted")

print("\n" + "=" * 90)
print("C. PROJECT SOURCE / LOG FILES")
print("=" * 90)

file_hits = []
scanned = 0
MAX_FILES = 12000

for base in (ROOT, NB_DIR):
    if not base.exists():
        continue
    for dirpath, dirnames, filenames in os.walk(base):
        dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS]
        for name in filenames:
            p = Path(dirpath) / name
            if p.suffix.lower() not in SOURCE_EXTS:
                continue
            if p.suffix.lower() == ".ipynb":
                continue  # already inspected with cell/output context
            scanned += 1
            if scanned > MAX_FILES:
                break
            content = safe_read_text(p)
            if content is None:
                continue

            matches = []
            for ln, line in enumerate(content.splitlines(), 1):
                cats = [k for k, pat in PATTERNS.items() if pat.search(line)]
                if cats:
                    matches.append((ln, cats, short(line)))

            if matches:
                file_hits.append((p, matches))
        if scanned > MAX_FILES:
            break
    if scanned > MAX_FILES:
        break

print("Text/source files scanned:", min(scanned, MAX_FILES))
for p, matches in sorted(file_hits, key=lambda x: (
    0 if TARGET.lower() in str(x[0]).lower() else 1,
    -len(x[1])
))[:100]:
    print(f"\nFILE: {p} | modified={stamp(p.stat().st_mtime)} | hits={len(matches)}")
    for ln, cats, line in matches[:10]:
        print(f"  L{ln} [{','.join(cats)}] {line}")
    if len(matches) > 10:
        print(f"  ... {len(matches)-10} additional matches")

print("\n" + "=" * 90)
print("D. CANDIDATE ARTIFACTS — CSV / PREDICTION / LOG / CHECKPOINT METADATA")
print("=" * 90)

candidate_pattern = re.compile(
    r"(predict|preds?|infer|test|submission|result|output|checkpoint|"
    r"best|last|v5|audit|log|history)",
    re.I
)
candidate_files = []

for base in (ROOT, NB_DIR):
    if not base.exists():
        continue
    for dirpath, dirnames, filenames in os.walk(base):
        dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS]
        for name in filenames:
            p = Path(dirpath) / name
            if candidate_pattern.search(name):
                try:
                    st = p.stat()
                    candidate_files.append((
                        p, st.st_size, st.st_mtime, p.suffix.lower()
                    ))
                except Exception:
                    pass

# Show likely predictions first; then newest files.
candidate_files.sort(key=lambda x: (
    0 if re.search(r"predict|preds?|inference|submission", x[0].name, re.I) else 1,
    -x[2]
))
print("Candidate files:", len(candidate_files))
for p, size, mtime, ext in candidate_files[:180]:
    print(f"{stamp(mtime)} | {size:>12} bytes | {p}")

if len(candidate_files) > 180:
    print("... remaining candidates omitted")

print("\n" + "=" * 90)
print("E. IPYTHON HISTORY — IF THIS RUNTIME HAS PERSISTED IT")
print("=" * 90)

history_candidates = [
    Path.home() / ".ipython/profile_default/history.sqlite",
    Path("/root/.ipython/profile_default/history.sqlite"),
]
history_found = False

for hp in history_candidates:
    if not hp.exists():
        continue
    history_found = True
    print("\nHistory DB:", hp, "| modified:", stamp(hp.stat().st_mtime))
    try:
        # SQLite read-only URI: do not alter the database.
        con = sqlite3.connect("file:" + str(hp) + "?mode=ro", uri=True)
        tables = [
            row[0] for row in con.execute(
                "SELECT name FROM sqlite_master WHERE type='table'"
            )
        ]
        print("Tables:", tables)
        if "history" in tables:
            rows = con.execute(
                "SELECT session, line, source FROM history "
                "ORDER BY session DESC, line DESC LIMIT 20000"
            ).fetchall()
            selected = []
            hist_pat = re.compile(
                r"test_predictions|to_csv|generate|decode|"
                r"best\.pt|last\.pt|torch\.load|load_state_dict|"
                r"prediction|inference|baseline_model_v5",
                re.I
            )
            for session, line_no, source in rows:
                if hist_pat.search(source or ""):
                    selected.append((session, line_no, source))
            print("Relevant history commands:", len(selected))
            for session, line_no, source in selected[:100]:
                print(f"\n  session={session} line={line_no}")
                print(" ", short(source, 500))
            if len(selected) > 100:
                print(" ... remaining history matches omitted")
        con.close()
    except Exception as e:
        print("History unreadable:", type(e).__name__, str(e)[:200])

if not history_found:
    print("No persisted IPython history DB found in standard paths.")
    print("This is normal if the runtime was reset or history was not persisted.")

print("\n" + "=" * 90)
print("F. CURRENT RUNTIME — RECENT RELEVANT FILES")
print("=" * 90)

runtime_roots = [Path("/content"), Path("/tmp")]
runtime_candidates = []
for base in runtime_roots:
    if not base.exists():
        continue
    try:
        for dirpath, dirnames, filenames in os.walk(base):
            dirnames[:] = [
                d for d in dirnames
                if d not in {"node_modules", "__pycache__", ".git"}
            ]
            for name in filenames:
                p = Path(dirpath) / name
                if not candidate_pattern.search(name):
                    continue
                try:
                    st = p.stat()
                    # Avoid listing huge unrelated model/data files.
                    runtime_candidates.append((p, st.st_size, st.st_mtime))
                except Exception:
                    pass
    except Exception:
        pass

runtime_candidates.sort(key=lambda x: -x[2])
print("Runtime candidates:", len(runtime_candidates))
for p, size, mtime in runtime_candidates[:100]:
    print(f"{stamp(mtime)} | {size:>12} bytes | {p}")

print("\n" + "=" * 90)
print("SEARCH COMPLETE")
print("Read-only: no checkpoint loaded, no training, no predictions generated,")
print("and no existing project artifacts modified.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

A. ENVIRONMENT / EXPECTED ARTIFACT
CSV: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/test_predictions_712.csv
Bytes: 62941 | Modified: 2026-09-30 16:34:27
SHA256(first 4KB): b3ffae61834f82a3b25250a9f864b65b33540c767354a16017dc67618521511b
Header: ﻿case_id,prediction,ground_truth

B. NOTEBOOKS — SEARCH SOURCE + SAVED OUTPUTS
Notebook files found: 61

NOTEBOOK: /content/drive/MyDrive/Colab Notebooks/KMeans.ipynb
Modified: 2025-01-29 16:36:05 | matching lines: 6
  cell=10 exec=13 SOURCE:1 [TARGET / PREDICTION]
    y_pred=k_mean_model.predict(X)
  cell=10 exec=13 SOURCE:4 [TARGET / PREDICTION]
    plt.scatter(X[:, 0], X[:, 1], c=y_pred)
  cell=14 exec=15 SOURCE:1 [TARGET / PREDICTION]
    y_pred=k_mean_model.predict(X)
  cell=14 exec=15 SOURCE:4 [TARGET / PREDICTION]
    plt.scatter(X[:, 0], X[:, 1], c=y_pred)
  cell=18 exec=17 SOURCE:1 [TARGET / PR

KeyboardInterrupt: 

In [8]:
# === ENT V5 — read-only provenance + decoding audit (no checkpoint load, no GPU) ===
from google.colab import drive
drive.mount('/content/drive')

import os, re, json, datetime, unicodedata
import pandas as pd

ROOT = '/content/drive/MyDrive/NoiSoi_Matching'
V5   = f'{ROOT}/baseline_model_v5'
AUD  = f'{V5}/diagnostics/audit_20261015'
OUT_DIR = f'{V5}/diagnostics/audit_20261015_review'      # NEW folder, existing files untouched
os.makedirs(OUT_DIR, exist_ok=True)
REPORT = f'{OUT_DIR}/provenance_review_{datetime.datetime.now():%Y%m%d_%H%M%S}.txt'

lines = []
def log(*a):
    s = ' '.join(str(x) for x in a); print(s); lines.append(s)
def stat(p):
    if not os.path.exists(p): return 'MISSING'
    st = os.stat(p)
    return f'{st.st_size:,} B | modified {datetime.datetime.fromtimestamp(st.st_mtime)} (runtime clock, usually UTC)'

# ---------- 1. File status ----------
FILES = {
    'cell11':    f'{AUD}/cell11_prediction_generation_source.txt',
    'cell12':    f'{AUD}/cell12_checkpoint_and_prediction_source.txt',
    'cell13':    f'{AUD}/cell13_full_notebook_prediction_trace.txt',
    'summary':   f'{V5}/diagnostics/v5_final_training_summary.json',
    'paths':     f'{V5}/runtime_state/paths.json',
    'v4_status': f'{ROOT}/baseline_model_v4/v4_status_2026-09-24.txt',
}
log('=== 1. FILE STATUS ===')
for k, p in FILES.items():
    log(f'{k:10s} {stat(p)}  {p}')
for p in [f'{V5}/test_predictions_712.csv', f'{V5}/checkpoints/best.pt', f'{V5}/checkpoints/last.pt']:
    log(f'{"":10s} {stat(p)}  {p}')

texts = {}
for k, p in FILES.items():
    if os.path.exists(p):
        with open(p, encoding='utf-8', errors='replace') as f:
            texts[k] = f.read()

# ---------- 2. Keyword hits with context ----------
PATTERNS = [r'test_predictions', r'to_csv', r'best\.pt', r'last\.pt', r'torch\.load', r'load_state_dict',
            r'strict\s*=', r'missing_keys|unexpected_keys', r'def\s+\w*(generate|decode)\w*', r'\.generate\(',
            r'argmax', r'max_new_tokens', r'decoder_start_token_id', r'eos_token_id', r'pad_token_id',
            r'extra_id', r'skip_special_tokens', r'num_beams', r'max_target', r'max_images|MAX_IMAGES',
            r'NO_SIGNAL', r'model\.eval|\.eval\(\)', r'no_grad|inference_mode', r'\.lower\(\)|\.upper\(\)',
            r'execution_count']
log('\n=== 2. KEYWORD HITS (±2 lines, max 12 per pattern per file) ===')
for k, t in texts.items():
    if k in ('summary', 'paths'):
        continue
    L = t.splitlines()
    log(f'\n##### {k}: {len(L)} lines #####')
    for pat in PATTERNS:
        idx = [i for i, l in enumerate(L) if re.search(pat, l)]
        if not idx:
            continue
        log(f'\n-- /{pat}/ : {len(idx)} hits')
        for i in idx[:12]:
            for j in range(max(0, i-2), min(len(L), i+3)):
                log(f'{">>" if j == i else "  "} {j+1:5d}: {L[j][:220]}')
            log('   ...')

for k in ('summary', 'paths'):
    if k in texts:
        log(f'\n=== {k} (full) ===\n{texts[k]}')

# ---------- 3. Recompute prediction stats (CSV read-only) ----------
log('\n=== 3. PREDICTION CSV STATS ===')
df = pd.read_csv(f'{V5}/test_predictions_712.csv')
norm = lambda s: re.sub(r'\s+', ' ', str(s)).strip().lower()
pn, gn = df['prediction'].map(norm), df['ground_truth'].map(norm)
vc = pn.value_counts()
log('rows', len(df), '| unique pred', pn.nunique(), '| unique GT', gn.nunique())
log('top1 share', round(vc.iloc[0]/len(df), 4), '| top2 share', round(vc.iloc[:2].sum()/len(df), 4))
log('exact match', round((pn == gn).mean(), 4))
log('empty preds', (pn.str.len() == 0).sum(),
    '| preds w/ special tokens', df['prediction'].astype(str).str.contains(r'<extra_id|<pad>|</s>|<unk>').sum())
log('pred top 16:\n' + vc.head(16).to_string())
log('pred word count:', pn.str.split().str.len().describe().round(2).to_dict())
log('GT   word count:', gn.str.split().str.len().describe().round(2).to_dict())

# Unicode consistency (NFC vs NFD Vietnamese diacritics can silently split labels)
for name, col in [('GT', df['ground_truth']), ('pred', df['prediction'])]:
    nonnfc = col.astype(str).map(lambda s: unicodedata.normalize('NFC', s) != s).sum()
    log(f'{name}: strings not in NFC form = {nonnfc}')

# ---------- 4. Token lengths vs the 20-token generation cap ----------
log('\n=== 4. mT5 TOKEN LENGTHS ===')
try:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained('google/mt5-small')
    ids = lambda s: tok(str(s), add_special_tokens=True)['input_ids']    # includes </s>
    def lens(series): return series.map(lambda s: len(ids(s)))
    gt_len, pr_len = lens(df['ground_truth']), lens(df['prediction'])
    log('test GT tokens:', gt_len.describe().round(2).to_dict())
    log('test GT > 20 tokens:', (gt_len > 20).sum(), f'({(gt_len > 20).mean():.1%})', '| > 96:', (gt_len > 96).sum())
    log('pred tokens:', pr_len.describe().round(2).to_dict(), '| preds with >= 20 tokens:', (pr_len >= 20).sum())
    unk = tok.unk_token_id
    log('GT strings containing <unk> after tokenization:', df['ground_truth'].map(lambda s: unk in ids(s)).sum())
    rt = df['ground_truth'].map(lambda s: norm(tok.decode(ids(s), skip_special_tokens=True)) != norm(s)).sum()
    log('GT strings that do not round-trip through tokenizer:', rt)
    man = pd.read_csv(f'{ROOT}/baseline_model_v4/v4_training_manifest.csv', usecols=['split', 'ket_luan'])
    tr_len = lens(man.loc[man['split'] == 'train', 'ket_luan'].dropna())
    log('train GT tokens:', tr_len.describe().round(2).to_dict())
    log('train GT > 20 tokens:', (tr_len > 20).sum(), f'({(tr_len > 20).mean():.1%})')
except Exception as e:
    log('Tokenizer check skipped:', repr(e))

# ---------- Save report (summary + full source texts) ----------
with open(REPORT, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
    f.write('\n\n=== FULL SOURCE TEXTS ===\n')
    for k, t in texts.items():
        f.write(f'\n\n##### {k} ({FILES[k]}) #####\n{t}')
print('\nReport saved to:', REPORT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== 1. FILE STATUS ===
cell11     8,891 B | modified 2026-10-14 20:53:50 (runtime clock, usually UTC)  /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/diagnostics/audit_20261015/cell11_prediction_generation_source.txt
cell12     32,874 B | modified 2026-10-14 20:54:25 (runtime clock, usually UTC)  /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/diagnostics/audit_20261015/cell12_checkpoint_and_prediction_source.txt
cell13     5,548 B | modified 2026-10-14 20:55:51 (runtime clock, usually UTC)  /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/diagnostics/audit_20261015/cell13_full_notebook_prediction_trace.txt
summary    649 B | modified 2026-09-29 04:23:10 (runtime clock, usually UTC)  /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/diagnostics/v5_final_training_summary.json
paths      719 B | modified 2026-09-28 08:24:55 (ru

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

test GT tokens: {'count': 712.0, 'mean': 20.92, 'std': 12.01, 'min': 2.0, '25%': 10.0, '50%': 18.0, '75%': 30.0, 'max': 66.0}
test GT > 20 tokens: 321 (45.1%) | > 96: 0
pred tokens: {'count': 712.0, 'mean': 14.94, 'std': 7.15, 'min': 7.0, '25%': 10.0, '50%': 10.0, '75%': 22.0, 'max': 30.0} | preds with >= 20 tokens: 214
GT strings containing <unk> after tokenization: 0
GT strings that do not round-trip through tokenizer: 0
train GT tokens: {'count': 6138.0, 'mean': 19.75, 'std': 11.54, 'min': 2.0, '25%': 10.0, '50%': 17.0, '75%': 27.0, 'max': 70.0}
train GT > 20 tokens: 2491 (40.6%)

Report saved to: /content/drive/MyDrive/NoiSoi_Matching/baseline_model_v5/diagnostics/audit_20261015_review/provenance_review_20261015_125928.txt


In [9]:
# === Region-level collapse check (read-only, CPU) ===
from google.colab import drive
drive.mount('/content/drive')

import os, re, datetime
import pandas as pd

ROOT = '/content/drive/MyDrive/NoiSoi_Matching'
V5   = f'{ROOT}/baseline_model_v5'
OUT  = f'{V5}/diagnostics/audit_20261015_review'      # new folder only
os.makedirs(OUT, exist_ok=True)
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

df = pd.read_csv(f'{V5}/test_predictions_712.csv')    # read-only
REG = {
    'Tai':  r'\b(tai|nhĩ)\b',
    'Mũi':  r'\b(mũi|xoang|va|polyp|cuốn)\b',
    'Họng': r'\b(họng|thanh quản|amidan|amiđan|dị vật|trào ngược|xương cá)\b',
}
def regions(s):
    s = str(s).lower()
    r = [k for k, p in REG.items() if re.search(p, s)]
    return '+'.join(r) if r else 'Khác'

df['gt_region']   = df['ground_truth'].map(regions)
df['pred_region'] = df['prediction'].map(regions)

ct = pd.crosstab(df['gt_region'], df['pred_region'], margins=True)
print('=== GT region (rows) vs predicted region (cols) ===\n', ct, '\n')

print('=== Single-region GT cases ===')
rows = []
for reg in REG:
    sub = df[df['gt_region'] == reg]
    if sub.empty:
        continue
    r = {
        'region': reg, 'cases': len(sub),
        'pred_region_correct': round(sub['pred_region'].str.contains(reg).mean(), 3),
        'unique_GT': sub['ground_truth'].str.lower().str.strip().nunique(),
        'unique_pred': sub['prediction'].str.lower().str.strip().nunique(),
        'top_pred_share': round(sub['prediction'].str.lower().value_counts(normalize=True).iloc[0], 3),
        'exact': round((sub['prediction'].str.lower().str.strip() == sub['ground_truth'].str.lower().str.strip()).mean(), 3),
    }
    rows.append(r); print(r)

ct.to_csv(f'{OUT}/region_crosstab_{ts}.csv')
pd.DataFrame(rows).to_csv(f'{OUT}/region_summary_{ts}.csv', index=False)
print('\nSaved to', OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== GT region (rows) vs predicted region (cols) ===
 pred_region   Họng  Mũi  Tai  Tai+Mũi  All
gt_region                                 
Họng            75    0    0        0   75
Khác             5    0    0        0    5
Mũi              1  326   23        2  352
Mũi+Họng         0    7    0        0    7
Tai              0   20   97        2  119
Tai+Họng         0    0    1        0    1
Tai+Mũi          0   78   66        8  152
Tai+Mũi+Họng     0    0    1        0    1
All             81  431  188       12  712 

=== Single-region GT cases ===
{'region': 'Tai', 'cases': 119, 'pred_region_correct': np.float64(0.832), 'unique_GT': 53, 'unique_pred': 10, 'top_pred_share': np.float64(0.613), 'exact': np.float64(0.042)}
{'region': 'Mũi', 'cases': 352, 'pred_region_correct': np.float64(0.932), 'unique_GT': 78, 'unique_pred': 10, 'top_pred_share': np.float6

In [10]:
# === V5 vs region lookup-table baselines (read-only, CPU) ===
from google.colab import drive
drive.mount('/content/drive')

import os, re, datetime, collections
import pandas as pd

ROOT = '/content/drive/MyDrive/NoiSoi_Matching'
V5   = f'{ROOT}/baseline_model_v5'
OUT  = f'{V5}/diagnostics/audit_20261015_review'
os.makedirs(OUT, exist_ok=True)
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

REG = {
    'Tai':  r'\b(tai|nhĩ)\b',
    'Mũi':  r'\b(mũi|xoang|va|polyp|cuốn)\b',
    'Họng': r'\b(họng|thanh quản|amidan|amiđan|dị vật|trào ngược|xương cá)\b',
}
def regions(s):
    s = str(s).lower()
    r = [k for k, p in REG.items() if re.search(p, s)]
    return '+'.join(r) if r else 'Khác'
def norm(s):
    s = re.sub(r'[.,;:()\-–]', ' ', str(s).lower())
    return re.sub(r'\s+', ' ', s).strip()
def f1(p, g):
    pt, gt = norm(p).split(), norm(g).split()
    if not pt or not gt:
        return float(pt == gt)
    common = sum((collections.Counter(pt) & collections.Counter(gt)).values())
    if common == 0:
        return 0.0
    pr, rc = common / len(pt), common / len(gt)
    return 2 * pr * rc / (pr + rc)

# Lookup table from TRAIN labels only
man = pd.read_csv(f'{ROOT}/baseline_model_v4/v4_training_manifest.csv', usecols=['split', 'ket_luan'])
train = man[man['split'] == 'train'].dropna(subset=['ket_luan']).copy()
train['region'] = train['ket_luan'].map(regions)
train['n'] = train['ket_luan'].map(norm)
lookup = train.groupby('region')['n'].agg(lambda s: s.value_counts().index[0])
global_mode = train['n'].value_counts().index[0]
print('=== Most common TRAIN label per region ===')
print(lookup.to_string(), '\n')

df = pd.read_csv(f'{V5}/test_predictions_712.csv')          # read-only
df['gt_region']  = df['ground_truth'].map(regions)
df['v5_region']  = df['prediction'].map(regions)
df['gt_n']       = df['ground_truth'].map(norm)
df['V5 model']   = df['prediction'].map(norm)
df["Lookup (V5's region)"]  = df['v5_region'].map(lookup).fillna(global_mode)
df['Lookup (true region)']  = df['gt_region'].map(lookup).fillna(global_mode)
df['Always top label']      = global_mode

methods = ['V5 model', "Lookup (V5's region)", 'Lookup (true region)', 'Always top label']
rows = []
for m in methods:
    rows.append({
        'method': m,
        'exact': round((df[m] == df['gt_n']).mean(), 4),
        'token_f1': round(df.apply(lambda r: f1(r[m], r['gt_n']), axis=1).mean(), 4),
        'unique_outputs': df[m].nunique(),
    })
overall = pd.DataFrame(rows)
print('=== Overall (712 test cases) ===\n', overall.to_string(index=False), '\n')

per = []
for reg, sub in df.groupby('gt_region'):
    r = {'gt_region': reg, 'cases': len(sub)}
    for m in methods:
        r[f'{m} | exact'] = round((sub[m] == sub['gt_n']).mean(), 3)
        r[f'{m} | F1']    = round(sub.apply(lambda x: f1(x[m], x['gt_n']), axis=1).mean(), 3)
    per.append(r)
per = pd.DataFrame(per).sort_values('cases', ascending=False)
print('=== By GT region ===\n', per.to_string(index=False))

agree = (df['V5 model'] == df["Lookup (V5's region)"]).mean()
print(f"\nV5 output identical to lookup-by-its-own-region: {agree:.1%}")

overall.to_csv(f'{OUT}/v5_vs_lookup_overall_{ts}.csv', index=False)
per.to_csv(f'{OUT}/v5_vs_lookup_by_region_{ts}.csv', index=False)
print('Saved to', OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== Most common TRAIN label per region ===
region
Họng                    viêm họng mạn theo dõi trào ngược dịch vị
Khác                                            a mi đan quá phát
Mũi                                                      viêm mũi
Mũi+Họng                                        viêm họng mũi cấp
Tai             hiện không thấy hình ảnh bệnh lý tai ngoài tai...
Tai+Họng                                      dị vật tai p đã lấy
Tai+Mũi         hiện không thấy hình ảnh bệnh lý tai ngoài tai...
Tai+Mũi+Họng                viêm tai giữa p cấp viêm họng mũi cấp 

=== Overall (712 test cases) ===
               method  exact  token_f1  unique_outputs
            V5 model 0.1896    0.5979              16
Lookup (V5's region) 0.1713    0.5128               4
Lookup (true region) 0.2163    0.5993               8
    Always top label 0.0941    0.4575    